In [7]:
import numpy as np
import igl
import meshplot as mp
import openmesh as om
import networkx as nx
import trimesh
from collections import defaultdict
from collections import Counter
import time
# from shapely.geometry import Polygon 
import matplotlib.pyplot as plt
# from hole_filling import triangulate_refine_fair
from mpl_toolkits.mplot3d.art3d import Poly3DCollection, Line3DCollection


%load_ext autoreload
%autoreload 2
# import sys
# sys.path.append('..')
from utils import *
from primitive_fitting import *
from HoleFilling import *
from cut_calculate.intersection import *
from geometry_utils.geometry_process import *
from geometry_utils.geometry_computation import *
from fitting_geometric_primitives.primitives_2d import *
from fitting_geometric_primitives.geometry_primitive import *
from triangulate import *
from CylinderHoleFilling import *
from cut import *
from graph_cut import *
# from render.vtk_render import *


mp.jupyter()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
# read colors from file
file = open('global/colors_500.txt', 'r')
lines = file.readlines()
colors = np.zeros((len(lines),3))
for i in range(len(lines)):
    colors[i] = np.array(lines[i].split())


# def cluster_to_partition(proxy_cluster):
#     f_num = -1
#     for proxy2f_id in proxy2f:
#         f_num = max(f_num, max(proxy_cluster[cluster_id]))
#     f_num += 1
#     face_proxy = np.zeros(f_num, dtype=int)
#     for cluster_id in proxy_cluster:
#         for f_id in proxy_cluster[cluster_id]:
#             face_proxy[f_id] = cluster_id
#     return face_proxy
            

In [9]:
class InitPatchMesh:
    '''
        INPUT
            - v, f, patch_cluster_file
        OUTPUT
            - mesh attributes: f normal, area, center
            - patch topology: patch2v, patch2f, f2patch
    '''
    def __init__(self, v, f, patch_cluster_file=None, init_region_file=None):
        self.v = v
        self.f = f
        self.patch2f = self.read_cluster(patch_cluster_file)
        
        self.initialize()
        self.construct_graph()
    
    
    '''
        COMPUTE
        - v num, f num
        - f normal, f center, f area
        # - vv_idx, vf_idx
        - patch_num, patch2v: {[]}, patch2f: {[]}, f2patch: []
    '''
    def initialize(self):
        self.v_num = self.v.shape[0]
        self.f_num = self.f.shape[0]

        # mesh geometry
        self.f_normal = igl.per_face_normals(self.v, self.f, np.array([1., 0., 0.]))
        self.f_normal = self.f_normal / np.linalg.norm(self.f_normal, axis=1).reshape(-1, 1)
        self.f_center = np.mean(self.v[self.f], axis=1)
        self.f_area = igl.doublearea(self.v, self.f) / 2.
        
        
        # mesh topology
        self.om_mesh = om.TriMesh(self.v, self.f)
        # self.vv_idx = self.om_mesh.vv_indices()
        # self.vf_idx = self.om_mesh.vf_indices()
        
        # patch-mesh topology
        self.patch_num = len(self.patch2f)
        self.patch2v = defaultdict(list)
        for p_id in range(self.patch_num):
            self.patch2v[p_id] = list(np.unique(np.array(self.f[self.patch2f[p_id]]).flatten()))
        self.f2patch = np.zeros(self.f_num, dtype=int)
        for p_id in range(self.patch_num):
            self.f2patch[self.patch2f[p_id]] = p_id
                

    '''
        INPUT
            - patch2v
        OUTPUT:
            - p_p_adj: adjacency between patch, list(list(p0, p1))
                - only consider common-edge 
    '''
    def construct_adjacency(self):
        p_edges = []
        for p_id in range(self.patch_num):
            pf = self.patch2f[p_id]
            p_edge_set = set()
            for f in pf:
                for j in range(3):
                    v0, v1 = self.f[f][j], self.f[f][(j+1)%3]
                    p_edge_set.add((min(v0, v1), max(v0, v1)))
            p_edges.append(p_edge_set)
        
        self.p_p_adj = []
        for p0 in range(self.patch_num - 1):
            for p1 in range(p0 + 1, self.patch_num):
                if len(p_edges[p0] & p_edges[p1]) > 0:
                    self.p_p_adj.append([p0, p1])
    

    '''
        INPUT
            - patch2v
        OUTPUT
            - graph: patch graph with
                - nodes with v, f
                - edges
    '''
    def construct_graph(self):
        self.construct_adjacency()

        graph = nx.Graph()
        for p_id in range(self.patch_num):
            graph.add_node(p_id)
            graph.nodes[p_id]['v'] = self.patch2v[p_id]
            graph.nodes[p_id]['f'] = self.patch2f[p_id]
        
        for pair in self.p_p_adj:
            p0, p1 = pair
            graph.add_edge(p0, p1)
        self.graph = graph


    def read_region_cluster(self, region_file):
        file = open(region_file, 'r')
        lines = file.readlines()
        region2p = {}
        for i in range(len(lines)):
            region2p[i] = lines[i].split(' ')[:-1]
            region2p[i] = [int(pid) for pid in region2p[i]]
        return region2p
    

    def read_cluster(self, patch_cluster_file):
        file = open(patch_cluster_file, 'r')
        lines = file.readlines()
        cluster = {}
        for i in range(len(lines)):
            cluster[i] = []
            for j in lines[i].split():
                cluster[i].append(int(j))
        return cluster


In [10]:
class PatchMesh(InitPatchMesh):
    def __init__(self, v, f, patch2f):
        self.v = v
        self.f = f
        self.patch2f = {i: list(patch2f[i]) for i in range(len(patch2f))}
        
        self.initialize()
        self.construct_graph()

In [11]:
class BorderType:
    '''
        INPUT
            - patch mesh with:
                - patch<->f, patch<->v
                - connectivity of patches
                - connectivity of mesh
    '''
    def __init__(self, patch_mesh):
        self.patch_mesh = patch_mesh
        self.threshold_convexity = np.cos(87.5 * np.pi / 180)
        # self.threshold_convexity = np.cos(82.5 * np.pi / 180)
        
        
    
    '''
        fit vertices into to line, by eigen vector of max eigen value
        INPUT
            - indices of vertices
        OUTPUT
            - direction, center
    '''
    def get_line_fitting(self, v_idx):
        v = self.patch_mesh.v[v_idx]
        centroid = np.mean(v, axis=0)
        v_centered = v - centroid
        cov = np.cov(v_centered, rowvar=False)
        eigenvalues, eigenvectors = np.linalg.eig(cov)
        line_direction = eigenvectors[:, np.argmax(eigenvalues)]
        line_direction /= np.linalg.norm(line_direction)
        return line_direction, centroid


    '''
        get neighbor vertices of vertices in patch
        INPUT
            - indices of vertices, patch id
            - patch2v
            - vv neighbor
        OUTPUT
            - indices of vertices
    '''
    def get_neighbor_v(self, v_border, patch_id):
        # all_n_v = np.unique(np.array([self.patch_mesh.vv_idx[v] for v in v_border]).flatten())
        all_n_v = []
        for v in v_border:
            v_h = self.patch_mesh.om_mesh.vertex_handle(v)
            for n_v_h in self.patch_mesh.om_mesh.vv(v_h):
                all_n_v.append(n_v_h.idx())
        all_n_v = list(set(all_n_v))
        n_v = []
        for v in all_n_v:
            if (v not in v_border) & (v in self.patch_mesh.patch2v[patch_id]):
                n_v.append(v)
        n_v = np.array(n_v).astype(int)
        return n_v

    
    '''
        get neighbor faces of vertices in patch
        INPUT
            - indices of vertices, patch id
            - f2patch
            - vf neighbor
        OUTPUT
            - indices of faces
    '''
    def get_neighbor_f(self, v_border, patch_id):
        # all_n_f = np.unique(np.array([self.patch_mesh.vf_idx[v] for v in v_border]).flatten())
        all_n_f = []
        for v in v_border:
            v_h = self.patch_mesh.om_mesh.vertex_handle(v)
            for n_v_h in self.patch_mesh.om_mesh.vf(v_h):
                all_n_f.append(n_v_h.idx())
        all_n_f = list(set(all_n_f))
        n_f = []
        for f in all_n_f:
            if self.patch_mesh.f2patch[f] == patch_id:
                n_f.append(f)
        n_f = np.array(n_f)
        # if n_f.shape[0] == 0:
        #     a = 1
        return n_f


    ''' 
        project vertices into plane
        INPUT
            - plane's normal, center
            - indices of vertices
        OUTPUT
            - projected vertices
    '''
    def projected_v(self, normal, center, v_id):
        v = self.patch_mesh.v[v_id]
        
        line_proj = np.dot(v - center, normal) / np.dot(normal, normal)
        line_proj = line_proj[:, np.newaxis]
        v_proj = v - line_proj * normal
        return v_proj
    
    
    ''' 
        project vector into plane
        INPUT
            - plane's normal
            - vector
        OUTPUT
            - projected vector
    '''
    def projected_vec(self, normal, vec):
        line_proj = np.dot(vec, normal) / np.dot(normal, normal)
        line_proj = line_proj[:, np.newaxis]
        vec_proj = vec - line_proj * normal
        return vec_proj
    

    '''
        get types of all borders
        NEED
            - v, f, type of all patches
            - connectivities of patches
        COMPUTE
            - edge_type: border type of each edge
    '''
    def get_all_borders(self, visualize=False):        
        for edge in self.patch_mesh.graph.edges:
            p1, p2 = edge
            border_type, convexity = self.get_border_type(p1, p2)

            self.patch_mesh.graph.edges[p1, p2]["border_type"] = border_type
            self.patch_mesh.graph.edges[p1, p2]["convexity"] = convexity
            self.patch_mesh.graph.edges[p1, p2]['border_v'] = self.get_patch_border_v(p1, p2)


    '''
        get border type between two neighbor patches
        INPUT:
            - patch id p1, p2
            - patch<->f, patch->v
            - type of patch
            - vf, vv neighbor
        OUTPUT:
            - border type of p1 and p2
                1: concave
                2: convex
                3: flat
    '''
    def get_border_type(self, p1, p2, G=None):
        # if p1 == 2 and p2 == 1:
        #     print(1)

        if G:
            self.patch_mesh.graph = G
        # if same planar, combine
        # if (self.patch_mesh.patch_num == 18) & (p1 == 0) & (p2 == 17):
            # print('at')
        # if (self.patch_mesh.graph.nodes[p1]["type"] == "Plane") & (self.patch_mesh.graph.nodes[p2]["type"] == "Plane"):
        #     params1 = self.patch_mesh.graph.nodes[p1]["params"]
        #     params2 = self.patch_mesh.graph.nodes[p2]["params"]
        #     if np.linalg.norm(params1 - params2) < 1e-2:
        #         return 3

        def get_single_border_type(_tmp_border):
            
            # fitted line of border
            line_direction, line_center = self.get_line_fitting(_tmp_border)

            # neighbor v & f
            n_v_1, n_v_2 = self.get_neighbor_v(_tmp_border, p1), self.get_neighbor_v(_tmp_border, p2)
            n_f_1, n_f_2 = self.get_neighbor_f(_tmp_border, p1), self.get_neighbor_f(_tmp_border, p2)
            # print(p1, p2, len(_tmp_border), n_v_1.shape, n_v_2.shape, n_f_1.shape, n_f_2.shape)
            n_f_n_1, n_f_n_2 = self.patch_mesh.f_normal[n_f_1], self.patch_mesh.f_normal[n_f_2]

            # projected center of n_v in p1, p2
            c1 = self.projected_v(line_direction, line_center, n_v_1)
            c1 = np.mean(c1, axis=0)
            c2 = self.projected_v(line_direction, line_center, n_v_2)
            c2 = np.mean(c2, axis=0)
            
            # area-weighted projected normal of n_f in p1, p2
            n1 = self.projected_vec(line_direction, n_f_n_1)
            n1 *= self.patch_mesh.f_area[n_f_1][:, np.newaxis]
            n1 = np.sum(n1, axis=0)
            n1 /= np.linalg.norm(n1)
            n2 = self.projected_vec(line_direction, n_f_n_2)
            n2 *= self.patch_mesh.f_area[n_f_2][:, np.newaxis]
            n2 = np.sum(n2, axis=0)
            n2 /= np.linalg.norm(n2)
            
            
            # metric
            convexity = 0.5 * (np.dot(n1, (c2 - c1) / np.linalg.norm(c2 - c1)) + np.dot(n2, (c1 - c2) / np.linalg.norm(c1 - c2)))
            

            
            # first classify
            type1 = -1
            if convexity >= self.threshold_convexity:
                type1 = 1
            elif convexity <= -self.threshold_convexity:
                type1 = 2
            else:
                type1 = 3
            return type1, convexity
        # print(type1)
        # return type1    

        # print(f'{p1} and {p2} have {v_border.shape[0]}')

        # v_border = self.get_patch_border_v(p1, p2)

        # if closed border, detect radio of types of all edges
        has_circle, edges = self.detect_patch_border_v_circle(p1, p2)
        if has_circle:
            e_type = []
            for edge in edges:
                e_h = v_pair2e_h(self.patch_mesh.om_mesh, edge[0], edge[1])
                e_type.append(get_single_edge_convexity(
                    self.patch_mesh.om_mesh,
                    self.patch_mesh.f_center,
                    self.patch_mesh.f_normal,
                    e_h
                ))
            most_type = Counter(e_type).most_common()[0][0]
            all_convexity = []
            all_types = []
            for _tmp in self.get_patch_border_v(p1, p2):
                # length of border
                length = np.linalg.norm(self.patch_mesh.v[_tmp[-1]] - self.patch_mesh.v[_tmp[0]])
                _type, _convexity = get_single_border_type(_tmp)
                all_types.append(_type)
                all_convexity.append(abs(_convexity) * length)
            
            return most_type, np.sum(all_convexity)
        

        v1 = self.patch_mesh.patch2v[p1]
        v2 = self.patch_mesh.patch2v[p2]
        # v_border = np.intersect1d(v1, v2)
        v_border = self.get_patch_border_v(p1, p2)

        

        all_convexity = []
        all_types = []
        for _tmp in v_border:
            # length of border
            length = np.linalg.norm(self.patch_mesh.v[_tmp[-1]] - self.patch_mesh.v[_tmp[0]])
            _type, _convexity = get_single_border_type(_tmp)
            all_types.append(_type)
            all_convexity.append(abs(_convexity) * length)
        # all_types = [get_single_border_type(_tmp) for _tmp in v_border]
        most_type = Counter(all_types).most_common()[0][0]
        # most_type = get_single_border_type(v_border)
        # get most common type and correspond convexity

        # print(most_type)
        # print(all_convexity)
        return most_type, np.sum(all_convexity)


    def show_all_borders(self):
        # show all borders
        


        # get all edges
        edges_v_id = []
        edges_color = []
        G = self.patch_mesh.graph
        all_endpoints = []
        # for edge in G.edges:
        #     p1, p2 = edge
        #     border_type = G.edges[p1,p2]['border_type']
        #     if (border_type == 1) | (border_type == 2):
        #         v_borders = self.get_patch_border_v(p1, p2)
        #         for single_border in v_borders:
        #             for i in range(len(single_border) - 1):
        #                 edges_v_id.append([single_border[i], single_border[i+1]])
        #                 edges_color.append([1, 0, 0] if border_type == 1 else [0, 0, 1])
        for edge in G.edges:
            p1, p2 = edge
            # if p1 == 8 and p2 == 9:
            v_borders = self.get_patch_border_v(p1, p2)
            for single_border in v_borders:
                all_endpoints.append(self.patch_mesh.v[single_border[0]])
                all_endpoints.append(self.patch_mesh.v[single_border[-1]])

                for i in range(len(single_border) - 1):
                    edges_v_id.append([single_border[i], single_border[i+1]])
                    edges_color.append(colors[92])
        edges_color = np.array(edges_color)
        p_color = colors[92]

        f_colors = np.ones((self.patch_mesh.f.shape[0], 3))
        # f_colors[self.patch_mesh.patch2f[9]] = colors[41]
        # xml_path = '../meshsegment_paper/camera/2/concave_loop.xml'
        xml_path = '../meshsegment_paper/camera/2/raw_borders.xml'
        # xml_path = '../meshsegment_paper/camera/cad16.xml'
        # xml_path = '../meshsegment_paper/camera/140750.xml'
        render_mesh_vtk(
            self.patch_mesh.v, self.patch_mesh.f, xml_path=xml_path,
            # 'render/140750/reference.png',
            # 'render/140750/raw_borders.png',
            # 'render/cad16/raw_borders.png',
            # 'render/2/all_patch_borders.png',
            # 'render/2/concave_loop.png',
            write_img=False, 
            draw_e=True, e = edges_v_id, edge_radius=0.3, _e_color = edges_color, 
            draw_f=True, _f_color=f_colors,
            draw_p=True, _p=all_endpoints,  _p_color=p_color, _p_size=0.7
        )


        # f_merged_5 = []
        # f_colors = []
        # for p_id in [5, 6, 7]:
        #     f_merged_5.extend(self.patch_mesh.patch2f[p_id])
        #     f_colors.extend([colors[p_id]] * len(self.patch_mesh.patch2f[p_id]))
        # xml_path = '../meshsegment_paper/camera/2/raw_borders.xml'
        # render_mesh_vtk(
        #     self.patch_mesh.v, self.patch_mesh.f[f_merged_5],
        #     'render/2/merged_patch_5.png',
        #     draw_f=True, _f_color=f_colors, xml_path=xml_path
        # )

         
        # f_merged_7 = []
        # f_colors = []
        # for p_id in [9, 10, 11, 12, 13]:
        #     f_merged_7.extend(self.patch_mesh.patch2f[p_id])
        #     f_colors.extend([colors[p_id]] * len(self.patch_mesh.patch2f[p_id]))
        # xml_path = '../meshsegment_paper/camera/2/raw_borders.xml'
        # render_mesh_vtk(
        #     self.patch_mesh.v, self.patch_mesh.f[f_merged_7],
        #     'render/2/merged_patch_7.png',
        #     draw_f=True, _f_color=f_colors, xml_path=xml_path
        # )

        # f_colors = np.ones((self.patch_mesh.f.shape[0], 3))
        # for p_id in range(len(self.patch_mesh.patch2f)):
        #     f_colors[self.patch_mesh.patch2f[p_id]] = colors[p_id]
        # xml_path = '../meshsegment_paper/camera/2/raw_borders.xml'
        # render_mesh_vtk(
        #     self.patch_mesh.v, self.patch_mesh.f,
        #     'render/2/all_merged_patch.png',
        #     draw_f=True, _f_color=f_colors, xml_path=xml_path
        # )


        # xml_path = '../meshsegment_paper/camera/2/non_primitive_5.xml'
        # f_5 = self.patch_mesh.patch2f[5]
        # raw_f5_index = []
        # raw_f5_path = 'render/2/vsa/region_0.txt'
        # raw_f5_file = open(raw_f5_path, 'r')
        # lines = raw_f5_file.readlines()
        # for line in lines:
        #     raw_f5_index.append(int(line.strip()))
        # raw_f5_file.close()
        # folder_5 = 'render/2/vsa/5'
        # import os
        # txt_files = [f for f in os.listdir(folder_5) if f.endswith('.txt')]
        # for txt_file in txt_files:
        #     print(txt_file)
        #     f_colors = np.ones((self.patch_mesh.f[f_5].shape[0], 3))
        #     path = os.path.join(folder_5, txt_file)
        #     R = []
        #     with open(path, 'r') as f:
        #         lines = f.readlines()
        #         for line in lines:
        #             R.append(int(line.strip()))
        #     for i in range(self.patch_mesh.f[f_5].shape[0]):
        #         idx = [index for index, value in enumerate(f_5) if value == raw_f5_index[i]][0]
        #         f_colors[idx] = colors[5 + R[i]]
            
        #     render_mesh_vtk(
        #         self.patch_mesh.v, self.patch_mesh.f[f_5],
        #         f'render/2/non_primitive_5/{txt_file[:-4]}.png',
        #         draw_f=True, _f_color=f_colors, xml_path=xml_path
        #     )


        # xml_path = '../meshsegment_paper/camera/2/non_primitive_7.xml'
        # f_7 = self.patch_mesh.patch2f[7]
        # raw_f7_index = []
        # raw_f7_path = 'render/2/vsa/region_1.txt'
        # raw_f7_file = open(raw_f7_path, 'r')
        # lines = raw_f7_file.readlines()
        # for line in lines:
        #     raw_f7_index.append(int(line.strip()))
        # raw_f7_file.close()
        # folder_7 = 'render/2/vsa/7'
        # import os
        # txt_files = [f for f in os.listdir(folder_7) if f.endswith('.txt')]
        # for txt_file in txt_files:
        #     print(txt_file)
        #     f_colors = np.ones((self.patch_mesh.f[f_7].shape[0], 3))
        #     path = os.path.join(folder_7, txt_file)
        #     R = []
        #     with open(path, 'r') as f:
        #         lines = f.readlines()
        #         for line in lines:
        #             R.append(int(line.strip()))
        #     for i in range(self.patch_mesh.f[f_7].shape[0]):
        #         idx = [index for index, value in enumerate(f_7) if value == raw_f7_index[i]][0]
        #         f_colors[idx] = colors[7 + R[i]]

        #     render_mesh_vtk(
        #         self.patch_mesh.v, self.patch_mesh.f[f_7],
        #         f'render/2/non_primitive_7/{txt_file[:-4]}.png',
        #         draw_f=True, _f_color=f_colors, xml_path=xml_path
        #     )


        # xml_path = '../meshsegment_paper/camera/2/raw_borders.xml'
        # for p_id in [6, 8, 9, 15, 16]:
        #     f_cylinder = self.patch_mesh.patch2f[p_id]
        #     f_colors_cylinder = np.tile(colors[41], (self.patch_mesh.f[f_cylinder].shape[0], 1))

        #     render_mesh_vtk(
        #         self.patch_mesh.v, self.patch_mesh.f[f_cylinder],
        #         f'render/2/cylinder_{p_id}.png',
        #         draw_f=True, _f_color=f_colors_cylinder, xml_path=xml_path
        #     )
        
        # xml_path = '../meshsegment_paper/camera/2/raw_borders.xml'
        # for p_id in [0, 1, 2, 3, 4, 10, 11, 12, 13, 14]:
        #     f_plane = self.patch_mesh.patch2f[p_id]
        #     f_colors_plane = np.tile(colors[29], (self.patch_mesh.f[f_plane].shape[0], 1))

        #     render_mesh_vtk(
        #         self.patch_mesh.v, self.patch_mesh.f[f_plane],
        #         f'render/2/plane_{p_id}.png',
        #         draw_f=True, _f_color=f_colors_plane, xml_path=xml_path
        #     )
        
        

        # xml_path = '../meshsegment_paper/camera/2/raw_borders.xml'
        # f_7 = self.patch_mesh.patch2f[7]
        # f_colors_7 = np.tile(colors[7], (self.patch_mesh.f[f_7].shape[0], 1))
        # render_mesh_vtk(
        #     self.patch_mesh.v, self.patch_mesh.f[f_7],
        #     # 'render/2/non_primitive_7.png',
        #     'render/2/non_primitive_7_raw.png',
        #     draw_f=True, _f_color=f_colors_7, xml_path=xml_path
        # )

        # # xml_path = '../meshsegment_paper/camera/2/non_primitive_5.xml'
        # xml_path = '../meshsegment_paper/camera/2/raw_borders.xml'
        # f_5 = self.patch_mesh.patch2f[5]
        # f_colors_5 = np.tile(colors[5], (self.patch_mesh.f[f_5].shape[0], 1))
        # render_mesh_vtk(
        #     self.patch_mesh.v, self.patch_mesh.f[f_5],
        #     # 'render/2/non_primitive_5.png',
        #     'render/2/non_primitive_5_raw.png',
        #     draw_f=True, _f_color=f_colors_5, xml_path=xml_path
        # )
        
        
        # f_colors = np.ones((self.patch_mesh.f.shape[0], 3))
        # # for p_id in range(len(self.patch_mesh.patch2f)):
        # # # for p_id in [6, 8, 9, 15, 16]:
        # # # for p_id in [0, 1, 2, 3, 4, 10, 11, 12, 13, 14]:
        # #     f_colors[self.patch_mesh.patch2f[p_id]] = colors[p_id]
        # #     # f_colors[self.patch_mesh.patch2f[p_id]] = colors[41]
        # #     # f_colors[self.patch_mesh.patch2f[p_id]] = colors[29]
        # # # xml_path = '../meshsegment_paper/camera/2/non_primitive_7.xml'
        # # # xml_path = '../meshsegment_paper/camera/2/non_primitive_5.xml'
        # xml_path = '../meshsegment_paper/camera/2/raw_borders.xml'
        # render_mesh_vtk(
        #     self.patch_mesh.v, self.patch_mesh.f,
        #     # 'render/2/pre_patch.png',
        #     # 'render/2/cylinder_primitives.png',
        #     # 'render/2/plane_primitives.png',
        #     'render/2/raw_borders.png',
        #     e = edges_v_id, draw_f=True, draw_e=True, _f_color=f_colors, _e_color = edges_color, xml_path=xml_path
        # )


    def cut_by_convex_concave_border(self):
        # area
        for node in self.patch_mesh.graph.nodes:
            f_ids = self.patch_mesh.graph.nodes[node]['f']
            area = np.sum(self.patch_mesh.f_area[f_ids])
            self.patch_mesh.graph.nodes[node]['area'] = area
        

        seeds = set()
        for edge in self.patch_mesh.graph.edges:
            p1, p2 = edge
            border_type = self.patch_mesh.graph.edges[p1, p2]['border_type']
            if border_type == 1:
                seeds.add(p1)
                seeds.add(p2)

        convex_edges = []
        for edge in self.patch_mesh.graph.edges:
            p1, p2 = edge
            border_type = self.patch_mesh.graph.edges[p1, p2]['border_type']
            if border_type == 2:
                convex_edges.append((p1, p2))
        
        
        results = graph_min_cut(
            self.patch_mesh.graph,
            seeds,
            convex_edges
        )

        # get all edges
        edges_v_id = []
        edges_color = []
        G = self.patch_mesh.graph
        all_endpoints = []
        for edge in G.edges:
            p1, p2 = edge
            border_type = G.edges[p1,p2]['border_type']
            if (border_type == 1) | (border_type == 2):
                v_borders = self.get_patch_border_v(p1, p2)
                for single_border in v_borders:
                    all_endpoints.append(self.patch_mesh.v[single_border[0]])
                    if single_border[0] == single_border[-1]:
                        all_endpoints.append(self.patch_mesh.v[single_border[-1]])
                    else:
                        all_endpoints.append(self.patch_mesh.v[single_border[-2]])

                    for i in range(len(single_border) - 1):
                        edges_v_id.append([single_border[i], single_border[i+1]])
                        edges_color.append([1, 0, 0] if border_type == 1 else [0, 0, 1])
        edges_color = np.array(edges_color)
        

        for seed_id, sub_nodes in results.items():
            print(f'Seed patch {seed_id}:')
            
            f_colors = np.ones((self.patch_mesh.f.shape[0], 3))
            for node in sub_nodes:
                f_ids = self.patch_mesh.graph.nodes[node]['f']
                f_colors[f_ids] = [0, 0, 1]
            f_ids = self.patch_mesh.graph.nodes[seed_id]['f']
            f_colors[f_ids] = [1., 0., 0.]
            mp.plot(self.patch_mesh.v, self.patch_mesh.f, f_colors, shading={
                'wireframe': False
            })
        
            # xml_path = '../meshsegment_paper/camera/cad16.xml'
            # render_mesh_vtk(
            #     self.patch_mesh.v, self.patch_mesh.f,
            #     f'render/cad16/seed{seed_id}.png',
            #     # 'render/2/all_patch_borders.png',
            #     e = edges_v_id, draw_f=True, draw_e=True, _f_color=f_colors, _e_color = edges_color, xml_path=xml_path,
            #     edge_radius=0.007
            #     # draw_p=True, _p=all_endpoints, edge_radius=0.3, _p_size=0.7
            # )

        f_0 = []
        patch2f_0 = []
        for n in results[16]:
            f_0.extend(self.patch_mesh.graph.nodes[n]['f'])
        f_0 = np.array(f_0)
        for n in results[16]:
            cur_f = self.patch_mesh.graph.nodes[n]['f']
            patch2f_0.append(np.array([np.where(f_0 == f_id)[0][0] for f_id in cur_f]))
        
        # rest of self.patch_mesh.f
        f_1 = []
        patch2f_1 = []
        for p_id in self.patch_mesh.graph.nodes():
            if p_id not in results[16]:
                f_1.extend(self.patch_mesh.graph.nodes[p_id]['f'])
        f_1 = np.array(f_1)
        for p_id in self.patch_mesh.graph.nodes():
            if p_id not in results[16]:
                cur_f = self.patch_mesh.graph.nodes[p_id]['f']
                patch2f_1.append(np.array([np.where(f_1 == f_id)[0][0] for f_id in cur_f]))
        
        v0, f0 = get_submesh(self.patch_mesh.v, self.patch_mesh.f, f_0)
        v1, f1 = get_submesh(self.patch_mesh.v, self.patch_mesh.f, f_1)
        
        f_colors_0 = np.ones((f0.shape[0], 3))
        for p_id in range(len(patch2f_0)):
            f_colors_0[patch2f_0[p_id]] = colors[p_id % 500]
        f_colors_1 = np.ones((f1.shape[0], 3))
        for p_id in range(len(patch2f_1)):
            f_colors_1[patch2f_1[p_id]] = colors[p_id % 500]
            
        mp.plot(v0, f0, f_colors_0, shading={'wireframe': False})
        mp.plot(v1, f1, f_colors_1, shading={'wireframe': False})


        return [
            {
                'v': v0,
                'f': f0,
                'patch': patch2f_0
            },
            {
                'v': v1,
                'f': f1,
                'patch': patch2f_1
            }
        ]


    '''
        try cut patches by closed and seperable concave border between patches 
        INPUT
            - border type of all edges
            - patch2f
        OUTPUT
            - is seperable
            - sub objects, if seperable 
    '''
    def cut_by_concave_border(self):
        graph_cut = GraphCut(copy.deepcopy(self.patch_mesh))
        return graph_cut.result(viz=False)
     

    def show_merged_patch(self, proxy2f_file, merged_file=None):
        proxy2f = {}
        with open(proxy2f_file, 'r') as f0:
            lines = f0.readlines()
            for i in range(len(lines)):
                proxy2f[i] = []
                for j in lines[i].split():
                    proxy2f[i].append(int(j))
        merged_proxy = {}
        with open(merged_file, 'r') as f1:
            lines = f1.readlines()
            for i in range(len(lines)):
                merged_proxy[i] = []
                for j in lines[i].split():
                    merged_proxy[i].append(int(j))
        
        f_color = np.ones((self.patch_mesh.f.shape[0], 3))
        for region in merged_proxy:
            for p_id in merged_proxy[region]:
                f_id = proxy2f[p_id]
                f_color[f_id] = colors[region % 500]
        mp.plot(self.patch_mesh.v, self.patch_mesh.f, c=f_color, shading={
            'wireframe': False
        })

        # # raw proxy color
        # raw_f_color = np.ones((self.patch_mesh.f.shape[0], 3))
        # for p in proxy2f:
        #     f_id = proxy2f[p]
        #     raw_f_color[f_id] = colors[p % 500]
        # raw_file_name = proxy2f_file.split('/')[-1].split('.')[0]
        # raw_ply_filename = f"{raw_file_name}.ply"
        # mesh = trimesh.Trimesh(self.patch_mesh.v, self.patch_mesh.f)
        # mesh.visual.face_colors = (raw_f_color * 255).astype(np.uint8)
        # mesh.export(raw_ply_filename)

        # # save into ply file
        # file_name = merged_file.split('/')[-1].split('.')[0]
        # ply_filename = f"{file_name}_patch.ply"
        # mesh = trimesh.Trimesh(self.patch_mesh.v, self.patch_mesh.f)
        # mesh.visual.face_colors = (f_color * 255).astype(np.uint8)
        # mesh.export(ply_filename)

    
    def show_patch_type(self):
        patch2f = self.patch_mesh.patch2f
        f_colors = np.ones((self.patch_mesh.f.shape[0], 3))
        for p_id in patch2f:
            p_type = self.patch_mesh.graph.nodes[p_id]['type']
            if p_type == 'Plane':
                c = [1., 0., 0.]
            elif p_type == 'Cylinder':
                c = [0., 1., 0.]
            elif p_type == 'Sphere':
                c = [0., 0., 1.]
            elif p_type == 'Cone':
                c = [1., 1., 0]
            else:
                c = [1., 0., 1.]
            f_colors[patch2f[p_id]] = c
        mp.plot(self.patch_mesh.v, self.patch_mesh.f, f_colors)

    
    def show_patch(self):
        patch2f = self.patch_mesh.patch2f
        f_color = np.ones((self.patch_mesh.f.shape[0], 3))
        for p_id in patch2f:
            f_color[patch2f[p_id]] = colors[p_id % 500]
        mp.plot(self.patch_mesh.v, self.patch_mesh.f, c=f_color, shading={
            'wireframe': False
        })


    def show_each_patch(self):
        # patch2f = self.patch_mesh.patch2f
        # f_color = np.ones((self.patch_mesh.f.shape[0], 3))
        # plot = mp.plot(self.patch_mesh.v, self.patch_mesh.f, c=f_color, shading={
        #     'wireframe': False
        # })
        # for p_id in patch2f:
        #     print(f"patch {p_id}")
        #     f_color[patch2f[p_id]] = colors[p_id % 500]
        #     plot.update_object(colors=f_color)
        #     print(1)

        patch2f = self.patch_mesh.patch2f
        for p_id in patch2f:
            print(f"patch {p_id}")
            f_color = np.ones((self.patch_mesh.f.shape[0], 3))
            f_color[patch2f[p_id]] = colors[p_id % 500]
            plot = mp.plot(self.patch_mesh.v, self.patch_mesh.f, c=f_color, shading={
            'wireframe': False
            })


    def show_single_patch(self, patch_id):
        f_id = self.patch_mesh.graph.nodes[patch_id]['f']
        f_color = np.ones((self.patch_mesh.f.shape[0], 3))
        f_color[f_id] = colors[0]
        mp.plot(self.patch_mesh.v, self.patch_mesh.f, c=f_color, shading={
            'wireframe': False
        })


    def show_multi_patch(self, patch_list):
        f_color = np.ones((self.patch_mesh.f.shape[0], 3))
        for p_id in patch_list:
            f_id = self.patch_mesh.graph.nodes[p_id]['f']
            f_color[f_id] = colors[p_id % 500]
        mp.plot(self.patch_mesh.v, self.patch_mesh.f, f_color, shading={
            'wirefrace': False
        })

    
    '''
        compute normal attribute of all patches
    '''
    def compute_all_patch_normal_attribute(self, patch_list, visualize=False):
        
        om_mesh = self.patch_mesh.om_mesh
        f_normal = igl.per_face_normals(self.patch_mesh.v, self.patch_mesh.f, np.array([1., 0., 0.]))
        def get_n_ring_neighbot_f(f_id):
            nf_1_ring = set()
            nf_2_ring = set()
            nf_3_ring = set()
            nf_4_ring = set()
            f_h = om_mesh.face_handle(f_id)
            for n_f_h in om_mesh.ff(f_h):
                nf_1_ring.add(n_f_h.idx())
                for nn_f_h in om_mesh.ff(n_f_h):
                    nf_2_ring.add(nn_f_h.idx())
                    for nnn_f_h in om_mesh.ff(nn_f_h):
                        nf_3_ring.add(nnn_f_h.idx())
                        for nnnn_f_h in om_mesh.ff(nnn_f_h):
                            nf_4_ring.add(nnnn_f_h.idx())
            nf_1_ring = list(nf_1_ring)
            nf_2_ring = list(nf_2_ring)
            nf_3_ring = list(nf_3_ring)
            nf_4_ring = list(nf_4_ring)
            return nf_1_ring, nf_2_ring, nf_3_ring, nf_4_ring
        
        def pca(vectors):
            centroid = np.mean(vectors, axis=0)
            centroid_vectors = vectors - centroid
            cov = np.cov(centroid_vectors.T)
            eigen_value, eigen_vector = np.linalg.eigh(cov)
            
            return eigen_value

        # face_1_ring_pca = np.zeros((self.patch_mesh.f.shape[0], 3))
        
        pca_1_ring = []
        pca_2_ring = []
        pca_3_ring = []
        pca_4_ring = []
        
        
        
        pca_1_ring_on_patch_list = {}
        pca_2_ring_on_patch_list = {}
        pca_3_ring_on_patch_list = {}
        pca_4_ring_on_patch_list = {}
        
        for p_id in patch_list:
            pca_1_ring = []
            pca_2_ring = []
            pca_3_ring = []
            pca_4_ring = []
            p_faces = self.patch_mesh.graph.nodes[p_id]['f']
            
            # for f_id in range(self.patch_mesh.f.shape[0]):
            for f_id in p_faces:
                nf_1_ring, nf_2_ring, nf_3_ring, nf_4_ring = get_n_ring_neighbot_f(f_id)
                nf_1_ring_normal = f_normal[nf_1_ring]
                nf_1_ring_normal = np.array(nf_1_ring_normal).reshape((-1, 3))
                nf_1_ring_normal_pca = pca(nf_1_ring_normal)
                pca_1_ring.append(nf_1_ring_normal_pca)
                
                nf_2_ring_normal = f_normal[nf_2_ring]
                nf_2_ring_normal = np.array(nf_2_ring_normal).reshape((-1, 3))
                nf_2_ring_normal_pca = pca(nf_2_ring_normal)
                pca_2_ring.append(nf_2_ring_normal_pca)
                
                nf_3_ring_normal = f_normal[nf_3_ring]
                nf_3_ring_normal = np.array(nf_3_ring_normal).reshape((-1, 3))
                nf_3_ring_normal_pca = pca(nf_3_ring_normal)
                pca_3_ring.append(nf_3_ring_normal_pca)
                
                nf_4_ring_normal = f_normal[nf_4_ring]
                nf_4_ring_normal = np.array(nf_4_ring_normal).reshape((-1, 3))
                nf_4_ring_normal_pca = pca(nf_4_ring_normal)
                pca_4_ring.append(nf_4_ring_normal_pca)
                
            pca_1_ring = np.array(pca_1_ring)
            pca_2_ring = np.array(pca_2_ring)
            pca_3_ring = np.array(pca_3_ring)
            pca_4_ring = np.array(pca_4_ring)
            
            pca_1_ring_on_patch_list[p_id] = pca_1_ring
            pca_2_ring_on_patch_list[p_id] = pca_2_ring
            pca_3_ring_on_patch_list[p_id] = pca_3_ring
            pca_4_ring_on_patch_list[p_id] = pca_4_ring

            
        # show value
        if visualize:
            # normalize over all faces
            def pca_n_ring_all_normalized(pca_n_ring_on_patch_list):
                # concat all np array in dictory to one
                pca_n_ring_all = np.concatenate(list(pca_n_ring_on_patch_list.values()), axis=0)
                min_val, max_val = np.min(pca_n_ring_all, axis=0), np.max(pca_n_ring_all, axis=0)
                return (pca_n_ring_all - min_val) / (max_val - min_val)

            def show_all_normalized_mesh(pca_n_ring):
                f_color = np.ones((self.patch_mesh.f.shape[0], 3))
                cur_row_id = 0
                for i in range(len(patch_list)):
                    p_id = patch_list[i]
                    f_id = self.patch_mesh.graph.nodes[p_id]['f']
                    for cur_p_f_id in f_id:
                        f_color[cur_p_f_id] = pca_n_ring[cur_row_id]
                        cur_row_id += 1                
                mp.plot(self.patch_mesh.v, self.patch_mesh.f, f_color, shading={
                    'wireframe': False
                })

            pca_1_ring_all = pca_n_ring_all_normalized(pca_1_ring_on_patch_list)
            show_all_normalized_mesh(pca_1_ring_all)
            pca_2_ring_all = pca_n_ring_all_normalized(pca_2_ring_on_patch_list)
            show_all_normalized_mesh(pca_2_ring_all)
            pca_3_ring_all = pca_n_ring_all_normalized(pca_3_ring_on_patch_list)
            show_all_normalized_mesh(pca_3_ring_all)
            pca_4_ring_all = pca_n_ring_all_normalized(pca_4_ring_on_patch_list)
            show_all_normalized_mesh(pca_4_ring_all)



            # mean in patch + normalized over patch
            def show_patch_normalized_mesh(pca_n_ring_on_patch_list):
                pca_attr = []
                for p_id in patch_list:
                    f_id = self.patch_mesh.graph.nodes[p_id]['f']
                    pca_attribute = pca_n_ring_on_patch_list[p_id]
                    pca_attribute = np.mean(pca_attribute, axis=0)
                    pca_attr.append(pca_attribute)
                pca_attr = np.array(pca_attr)
                min_val, max_val = np.min(pca_attr, axis=0), np.max(pca_attr, axis=0)
                pca_attr = (pca_attr - min_val) / (max_val - min_val)
                f_color = np.ones((self.patch_mesh.f.shape[0], 3))
                for i in range(len(patch_list)):
                    p_id = patch_list[i]
                    f_id = self.patch_mesh.graph.nodes[p_id]['f']
                    f_color[f_id] = pca_attr[i]                
                mp.plot(self.patch_mesh.v, self.patch_mesh.f, f_color, shading={
                    'wireframe': False
                })
            show_patch_normalized_mesh(pca_1_ring_on_patch_list)
            show_patch_normalized_mesh(pca_2_ring_on_patch_list)
            show_patch_normalized_mesh(pca_3_ring_on_patch_list)
            show_patch_normalized_mesh(pca_4_ring_on_patch_list)

            
            a = 1
            
            # for j in range(3):
                # min_val, max_val = np.min(pca_1_ring[:, j]), np.max(pca_1_ring[:, j])
                # min_val, max_val = np.min(pca_2_ring[:, j]), np.max(pca_2_ring[:, j])
                # if max_val != min_val:
                #     for i in range(num_p_faces):
                #         f_id = p_faces[i]
                #         f_color[f_id, j] = (pca_1_ring[i, j] - min_val) / (max_val - min_val)
                # else:
                #     for i in range(num_p_faces):
                #         f_color[p_faces[i], j] = 0.5
                # if max_val != min_val:
                #     f_color[:, j] = (pca_1_ring[i, j] - min_val) / (max_val - min_val)
                # else:
                #     f_color[:, j] = 0.5
            # mp.plot(self.patch_mesh.v, self.patch_mesh.f, c=f_color, shading={
            #     'wireframe': False
            # })

        
        def show_cluster_by_val(labels):
            f_color = np.ones((self.patch_mesh.f.shape[0], 3))
            for i in range(len(patch_list)):
                p_id = patch_list[i]
                f_id = self.patch_mesh.graph.nodes[p_id]['f']
                f_color[f_id] = colors[labels[i]]
            mp.plot(self.patch_mesh.v, self.patch_mesh.f, f_color, shading={
                'wireframe': False
            })
        
        for pca_n_ring_on_patch_list in [pca_1_ring_on_patch_list, pca_2_ring_on_patch_list, pca_3_ring_on_patch_list, pca_4_ring_on_patch_list]:
            pca_val = []
            for p_id in patch_list:
                f_id = self.patch_mesh.graph.nodes[p_id]['f']
                pca_attribute = pca_n_ring_on_patch_list[p_id]
                pca_attribute = np.mean(pca_attribute, axis=0)
                pca_val.append(pca_attribute)
            
            pca_val = np.array(pca_val)
            ms = sklearn.cluster.MeanShift()
            ms.fit(pca_val)
            print(ms.cluster_centers_)
            pca_val_labels = ms.labels_
            show_cluster_by_val(pca_val_labels)
            
            min_val, max_val = np.min(pca_val, axis=0), np.max(pca_val, axis=0)
            pca_val_normalized = (pca_val - min_val) / (max_val - min_val)            
            ms = sklearn.cluster.MeanShift()
            ms.fit(pca_val_normalized)
            print(ms.cluster_centers_)
            pca_normalized_val_labels = ms.labels_
            show_cluster_by_val(pca_normalized_val_labels)
            
            # pca_val = np.array(pca_val)
            # ms = sklearn.cluster.MeanShift()
            # ms.fit(pca_val[:, 2])
            # print(ms.cluster_centers_)
            # pca_val_labels = ms.labels_
            # show_cluster_by_val(pca_val_labels)
            
            # min_val, max_val = np.min(pca_val, axis=0), np.max(pca_val, axis=0)
            # pca_val_normalized = (pca_val - min_val) / (max_val - min_val)
            # ms = sklearn.cluster.MeanShift()
            # ms.fit(pca_val_normalized[:, 2])
            # print(ms.cluster_centers_)
            # pca_normalized_val_labels = ms.labels_
            # show_cluster_by_val(pca_normalized_val_labels)
            
            ms = sklearn.cluster.MeanShift(bandwidth=0.5)
            ms.fit(pca_val_normalized)
            print(ms.cluster_centers_)
            pca_band_labels = ms.labels_
            show_cluster_by_val(pca_band_labels)

            a = 1


    '''
        compute dihedral attribute of all patches
    '''
    def compute_patch_dihedral_attribute(self, patch_list, visualize=False):
        
        om_mesh = self.patch_mesh.om_mesh
        f_normal = igl.per_face_normals(self.patch_mesh.v, self.patch_mesh.f, np.array([1., 0., 0.]))
        def get_n_ring_neighbot_f(f_id):
            nf_1_ring = set()
            f_h = om_mesh.face_handle(f_id)
            for n_f_h in om_mesh.ff(f_h):
                nf_1_ring.add(n_f_h.idx())
            nf_1_ring = list(nf_1_ring)
            return nf_1_ring
        

    ''' 
        compute attribute of patch
    '''
    def compute_patch_attribute(self, p, visualize=False):
        om_mesh = self.patch_mesh.om_mesh
        p_faces = self.patch_mesh.graph.nodes[p]['f']
        num_p_faces = len(p_faces)
        
        def get_n_ring_neighbot_f(f_id):
            nf_1_ring = set()
            nf_2_ring = set()
            nf_3_ring = set()
            f_h = om_mesh.face_handle(f_id)
            for n_f_h in om_mesh.ff(f_h):
                nf_1_ring.add(n_f_h.idx())
                for nn_f_h in om_mesh.ff(n_f_h):
                    nf_2_ring.add(nn_f_h.idx())
                    for nnn_f_h in om_mesh.ff(nn_f_h):
                        nf_3_ring.add(nnn_f_h.idx())
            nf_1_ring = list(nf_1_ring)
            nf_2_ring = list(nf_2_ring)
            nf_3_ring = list(nf_3_ring)
            return nf_1_ring, nf_2_ring, nf_3_ring
        
        def pca(vectors):
            centroid = np.mean(vectors, axis=0)
            centroid_vectors = vectors - centroid
            cov = np.cov(centroid_vectors.T)
            eigen_value, eigen_vector = np.linalg.eig(cov)
            
            return eigen_value
        
        # pca
        f_normal = igl.per_face_normals(self.patch_mesh.v, self.patch_mesh.f, np.array([1., 0., 0.]))
        
        pca_1_ring = []
        pca_2_ring = []
        pca_3_ring = []
        
        for f_id in p_faces:
            nf_1_ring, nf_2_ring, nf_3_ring = get_n_ring_neighbot_f(f_id)
            nf_1_ring_normal = f_normal[nf_1_ring]
            nf_1_ring_normal = np.array(nf_1_ring_normal).reshape((-1, 3))
            
            nf_1_ring_normal_pca = pca(nf_1_ring_normal)
            pca_1_ring.append(nf_1_ring_normal_pca)
        pca_1_ring = np.array(pca_1_ring)
        # pca_2_ring = np.array(pca_2_ring)

        
        # show value
        if visualize:
            f_color = np.ones((self.patch_mesh.f.shape[0], 3))
            for j in range(3):
                min_val, max_val = np.min(pca_1_ring[:, j]), np.max(pca_1_ring[:, j])
                if max_val != min_val:
                    for i in range(num_p_faces):
                        f_id = p_faces[i]
                        f_color[f_id, j] = (pca_1_ring[i, j] - min_val) / (max_val - min_val)
                else:
                    for i in range(num_p_faces):
                        f_color[p_faces[i], j] = 0.5
            mp.plot(self.patch_mesh.v, self.patch_mesh.f, c=f_color, shading={
                'wireframe': False
            })
        
        return pca_1_ring

    
    '''
        get type of all patches
        INPUT
            - patch2f
        OUTPUT
            - type of all patches
    '''
    def get_patch_type(self):
        patch_type = {}
        fitting = PrimitiveFitting(self.patch_mesh.v, self.patch_mesh.f, self.patch_mesh.f_normal, self.patch_mesh.f_center, self.patch_mesh.f_area)
        for patch_id in self.patch_mesh.graph.nodes:
            f_id = np.array(self.patch_mesh.patch2f[patch_id])

            # if patch_id == 86:
            #     print(' ')

            f_id = np.array(self.patch_mesh.graph.nodes[patch_id]['f'])
            # mp.plot(self.patch_mesh.v, self.patch_mesh.f[f_id], shading={'wireframe': True})

            planar_rate, planar_params = fitting.fit_planar(f_id)
            if (planar_rate > 0.9):
                # print('PLANAR')
                patch_type[patch_id] = {
                    "type": "Plane",
                    "params": planar_params
                }
                continue

            cylinder_rate, cylinder_params = fitting.fit_cylinder(f_id)
            if (cylinder_rate > 0.8):
                # print('CYLINDER')
                patch_type[patch_id] = {
                    "type": "Cylinder",
                    "params": cylinder_params
                }
                continue

            sphere_rate, sphere_params = fitting.fit_sphere(f_id)
            if (sphere_rate > 0.9):
                print('SPHERE')
                patch_type[patch_id] = {
                    "type": "Sphere",
                    "params": sphere_params
                }
                continue

            cone_rate, cone_params = fitting.fit_cone(f_id)
            if cone_rate > 0.9:
                patch_type[patch_id] = {
                    'type': 'Cone',
                    'params': cone_params
                }

            # print('OTHERS')
            patch_type[patch_id] = {
                "type": "Others",
                "params": []
            }

        nx.set_node_attributes(self.patch_mesh.graph, patch_type)


    def show_patch_type(self):
        f_color = np.ones((len(self.patch_mesh.f), 3))
        # plot = mp.plot(self.patch_mesh.v, self.patch_mesh.f, np.ones((self.patch_mesh.f.shape[0], 3)), shading={
        #     'wireframe': False
        # })
        for p_id in self.patch_mesh.graph.nodes:
            p_type = self.patch_mesh.graph.nodes[p_id]["type"]
            f_id = f_id = self.patch_mesh.patch2f[p_id]
            if p_type == "Plane":
                f_color[f_id] = [1., 0., 0.]
            elif p_type == 'Cylinder':
                f_color[f_id] = [0., 1., 0.]
            elif p_type == 'Sphere':
                f_color[f_id] = [1., 1., 0.]
            else:
                f_color[f_id] = [0., 0., 1.]

            # print(p_id)
            # plot.update_object(colors=f_color)    
    
        mp.plot(self.patch_mesh.v, self.patch_mesh.f, c=f_color, shading={
            'wireframe': False
        })



    '''
        Merge Condition:
            1. same type and nearly same param
            2. diff type: try fitting
    '''
    def combine_patches_by_fitting(self, visualize=False):
        if visualize:
            print('before merge:')
            f_colors = np.ones((self.patch_mesh.f.shape[0], 3))
            for p in self.patch_mesh.graph.nodes:
                f_colors[self.patch_mesh.patch2f[p]] = colors[p%500]
            mp.plot(self.patch_mesh.v, self.patch_mesh.f, c=f_colors, shading={'wireframe':False})

        G = self.patch_mesh.graph
        p_plane, p_cylinder, p_others = set(), set(), set()
        for p in G.nodes:
            p_node = G.nodes[p]
            if p_node['type'] == "Plane":
                p_plane.add(p)
            elif p_node['type'] == "Cylinder":
                p_cylinder.add(p)
            elif p_node['type'] == "Others":
                p_others.add(p)
        fitting = PrimitiveFitting(self.patch_mesh.v, self.patch_mesh.f, self.patch_mesh.f_normal, self.patch_mesh.f_center, self.patch_mesh.f_area)
        
        merging = True
        while merging:
            merging = False
            for p_type in [p_plane, p_cylinder]:
                for p in p_type:
                    for n_p in list(G.neighbors(p)):
                        # if not patch_in_same_region(self.patch_mesh.region2patch, p, n_p):
                        #     continue

                        if G[p][n_p]['border_type'] != 3:
                            continue

                        merged_f_set = np.array(list(set(G.nodes[p]['f']).union(set(G.nodes[n_p]['f']))))
                        planar_rate, planar_params = fitting.fit_planar(merged_f_set)
                        cylinder_rate, cylinder_params = fitting.fit_cylinder(merged_f_set)

                        if (planar_rate > 0.9):
                            G.nodes[p]['type'] = 'Plane'
                            G.nodes[p]['params'] = planar_params
                            G = self.combine_patches([[p, n_p]], G)
                            # for key in G.nodes[p]:
                            #     if (key == 'f') | (key == 'v'):
                            #         G.nodes[p][key].extend(G.nodes[n_p][key])
                            # for n_n_p in list(G.neighbors(n_p)):
                            #     if n_n_p == p:
                            #         continue
                            #     G.add_edge(p, n_n_p)
                            # G.remove_node(n_p)

                            
                            p_type.remove(p)
                            for n_p_type in [p_plane, p_cylinder, p_others]:
                                if n_p in n_p_type:
                                    n_p_type.remove(n_p)
                                    break
                            p_plane.add(p)
                            
                            merging = True
                            break

                        if (cylinder_rate > 0.9):
                            G.nodes[p]['type'] = 'Cylinder'
                            G.nodes[p]['params'] = cylinder_params
                            G = self.combine_patches([[p, n_p]], G)
                            # for key in G.nodes[p]:
                            #     if (key == 'f') | (key == 'v'):
                            #         G.nodes[p][key].extend(G.nodes[n_p][key])
                            # for n_n_p in list(G.neighbors(n_p)):
                            #     if n_n_p == p:
                            #         continue
                            #     G.add_edge(p, n_n_p)
                            # G.remove_node(n_p)

                            
                            p_type.remove(p)
                            for n_p_type in [p_plane, p_cylinder, p_others]:
                                if n_p in n_p_type:
                                    n_p_type.remove(n_p)
                                    break
                            p_cylinder.add(p)
                            
                            merging = True
                            break
                        
                    if merging:
                        break
                if merging:
                    break
              
        self.patch_mesh.graph = G
        # update v2patch
        self.patch_mesh.v2patch = defaultdict(list)
        for p_id in self.patch_mesh.patch2f:
            f_id = self.patch_mesh.patch2f[p_id]
            v_id = np.unique(self.patch_mesh.f[f_id].flatten())
            for tmp_v_id in v_id:
                self.patch_mesh.v2patch[tmp_v_id].append(p_id)

        
        # update patch2f
        # self.patch_mesh.patch2f = {}
        # for p in G.nodes:
        #     self.patch_mesh.patch2f[p] = G.nodes[p]['f']
        
            
        # # update patch-edge v
        # update_function = getattr(self.patch_mesh, 'update_graph_edge', None)
        # if callable(update_function):
        #     self.patch_mesh.update_graph_edge()

        if visualize:
            print('after merge:')
            f_colors = np.ones((self.patch_mesh.f.shape[0], 3))
            for p in G.nodes:
                f_colors[self.patch_mesh.patch2f[p]] = colors[p%500]
            mp.plot(self.patch_mesh.v, self.patch_mesh.f, c=f_colors, shading={'wireframe':False})

    
    def combine_patches_by_border(self, visualize=False):
        combine_pairs = []
        G = self.patch_mesh.graph
        for edge in G.edges:
            p1, p2 = edge
            border_type = G[p1][p2]["border_type"]
            # if (border_type == 1) | (border_type == 2):
            if (border_type == 1):
                continue
            patch_type1, patch_type2 = G.nodes[p1]["type"], G.nodes[p2]["type"]
            
            # if (patch_type1 == "Plane") & (patch_type2 == "Plane"):
            #     if (np.linalg.norm(G.nodes[p1]["params"] - G.nodes[p2]["params"]) < 1e-2) or (np.linalg.norm(G.nodes[p1]["params"] + G.nodes[p2]["params"]) < 1e-2):
            #         if p2 < p1:
            #             p1, p2 = p2, p1
            #         combine_pairs.append([p1, p2])
            #         continue
                
            if (patch_type1 == "Plane") | (patch_type2 == "Plane"):
                continue
            if p2 < p1:
                p1, p2 = p2, p1
            combine_pairs.append([p1, p2])
        combine_pairs = np.array(combine_pairs)
        # print(combine_pairs.shape)

        for i in range(combine_pairs.shape[0]):
            edge = combine_pairs[i]
            p1, p2 = edge
            if p1 == p2:
                continue
            if p2 < p1:
                p1, p2 = p2, p1
            
            # combine p2 into p1
            # update graph
            for key in G.nodes[p1]:
                if (key == 'f') | (key == 'v'):
                    G.nodes[p1][key].extend(G.nodes[p2][key])
            # update patch2f, f2patch
            
            for f_id in G.nodes[p2]['f']:
                self.patch_mesh.f2patch[f_id] = p1
            # for n_p in list(G.neighbors(p2)):
            #     self.patch_mesh.p_p_adj[p1][n_p].extend(self.patch_mesh.p_p_adj[p2][n_p])
            
            for n_p in list(G.neighbors(p2)):
                if n_p == p1:
                    continue
                G.add_edge(p1, n_p)
                
            G.remove_node(p2)
            combine_pairs[combine_pairs == p2] = p1
        
        # update patch2f
        self.patch_mesh.patch2f = {}
        for p in G.nodes:
            self.patch_mesh.patch2f[p] = G.nodes[p]['f']
        
            
        # update patch-edge v
        update_function = getattr(self.patch_mesh, 'update_graph_edge', None)
        if callable(update_function):
            self.patch_mesh.update_graph_edge()

        if visualize:
            print('after merge:')
            f_colors = np.ones((self.patch_mesh.f.shape[0], 3))
            for p in G.nodes:
                f_colors[self.patch_mesh.patch2f[p]] = colors[p%500]
            mp.plot(self.patch_mesh.v, self.patch_mesh.f, c=f_colors, shading={'wireframe':False})


    def combine_plane_patches(self, visualize=False):
        G = self.patch_mesh.graph
        combine_pairs = []
        for edge in self.patch_mesh.graph.edges:
            p1, p2 = edge
            if (self.patch_mesh.graph.nodes[p1]["type"] == "Plane") & (self.patch_mesh.graph.nodes[p2]["type"] == "Plane"):
                fitting_primitive = PrimitiveFitting(self.patch_mesh.v, self.patch_mesh.f, self.patch_mesh.f_normal, self.patch_mesh.f_center, self.patch_mesh.f_area)
                f1 = self.patch_mesh.graph.nodes[p1]["f"]
                f2 = self.patch_mesh.graph.nodes[p2]["f"]
                merged_f_set = np.array(list(set(f1).union(set(f2))))
                planar_rate, _ = fitting_primitive.fit_planar(merged_f_set)
                if planar_rate > 0.9:
                    if p2 < p1:
                        p1, p2 = p2, p1
                    combine_pairs.append([p1, p2])

        for pair in combine_pairs:
            p1, p2 = pair
            if p1 == p2:
                continue

            # combine p2 into p1
            # update graph
            for key in G.nodes[p1]:
                if (key == 'f') | (key == 'v'):
                    G.nodes[p1][key].extend(G.nodes[p2][key])
            # update patch2f, f2patch
            for f_id in G.nodes[p2]['f']:
                self.patch_mesh.f2patch[f_id] = p1
            # for n_p in list(G.neighbors(p2)):
            #     self.patch_mesh.p_p_adj[p1][n_p].extend(self.patch_mesh.p_p_adj[p2][n_p])
            
            for n_p in list(G.neighbors(p2)):
                if n_p == p1:
                    continue
                G.add_edge(p1, n_p)
                
            G.remove_node(p2)
        
        # update patch2f
        self.patch_mesh.patch2f = {}
        for p in G.nodes:
            self.patch_mesh.patch2f[p] = G.nodes[p]['f']
        
            
        # update patch-edge v
        update_function = getattr(self.patch_mesh, 'update_graph_edge', None)
        if callable(update_function):
            self.patch_mesh.update_graph_edge()

        if visualize:
            print('after merge:')
            f_colors = np.ones((self.patch_mesh.f.shape[0], 3))
            for p in G.nodes:
                f_colors[self.patch_mesh.patch2f[p]] = colors[p%500]
            mp.plot(self.patch_mesh.v, self.patch_mesh.f, c=f_colors, shading={'wireframe':False})


    '''
        Combine two patches p1, p2, p2 -> p1
        INPUT
            - (p1, p2)
            - node attributes
                - v, f
            - border type
                - patch<->f, patch->v
                - type of patch
                - vf, vv neighbor
        UPDATE
            - node attributes
                - v, f
                - type, params
            - neighbor attributes
                - connectivity: new neighbor = raw neighbor U (N(p2) \ p1)
                - border_type between p1, N(p2)\p1
            - patch <-> f, patch -> v
    '''
    def combine_patches(self, pairs, G, visualize=False):
        # G = self.patch_mesh.graph
        for pair in pairs:
            p0, p1 = pair

            # if visualize:
            #     print('before merge')
            #     patch2f = self.patch_mesh.patch2f
            #     f_color = np.ones((self.patch_mesh.f.shape[0], 3))
            #     for p_id in [p0, p1]:
            #         f_color[patch2f[p_id]] = colors[p_id % 500]
            #     mp.plot(self.patch_mesh.v, self.patch_mesh.f, c=f_color, shading={
            #         'wireframe': False
            #     })
            # if p0 > p1:
            #     p0, p1 = p1, p0

            # combine p1 into p0
            # merge v, f
            for key in G.nodes[p1]:
                if (key == 'f') | (key == 'v'):
                    G.nodes[p0][key].extend(G.nodes[p1][key])

            # update f -> patch, patch -> v
            self.patch_mesh.f2patch[G.nodes[p1]['f']] = p0
            self.patch_mesh.patch2v[p0].extend(G.nodes[p1]['v'])

    
            # update neighbor
            for n_p in list(G.neighbors(p1)):
                if n_p == p0:
                    continue
                G.add_edge(p0, n_p)
                G.edges[p0, n_p]["border_type"] = self.get_border_type(p0, n_p, G)


            G.remove_node(p1)


        # update self.patch_mesh.patch2f
            self.patch_mesh.patch2f[p0] = G.nodes[p0]['f']
        # for p in G.nodes:
        #     self.patch_mesh.patch2f[p] = G.nodes[p]['f']
        # self.patch_mesh.graph = G
            if visualize:
                print('after merge')
                patch2f = self.patch_mesh.patch2f
                f_color = np.ones((self.patch_mesh.f.shape[0], 3))
                f_color[patch2f[p0]] = colors[p0 % 500]
                mp.plot(self.patch_mesh.v, self.patch_mesh.f, c=f_color, shading={
                    'wireframe': False
                })

        return G


    def detect_patch_border_v_circle(self, p1, p2):
        p1_f = self.patch_mesh.patch2f[p1]
        p2_f = self.patch_mesh.patch2f[p2]

        # # 1. edge to neighbor face
        # edge2face = defaultdict(list)
        # for f_id in p1_f:
        #     f = self.patch_mesh.f[f_id]
        #     for i in range(3):
        #         v0, v1 = f[i], f[(i + 1) % 3]
        #         if v0 > v1:
        #             v0, v1 = v1, v0
        #         edge2face[(v0, v1)].append(f_id)
        # for f_id in p2_f:
        #     f = self.patch_mesh.f[f_id]
        #     for i in range(3):
        #         v0, v1 = f[i], f[(i + 1) % 3]
        #         if v0 > v1:
        #             v0, v1 = v1, v0
        #         edge2face[(v0, v1)].append(f_id)

        # # 2. get border edges: num(n_f) == 2
        # border_edges = []
        # for edge, f_id in edge2face.items():
        #     if len(f_id) == 2:
        #         f1, f2 = f_id
        #         if (f1 in p1_f and f2 in p2_f) or (f1 in p2_f and f2 in p1_f):
        #             border_edges.append([edge[0], edge[1]])

        def face_edges(face):
            return {tuple(sorted((face[i], face[(i+1) % len(face)]))) for i in range(len(face))}

        edges1 = set()
        for fi in p1_f:
            edges1 |= face_edges(self.patch_mesh.f[fi])
        edges2 = set()
        for fi in p2_f:
            edges2 |= face_edges(self.patch_mesh.f[fi])

        edges = list(edges1 & edges2)

        return detect_circle_in_edge(edges), edges
        

    def get_patch_border_v(self, p1, p2):
        p1_f = self.patch_mesh.patch2f[p1]
        p2_f = self.patch_mesh.patch2f[p2]

        def face_edges(face):
            return {tuple(sorted((face[i], face[(i+1) % len(face)]))) for i in range(len(face))}
        edges1 = set()
        for fi in p1_f:
            edges1 |= face_edges(self.patch_mesh.f[fi])
        edges2 = set()
        for fi in p2_f:
            edges2 |= face_edges(self.patch_mesh.f[fi])
        border_edges = np.array(list(edges1 & edges2))

        # 3. order border vertices
        edge_ordered, vert_ordered = connect_edge_to_line(border_edges)

        return vert_ordered
        # return vert_ordered[0]



    def cut_mesh_by_plane_tracing(self, plane_params, v_from, v_to, v_border):
        mesh = self.patch_mesh.om_mesh

        '''
            do
                target visit faces
                if v on vertex:
                    get all 'vv, ve' in neighbor faces
                if v on edge:
                    get all 'ev, ee' in neighbor face
                remove duplicate until one 
            until
                next v == v_to
        ''' 
        v_from_id = v_from
        v_from_pos = mesh.point(mesh.vertex_handle(v_from_id))
        v_from_type = 'V'
        f_visit = set()
        # is_find_next = True
        prev_v_pos = v_from_pos
        prev_v_id = v_from_id

        insert_v = []
        insert_v_num = -1
        raw_v_num = self.patch_mesh.v.shape[0]
        insert_f = defaultdict(list)
        # remove_f = []
        
        f_colors = np.ones((self.patch_mesh.f.shape[0], 3))
        plot = mp.plot(self.patch_mesh.v, self.patch_mesh.f, c=f_colors, shading={
            'wireframe': True
        })
        cutting_border_vert = []
        for i in range(len(v_border) - 1):
            cutting_border_vert.append([v_border[i], v_border[i + 1]])
        
        # first direction: vertex to edge

        while True:
            if v_from_id == v_to:
                print('Reach v_border')
                break
            plot.add_lines(
                np.array([prev_v_pos]),
                np.array([v_from_pos]),
                shading={
                    "line_color": "red",
                    "line_width": 1
                }
            )
            plot.update_object()

            if v_from_type == 'V':
                v_from_h = mesh.vertex_handle(v_from_id)
            else:
                v_from_h = mesh.edge_handle(v_from_id)

            insert_type = None
            if v_from_type == 'V':
                neighbor_f = []
                for f_h in mesh.vf(v_from_h):
                    f_id = f_h.idx()
                    if f_id not in f_visit:
                        neighbor_f.append(f_id)
                        f_visit.add(f_id)
                
                next_v_info = []
                for f_id in neighbor_f:
                    result = insection_plane_f(mesh, plane_params, mesh.face_handle(f_id), v_from_pos)
                    # f_colors = np.ones((self.patch_mesh.f.shape[0], 3))
                    # f_colors[f_id] = [1, 0, 0]
                    # plot.update_object(colors=f_colors)
                    if (result['type'] == 'vv') or (result['type'] == 've'):
                        next_v_info.append(result)
                if len(next_v_info) == 0:
                    print('Cannot find next intersect face')
                    break

                # get right next vert
                # 1. ve > vv
                # 2. remove vert in v_border
                # 3. choose the duplicate one
                next_v_info_id = -1
                next_v_set = defaultdict(list)
                for i in range(len(next_v_info)):
                    tmp_next_v_info = next_v_info[i]
                    if tmp_next_v_info['type'] == 've':
                        next_v_info_id = i
                        break
                    elif tmp_next_v_info['next_v_id'] == v_to:
                        next_v_info_id = i
                        break                       
                    elif tmp_next_v_info['next_v_id'] in v_border:
                        continue
                    else:
                        tmp_next_v_id = tmp_next_v_info['next_v_id']
                        if tmp_next_v_id not in next_v_set:
                            next_v_info_id = i
                            next_v_set[tmp_next_v_id].append(tmp_next_v_info)
                        else:
                            next_v_info_id = i
                            break
                if next_v_info_id == -1:
                    raise('[Trace Cut]: cannot find next v id')
                next_v_info = next_v_info[next_v_info_id]
                # next_v_info = next_v_info[0]

                if next_v_info['type'] == 'vv':
                    prev_v_id = v_from_id
                    prev_v_pos = v_from_pos
                    v_from_id = next_v_info['next_v_id']
                    v_from_pos = mesh.point(mesh.vertex_handle(v_from_id))
                    v_from_type = 'V'
                    f_id = next_v_info['f_id']

                    insert_type = 'vv'
                    cutting_border_vert.append([prev_v_id, v_from_id])
                else:
                    prev_v_id = v_from_id
                    prev_v_pos = v_from_pos
                    v_from_id = next_v_info['next_e_id']
                    v_from_pos = next_v_info['v_insert']
                    v_from_type = 'E'
                    f_id = next_v_info['f_id']
                    
                    insert_type = 've'
                    insert_v.append(v_from_pos)
                    insert_v_num += 1
                    insert_v_id = raw_v_num + insert_v_num
                    cutting_border_vert.append([prev_v_id, insert_v_id])
                    

            elif v_from_type == 'E':
                neighbor_f = -1
                for f_id in e_h2f(mesh, v_from_h):
                # for f_h in mesh.ef(v_from_h):
                    # f_id = f_h.idx()
                    if f_id not in f_visit:
                        neighbor_f = f_id
                        f_visit.add(f_id)
                if neighbor_f == -1:
                    print('Cannot find next not visited face')
                    break
                
                result = insection_plane_f(mesh, plane_params, mesh.face_handle(neighbor_f), v_from_pos)

                if result['type'] == 'ev':
                    prev_v_id = v_from_id
                    prev_v_pos = v_from_pos
                    v_from_id = result['next_v_id']
                    v_from_pos = mesh.point(mesh.vertex_handle(v_from_id))
                    v_from_type = 'V'
                    f_id = result['f_id']

                    insert_type = 'ev'
                    prev_insert_v_id = raw_v_num + insert_v_num
                    cutting_border_vert.append([prev_insert_v_id, v_from_id])

                elif result['type'] == 'ee':
                    prev_v_id = v_from_id
                    prev_v_pos = v_from_pos
                    v_from_id = result['to_e_id']
                    v_from_pos = result['v_insert']
                    v_from_type = 'E'
                    f_id = result['f_id']

                    insert_type = 'ee'
                    prev_insert_v_id = raw_v_num + insert_v_num
                    insert_v.append(v_from_pos)
                    insert_v_num += 1
                    insert_v_id = raw_v_num + insert_v_num
                    cutting_border_vert.append([prev_insert_v_id, insert_v_id])
            

            # insert faces
            if insert_type == 'vv':
                pass
            elif insert_type == 've':
                raw_face = self.patch_mesh.f[f_id]
                pre_v_index = np.where(raw_face == prev_v_id)[0][0]

                insert_f[f_id].append([
                    raw_face[pre_v_index],
                    raw_face[(pre_v_index + 1) % 3],
                    insert_v_id
                ])
                insert_f[f_id].append([
                    raw_face[(pre_v_index - 1) % 3],
                    raw_face[pre_v_index],
                    insert_v_id
                ])
            elif insert_type == 'ev':
                raw_face = self.patch_mesh.f[f_id]
                pre_v_index = np.where(raw_face == v_from_id)[0][0]

                insert_f[f_id].append([
                    raw_face[pre_v_index],
                    raw_face[(pre_v_index + 1) % 3],
                    prev_insert_v_id
                ])
                insert_f[f_id].append([
                    raw_face[(pre_v_index - 1) % 3],
                    raw_face[pre_v_index],
                    prev_insert_v_id
                ])
            elif insert_type == 'ee':
                raw_face = self.patch_mesh.f[f_id]
                
                prev_e_id = prev_v_id
                cur_e_id = v_from_id
                op_v_id0 = e_h2ov_in_face(mesh, mesh.edge_handle(prev_e_id), raw_face)
                op_v_index0 = np.where(raw_face == op_v_id0)[0][0]
                op_v_id1 = e_h2ov_in_face(mesh, mesh.edge_handle(cur_e_id), raw_face)
                op_v_index1 = np.where(raw_face == op_v_id1)[0][0]

        
                if op_v_index1 == (op_v_index0 + 1) % 3:
                    insert_f[f_id].append([
                        raw_face[op_v_index0],
                        raw_face[(op_v_index0 + 1) % 3],
                        prev_insert_v_id
                    ])
                    insert_f[f_id].append([
                        prev_insert_v_id,
                        raw_face[(op_v_index0 - 1) % 3],
                        insert_v_id
                    ])
                    insert_f[f_id].append([
                        raw_face[op_v_index0],
                        prev_insert_v_id,
                        insert_v_id
                    ])
                else:
                    insert_f[f_id].append([
                        raw_face[(op_v_index0 - 1) % 3],
                        raw_face[op_v_index0],
                        prev_insert_v_id
                    ])
                    insert_f[f_id].append([
                        raw_face[(op_v_index0 + 1) % 3],
                        prev_insert_v_id,
                        insert_v_id
                    ])
                    insert_f[f_id].append([
                        raw_face[op_v_index0],
                        insert_v_id,
                        prev_insert_v_id
                    ])
            
        for f_id in insert_f.keys():
            insert_f[f_id] = np.array(insert_f[f_id])

        # update vertices
        insert_v = np.array(insert_v)
        new_mesh_vertices = np.concatenate((self.patch_mesh.v, insert_v), axis=0)       

        # remove old faces & insert new faces with patch id
        remove_f = list(insert_f.keys())
        new_mesh_faces = np.delete(self.patch_mesh.f, remove_f, axis = 0)
        new_f_id2raw_f_id = [i for i in range(len(self.patch_mesh.f)) if i not in remove_f]
        raw_f_len = new_mesh_faces.shape[0]
        
        new_f_p_id = {}
        new_f_id = raw_f_len
        for raw_f_id in insert_f.keys():
            for _ in range(insert_f[raw_f_id].shape[0]):
                new_f_p_id[new_f_id] = self.patch_mesh.f2patch[raw_f_id]
                new_f_id += 1
            new_mesh_faces = np.concatenate((new_mesh_faces, insert_f[raw_f_id]))
        
        # check
        plot = mp.plot(new_mesh_vertices, new_mesh_faces, np.ones((new_mesh_faces.shape[0], 3)), shading={'wireframe': True})
        cutting_border_vert_array = np.array(cutting_border_vert)
        plot.add_lines(
            new_mesh_vertices[cutting_border_vert_array[:, 0]],
            new_mesh_vertices[cutting_border_vert_array[:, 1]],
            shading={
                "line_color": "red",
                "line_width": 1
            }
        )

        # 



        # split mesh
        # noting the insert edge, and split by connectivity of face Graph
        cutting_border_vert_ordered = []
        for v_pair in cutting_border_vert:
            v0_id, v1_id = v_pair
            if v0_id > v1_id:
                cutting_border_vert_ordered.append([v1_id, v0_id])
            else:
                cutting_border_vert_ordered.append([v0_id, v1_id])
        # cutting_border_vert = np.array(cutting_border_vert)
        # cutting_mesh = om.TriMesh(new_mesh_vertices, new_mesh_faces)
        # cutting_border_edge = []
        # for i in range(cutting_border_vert.shape[0]):
        #     v0_id, v1_id = cutting_border_edge[i]
        #     cutting_border_edge.append(v_pair2e(cutting_mesh, v0_id, v1_id))
        
        cutting_graph = nx.Graph()
        # add node
        for i in range(new_mesh_faces.shape[0]):
            cutting_graph.add_node(i)
        # edge to face
        edge2face = defaultdict(list)
        for f_id in range(new_mesh_faces.shape[0]):
            for i in range(3):
                v0, v1 = new_mesh_faces[f_id][i], new_mesh_faces[f_id][(i + 1) % 3]
                if v0 > v1:
                    v0, v1 = v1, v0
                edge2face[(v0, v1)].append(f_id)
        # add edges
        for edge, n_f in edge2face.items():
            if len(n_f) == 2:
                if [edge[0], edge[1]] in cutting_border_vert_ordered:
                    continue
                cutting_graph.add_edge(n_f[0], n_f[1])
        components = list(nx.connected_components(cutting_graph))
        if len(components) != 2:
            raise ValueError("SPLIT ERROR]: tracing cut graph components != 2")
        f_in, f_out = list(components[0]), list(components[1])


        # # show cutting plane
        # self.plot_mesh_cut_with_plane(
        #     new_mesh_vertices, new_mesh_faces, f_out, f_in, cutting_border_vert_array[0, :].flatten(),
        #     plane_params,
        #     plane_alpha=0.1
        # )
        a=1
        self.plot_mesh_with_cutting_plane(
            new_mesh_vertices,
            # new_mesh_faces[f_in],
            new_mesh_faces[f_out],
            new_mesh_faces[f_in],
            plane_params,
            color_A=(0.55, 0.78, 0.98),  # 近似浅蓝
            color_B=(0.15, 0.68, 0.2),   # 近似绿色
            plane_color=(0.98, 0.92, 0.70),  # 淡黄
            plane_alpha=0.5,
            edge_alpha=0.15,
            linewidth=0.2,
            figsize=(10, 8),
            margin_ratio=0.10,
            show_reference_arrow=False
        )

        # convert patch
        patch2f_in = defaultdict(list)
        for f_id in f_in:
            if f_id >= raw_f_len:
                p_id = new_f_p_id[f_id]
            else:
                raw_f_id = new_f_id2raw_f_id[f_id]
                p_id = self.patch_mesh.f2patch[raw_f_id]
            patch2f_in[p_id].append(f_id)
        patch2f_in = [patch2f_in[p_id] for p_id in patch2f_in]

        for i in range(len(patch2f_in)):
            for j in range(len(patch2f_in[i])):
                if patch2f_in[i][j] in f_in:
                    patch2f_in[i][j] = f_in.index(patch2f_in[i][j])


        patch2f_out = defaultdict(list)
        for f_id in f_out:
            if f_id >= raw_f_len:
                p_id = new_f_p_id[f_id]
            else:
                raw_f_id = new_f_id2raw_f_id[f_id]
                p_id = self.patch_mesh.f2patch[raw_f_id]
            patch2f_out[p_id].append(f_id)
        patch2f_out = [patch2f_out[p_id] for p_id in patch2f_out]

        for i in range(len(patch2f_out)):
            for j in range(len(patch2f_out[i])):
                if patch2f_out[i][j] in f_out:
                    patch2f_out[i][j] = f_out.index(patch2f_out[i][j])
        

        # final split result
        v0, f0 = get_submesh(new_mesh_vertices, new_mesh_faces, f_in)
        # hole_filling0 = HoleFilling(v0, f0, visualize=False)
        # new_v0, new_f0, filled_regions = hole_filling0.new_v, hole_filling0.new_f, hole_filling0.region_triangles
        # if len(filled_regions) != 0:
        #     cur_id = f0.shape[0]
        #     for region in filled_regions:
        #         patch2f_in.append(list(np.arange(cur_id, cur_id + region.shape[0])))
        #         cur_id += len(region)
        # v0, f0 = new_v0, new_f0
        # v0, f0 = triangulate_refine_fair(v0, f0)
        
    

        v1, f1 = get_submesh(new_mesh_vertices, new_mesh_faces, f_out)
        # hole_filling1 = HoleFilling(v1, f1, visualize=False)
        # new_v1, new_f1, filled_regions =  hole_filling1.new_v, hole_filling1.new_f, hole_filling1.region_triangles
        # if len(filled_regions) != 0:
        #     cur_id = f1.shape[0]
        #     for region in filled_regions:
        #         patch2f_out.append(list(np.arange(cur_id, cur_id + region.shape[0])))
        #         cur_id += len(region)
        # v1, f1 = new_v1, new_f1
        # v1, f1 = triangulate_refine_fair(v1, f1)
        

        mp.plot(v0, f0, np.ones((f0.shape[0], 3)), shading={'wireframe': True})
        mp.plot(v1, f1, np.ones((f1.shape[0], 3)), shading={'wireframe': True})

        return True, [
            {
                'v': v0, 'f': f0, 'patch': patch2f_in
            },
            {
                'v': v1, 'f': f1, 'patch': patch2f_out
            }
        ]
    

    def plot_mesh_with_cutting_plane(self,
        V, F_A, F_B, plane,
        color_A=(0.55, 0.78, 0.98),  # 近似浅蓝
        color_B=(0.15, 0.68, 0.2),   # 近似绿色
        plane_color=(0.98, 0.92, 0.70),  # 淡黄
        plane_alpha=0.5,
        edge_alpha=0.15,
        linewidth=0.2,
        figsize=(10, 8),
        margin_ratio=0.10,
        show_reference_arrow=False
    ):  
        # ---------- helpers ----------
        def _plane_from_input(plane):
            """
            支持两种输入：
            1) (A, B, C, D) -> Ax + By + Cz + D = 0
            2) {'point': p0, 'normal': n}
            返回 (p0, n_unit)
            """
            if isinstance(plane, (list, tuple, np.ndarray)) and len(plane) == 4:
                A, B, C, D = map(float, plane)
                n = np.array([A, B, C], dtype=float)
                n_norm = np.linalg.norm(n)
                if n_norm == 0:
                    raise ValueError("Plane normal is zero in (A,B,C,D).")
                n = n / n_norm
                # 任取平面上一点：选择使得 Ax+By+Cz = -D
                # 把某两维设为0，第三维解出（优先选法向量最大分量对应坐标）
                idx = np.argmax(np.abs(n))
                p0 = np.zeros(3, dtype=float)
                p0[idx] = -D / n[idx]
                return p0, n
            elif isinstance(plane, dict) and 'point' in plane and 'normal' in plane:
                p0 = np.asarray(plane['point'], dtype=float)
                n = np.asarray(plane['normal'], dtype=float)
                n_norm = np.linalg.norm(n)
                if n_norm == 0:
                    raise ValueError("Plane normal is zero in {'point','normal'}.")
                n = n / n_norm
                return p0, n
            else:
                raise ValueError("Unsupported plane format. Use (A,B,C,D) or {'point':p0,'normal':n}.")

        def _orthonormal_basis_from_normal(n):
            """给定单位法向 n，返回平面内一组正交单位基 (u, v)。"""
            # 取与 n 不平行的向量做叉乘
            a = np.array([1.0, 0.0, 0.0]) if abs(n[0]) < 0.9 else np.array([0.0, 1.0, 0.0])
            u = np.cross(n, a)
            u /= np.linalg.norm(u)
            v = np.cross(n, u)
            v /= np.linalg.norm(v)
            return u, v

        def _plane_polygon_covering_mesh(V, p0, n, margin_ratio=0.10):
            """
            在给定平面上构造一个覆盖模型包围盒的矩形。
            做法：把所有顶点投影到平面基 (u,v)，取 min/max 并外扩 margin。
            返回 4 个顶点的 3D 坐标（按环顺序）。
            """
            u, v = _orthonormal_basis_from_normal(n)
            # 顶点到平面基的坐标（平移到p0后再投影）
            rel = V - p0
            U = rel @ u
            Vv = rel @ v
            umin, umax = U.min(), U.max()
            vmin, vmax = Vv.min(), Vv.max()
            du, dv = umax - umin, vmax - vmin
            # 外扩 margin
            umin -= du * margin_ratio
            umax += du * margin_ratio
            vmin -= dv * margin_ratio
            vmax += dv * margin_ratio
            # 矩形四角（u,v 坐标系）
            corners_uv = np.array([
                [umin, vmin],
                [umax, vmin],
                [umax, vmax],
                [umin, vmax],
            ], dtype=float)
            # 还原到三维：p = p0 + u*u_ + v*v_
            corners_xyz = p0[None, :] + corners_uv[:, 0:1] * u[None, :] + corners_uv[:, 1:2] * v[None, :]
            return corners_xyz

        def _tri_faces_to_vertices(V, F):
            """把 (N,3) 顶点和 (M,3) 三角面索引转成 Poly3DCollection 需要的顶点列表"""
            return [V[tri] for tri in F]

        def _set_equal_aspect(ax, V):
            """3D等比例显示"""
            mins = V.min(axis=0)
            maxs = V.max(axis=0)
            ctr = (mins + maxs) / 2.0
            size = (maxs - mins).max()
            ax.set_xlim(ctr[0]-size/2, ctr[0]+size/2)
            ax.set_ylim(ctr[1]-size/2, ctr[1]+size/2)
            ax.set_zlim(ctr[2]-size/2, ctr[2]+size/2)
            ax.set_box_aspect([1, 1, 1])
        """
        渲染两部分网格 + 半透明切割平面。
        - plane: (A,B,C,D) 或 {'point':p0,'normal':n}
        """
        V = np.asarray(V, dtype=float)
        F_A = np.asarray(F_A, dtype=int)
        F_B = np.asarray(F_B, dtype=int)

        p0, n = _plane_from_input(plane)
        plane_quad = _plane_polygon_covering_mesh(V, p0, n, margin_ratio=margin_ratio)

        fig = plt.figure(figsize=figsize, dpi=150)
        ax = fig.add_subplot(111, projection='3d')

        # Part A
        poly_A = Poly3DCollection(_tri_faces_to_vertices(V, F_A))
        poly_A.set_facecolor((*color_A, 1.0))
        poly_A.set_edgecolor((0, 0, 0, edge_alpha))
        poly_A.set_linewidth(linewidth)
        ax.add_collection3d(poly_A)
        # Cutting plane (大矩形)
        plane_poly = Poly3DCollection([plane_quad])
        plane_poly.set_facecolor((*plane_color, plane_alpha))
        plane_poly.set_edgecolor((0.6, 0.6, 0.5, 0.6))
        plane_poly.set_linewidth(0.6)
        ax.add_collection3d(plane_poly)
        # Part B
        poly_B = Poly3DCollection(_tri_faces_to_vertices(V, F_B))
        poly_B.set_facecolor((*color_B, 1.0))
        poly_B.set_edgecolor((0, 0, 0, edge_alpha))
        poly_B.set_linewidth(linewidth)
        ax.add_collection3d(poly_B)



        _set_equal_aspect(ax, V)

        # 视觉风格：去坐标轴、边框淡化
        ax.set_axis_off()
        # 轻微视角
        ax.view_init(elev=22, azim=-55)

        # （可选）加一个“reference plane”箭头文字
        if show_reference_arrow:
            c = plane_quad.mean(axis=0)
            ax.text(c[0], c[1], c[2], "reference plane", zdir=None, fontsize=10)

        plt.tight_layout(pad=0)
        plt.show()

    
    def plot_mesh_with_cutting_cylinder(self,
        V, F_A, F_B,
        cylinder,                 # {'center': c, 'axis': a, 'radius': r, 'length': L(optional)}
        color_A=(0.55, 0.78, 0.98),
        color_B=(0.15, 0.68, 0.20),
        cyl_color=(0.98, 0.92, 0.70),
        cyl_alpha=0.45,
        edge_alpha=0,
        linewidth=0.2,
        seg=96,
        figsize=(10, 8),
        margin_ratio=0.10,
        add_caps=False,

        elev=20,
        azim=-60
    ):
        def _unit(v):
            v = np.asarray(v, dtype=float)
            n = np.linalg.norm(v)
            if n == 0: raise ValueError("zero-length vector")
            return v / n

        def _orthonormal_from_axis(a):
            a = _unit(a)
            t = np.array([1.0, 0.0, 0.0]) if abs(a[0]) < 0.9 else np.array([0.0, 1.0, 0.0])
            u = np.cross(a, t); u /= np.linalg.norm(u)
            v = np.cross(a, u); v /= np.linalg.norm(v)
            return a, u, v

        def _auto_cyl_span_along_axis(V, c, a, margin_ratio=0.10):
            """
            Project all vertices to axis direction 'a' and get [smin, smax] as cylinder span.
            Returns endpoints p0, p1 (center line ends).
            """
            rel = V - c[None, :]
            s = rel @ a
            smin, smax = s.min(), s.max()
            L = smax - smin
            smin -= L * margin_ratio
            smax += L * margin_ratio
            p0 = c + smin * a
            p1 = c + smax * a
            return p0, p1

        def _cylinder_surface_mesh(V, center, axis, radius, length=None, seg=72, margin_ratio=0.10, add_caps=False):
            """
            Build a triangle mesh for a (finite) cylinder that visually covers the mesh.
            If `length` is None, span is auto-chosen from V along axis with margin.
            center: any point on cylinder axis (3,)
            axis: cylinder axis direction (3,)
            radius: scalar
            length: scalar total length along axis, optional
            """
            a, u, v = _orthonormal_from_axis(axis)
            c = np.asarray(center, dtype=float)

            if length is None:
                p0, p1 = _auto_cyl_span_along_axis(V, c, a, margin_ratio)
            else:
                p0 = c - 0.5 * length * a
                p1 = c + 0.5 * length * a

            angles = np.linspace(0, 2*np.pi, seg, endpoint=False)
            ring0 = p0[None,:] + radius*np.cos(angles)[:,None]*u[None,:] + radius*np.sin(angles)[:,None]*v[None,:]
            ring1 = p1[None,:] + radius*np.cos(angles)[:,None]*u[None,:] + radius*np.sin(angles)[:,None]*v[None,:]

            Vc = np.vstack([ring0, ring1])
            F = []

            # lateral surface (as two triangles per quad)
            for i in range(seg):
                a0 = i
                a1 = (i+1) % seg
                b0 = seg + i
                b1 = seg + (i+1) % seg
                F += [[a0, a1, b1], [a0, b1, b0]]

            if add_caps:
                # optional discs for visual emphasis (kept very light)
                Vc = np.vstack([Vc, p0[None,:], p1[None,:]])
                id0, id1 = 2*seg, 2*seg+1
                for i in range(seg):
                    j = (i+1) % seg
                    F += [[id0, j, i]]
                    F += [[id1, seg+i, seg+j]]

            return Vc, np.asarray(F, dtype=int)
        V = np.asarray(V, dtype=float)
        F_A = np.asarray(F_A, dtype=int)
        F_B = np.asarray(F_B, dtype=int)

        # c = np.asarray(cylinder['center'], dtype=float)
        # a = np.asarray(cylinder['axis'], dtype=float)
        # r = float(cylinder['radius'])
        # L = float(cylinder['length']) if 'length' in cylinder and cylinder['length'] is not None else None
        c = cylinder[:3].reshape(-1)
        a = cylinder[3:6].reshape(-1)
        r = float(cylinder[6])
        L = None

        cyl_V, cyl_F = _cylinder_surface_mesh(
            V, c, a, r, length=L, seg=seg, margin_ratio=margin_ratio, add_caps=add_caps
        )

        fig = plt.figure(figsize=figsize, dpi=150)
        ax = fig.add_subplot(111, projection='3d')

        # Part A
        ax.add_collection3d(Poly3DCollection([V[f] for f in F_A],
            facecolors=(*color_A, 1.0), edgecolors=(0,0,0,edge_alpha), linewidths=linewidth))
        
        # Cutting cylinder (semi-transparent)
        ax.add_collection3d(Poly3DCollection([cyl_V[f] for f in cyl_F],
            facecolors=(*cyl_color, cyl_alpha), edgecolors=(0.,0.,0.,0.), linewidths=0.6))
        
        # Part B
        ax.add_collection3d(Poly3DCollection([V[f] for f in F_B],
            facecolors=(*color_B, 1.0), edgecolors=(0,0,0,edge_alpha), linewidths=linewidth))

        

        # equal aspect + clean look
        mins = V.min(axis=0); maxs = V.max(axis=0)
        ctr = (mins + maxs) / 2.0; size = (maxs - mins).max()
        ax.set_xlim(ctr[0]-size/2, ctr[0]+size/2)
        ax.set_ylim(ctr[1]-size/2, ctr[1]+size/2)
        ax.set_zlim(ctr[2]-size/2, ctr[2]+size/2)
        ax.set_box_aspect([1,1,1])
        ax.set_axis_off()
        ax.view_init(elev=elev, azim=azim)
        # ax.view_init(elev=42, azim=-150)
        plt.tight_layout(pad=0)
        plt.show()


    '''
        Cut the mesh by plane for rotation symmetry items
            - for p_id, next_p_id: cut from common_patch_border[p_id][v_to] to common_patch_border[next_p_id][v_from]
        INPUT
            - common_patch: ordered p_id list
            - common_patch_border: [dict{'v_border', 'v_from', 'v_to'}]
    '''
    def cut_mesh_by_rotation_symmetric_plane_tracing(self, plane_param, common_patch, common_patch_border):
        mesh = self.patch_mesh.om_mesh

        cutting_border_vert = []
        for p_id in common_patch_border:
            v_border = common_patch_border[p_id]['v_border']
            for i in range(len(v_border) - 1):
                cutting_border_vert.append([v_border[i], v_border[i + 1]])
        insert_v = []
        insert_v_num = -1
        raw_v_num = self.patch_mesh.v.shape[0]
        insert_f = defaultdict(list)
        f_visit = set()


        f_colors = np.ones((self.patch_mesh.f.shape[0], 3))
        plot = mp.plot(self.patch_mesh.v, self.patch_mesh.f, c=f_colors, shading={
            'wireframe': False
        })
        for i in range(len(common_patch)):
            p_id = common_patch[i]
            next_p_id = common_patch[(i + 1) % len(common_patch)]

            # initial
            v_from_id = common_patch_border[p_id]['v_to']
            v_from_pos = mesh.point(mesh.vertex_handle(v_from_id))
            v_from_type = 'V'
            v_to = common_patch_border[next_p_id]['v_from']

            prev_v_pos = v_from_pos
            prev_v_id = v_from_id


            # v_from_id = v_from
            # v_from_pos = mesh.point(mesh.vertex_handle(v_from_id))
            # v_from_type = 'V'
            # prev_v_pos = v_from_pos
            # prev_v_id = v_from_id

            # cutting_border_vert = []
            # for i in range(len(v_border) - 1):
            #     cutting_border_vert.append([v_border[i], v_border[i + 1]])
            
            # first direction: vertex to edge

            while True:
                if v_from_id == v_to:
                    print('Reach v_border')
                    break
                plot.add_lines(
                    np.array([prev_v_pos]),
                    np.array([v_from_pos]),
                    shading={
                        "line_color": "red",
                        "line_width": 1
                    }
                )
                plot.update_object()

                if v_from_type == 'V':
                    v_from_h = mesh.vertex_handle(v_from_id)
                else:
                    v_from_h = mesh.edge_handle(v_from_id)

                insert_type = None
                if v_from_type == 'V':
                    neighbor_f = []
                    for f_h in mesh.vf(v_from_h):
                        f_id = f_h.idx()
                        if f_id not in f_visit:
                            neighbor_f.append(f_id)
                            f_visit.add(f_id)
                    
                    next_v_info = []
                    for f_id in neighbor_f:
                        result = insection_plane_f(mesh, plane_param, mesh.face_handle(f_id), v_from_pos)
                        # f_colors = np.ones((self.patch_mesh.f.shape[0], 3))
                        # f_colors[f_id] = [1, 0, 0]
                        # plot.update_object(colors=f_colors)
                        if (result['type'] == 'vv') or (result['type'] == 've'):
                            next_v_info.append(result)
                    if len(next_v_info) == 0:
                        print('Cannot find next intersect face')
                        break



                    # get right next vert
                    # 1. ve > vv
                    # 2. remove vert in v_border
                    # 3. choose the duplicate one
                    next_v_info_id = -1
                    next_v_set = defaultdict(list)
                    for i in range(len(next_v_info)):
                        tmp_next_v_info = next_v_info[i]
                        if tmp_next_v_info['type'] == 've':
                            next_v_info_id = i
                            break
                        elif tmp_next_v_info['next_v_id'] == v_to:
                            next_v_info_id = i
                            break                       
                        elif tmp_next_v_info['next_v_id'] in v_border:
                            continue
                        else:
                            tmp_next_v_id = tmp_next_v_info['next_v_id']
                            if tmp_next_v_id not in next_v_set:
                                next_v_info_id = i
                                next_v_set[tmp_next_v_id].append(tmp_next_v_info)
                            else:
                                next_v_info_id = i
                                break
                    if next_v_info_id == -1:
                        raise('[Trace Cut]: cannot find next v id')
                    next_v_info = next_v_info[next_v_info_id]
                    # next_v_info = next_v_info[0]

                    if next_v_info['type'] == 'vv':
                        prev_v_id = v_from_id
                        prev_v_pos = v_from_pos
                        v_from_id = next_v_info['next_v_id']
                        v_from_pos = mesh.point(mesh.vertex_handle(v_from_id))
                        v_from_type = 'V'
                        f_id = next_v_info['f_id']

                        insert_type = 'vv'
                        cutting_border_vert.append([prev_v_id, v_from_id])
                    else:
                        prev_v_id = v_from_id
                        prev_v_pos = v_from_pos
                        v_from_id = next_v_info['next_e_id']
                        v_from_pos = next_v_info['v_insert']
                        v_from_type = 'E'
                        f_id = next_v_info['f_id']
                        
                        insert_type = 've'
                        insert_v.append(v_from_pos)
                        insert_v_num += 1
                        insert_v_id = raw_v_num + insert_v_num
                        cutting_border_vert.append([prev_v_id, insert_v_id])
                        

                elif v_from_type == 'E':
                    neighbor_f = -1
                    for f_id in e_h2f(mesh, v_from_h):
                    # for f_h in mesh.ef(v_from_h):
                        # f_id = f_h.idx()
                        if f_id not in f_visit:
                            neighbor_f = f_id
                            f_visit.add(f_id)
                    if neighbor_f == -1:
                        print('Cannot find next not visited face')
                        break
                    
                    result = insection_plane_f(mesh, plane_param, mesh.face_handle(neighbor_f), v_from_pos)

                    if result['type'] == 'ev':
                        prev_v_id = v_from_id
                        prev_v_pos = v_from_pos
                        v_from_id = result['next_v_id']
                        v_from_pos = mesh.point(mesh.vertex_handle(v_from_id))
                        v_from_type = 'V'
                        f_id = result['f_id']

                        insert_type = 'ev'
                        prev_insert_v_id = raw_v_num + insert_v_num
                        cutting_border_vert.append([prev_insert_v_id, v_from_id])

                    elif result['type'] == 'ee':
                        prev_v_id = v_from_id
                        prev_v_pos = v_from_pos
                        v_from_id = result['to_e_id']
                        v_from_pos = result['v_insert']
                        v_from_type = 'E'
                        f_id = result['f_id']

                        insert_type = 'ee'
                        prev_insert_v_id = raw_v_num + insert_v_num
                        insert_v.append(v_from_pos)
                        insert_v_num += 1
                        insert_v_id = raw_v_num + insert_v_num
                        cutting_border_vert.append([prev_insert_v_id, insert_v_id])
                

                # insert faces
                if insert_type == 'vv':
                    pass
                elif insert_type == 've':
                    raw_face = self.patch_mesh.f[f_id]
                    pre_v_index = np.where(raw_face == prev_v_id)[0][0]

                    insert_f[f_id].append([
                        raw_face[pre_v_index],
                        raw_face[(pre_v_index + 1) % 3],
                        insert_v_id
                    ])
                    insert_f[f_id].append([
                        raw_face[(pre_v_index - 1) % 3],
                        raw_face[pre_v_index],
                        insert_v_id
                    ])
                elif insert_type == 'ev':
                    raw_face = self.patch_mesh.f[f_id]
                    pre_v_index = np.where(raw_face == v_from_id)[0][0]

                    insert_f[f_id].append([
                        raw_face[pre_v_index],
                        raw_face[(pre_v_index + 1) % 3],
                        prev_insert_v_id
                    ])
                    insert_f[f_id].append([
                        raw_face[(pre_v_index - 1) % 3],
                        raw_face[pre_v_index],
                        prev_insert_v_id
                    ])
                elif insert_type == 'ee':
                    raw_face = self.patch_mesh.f[f_id]
                    
                    prev_e_id = prev_v_id
                    cur_e_id = v_from_id
                    op_v_id0 = e_h2ov_in_face(mesh, mesh.edge_handle(prev_e_id), raw_face)
                    op_v_index0 = np.where(raw_face == op_v_id0)[0][0]
                    op_v_id1 = e_h2ov_in_face(mesh, mesh.edge_handle(cur_e_id), raw_face)
                    op_v_index1 = np.where(raw_face == op_v_id1)[0][0]

            
                    if op_v_index1 == (op_v_index0 + 1) % 3:
                        insert_f[f_id].append([
                            raw_face[op_v_index0],
                            raw_face[(op_v_index0 + 1) % 3],
                            prev_insert_v_id
                        ])
                        insert_f[f_id].append([
                            prev_insert_v_id,
                            raw_face[(op_v_index0 - 1) % 3],
                            insert_v_id
                        ])
                        insert_f[f_id].append([
                            raw_face[op_v_index0],
                            prev_insert_v_id,
                            insert_v_id
                        ])
                    else:
                        insert_f[f_id].append([
                            raw_face[(op_v_index0 - 1) % 3],
                            raw_face[op_v_index0],
                            prev_insert_v_id
                        ])
                        insert_f[f_id].append([
                            raw_face[(op_v_index0 + 1) % 3],
                            prev_insert_v_id,
                            insert_v_id
                        ])
                        insert_f[f_id].append([
                            raw_face[op_v_index0],
                            insert_v_id,
                            prev_insert_v_id
                        ])
            
        for f_id in insert_f.keys():
            insert_f[f_id] = np.array(insert_f[f_id])

        # update vertices
        if len(insert_v) > 0:
            insert_v = np.array(insert_v)
            new_mesh_vertices = np.concatenate((self.patch_mesh.v, insert_v), axis=0)
        else:
            new_mesh_vertices = self.patch_mesh.v

        # remove old faces & insert new faces with patch id
        remove_f = list(insert_f.keys())
        new_mesh_faces = np.delete(self.patch_mesh.f, remove_f, axis = 0)
        new_f_id2raw_f_id = [i for i in range(len(self.patch_mesh.f)) if i not in remove_f]
        raw_f_len = new_mesh_faces.shape[0]
        
        new_f_p_id = {}
        new_f_id = raw_f_len
        for raw_f_id in insert_f.keys():
            for _ in range(insert_f[raw_f_id].shape[0]):
                new_f_p_id[new_f_id] = self.patch_mesh.f2patch[raw_f_id]
                new_f_id += 1
            new_mesh_faces = np.concatenate((new_mesh_faces, insert_f[raw_f_id]))
        
        # check
        f_colors = np.ones((new_mesh_faces.shape[0], 3))
        # color = colors[41]
        # f_colors = np.tile(color, (new_mesh_faces.shape[0], 1))
        plot = mp.plot(new_mesh_vertices, new_mesh_faces, f_colors, shading={'wireframe': False, 'point_size': 0})
        cutting_border_vert_array = np.array(cutting_border_vert)
        plot.add_lines(
            new_mesh_vertices[cutting_border_vert_array[:, 0]],
            new_mesh_vertices[cutting_border_vert_array[:, 1]],
            shading={
                "line_color": "red",
                "line_width": 5
            }
        )

        # split mesh
        # noting the insert edge, and split by connectivity of face Graph
        cutting_border_vert_ordered = []
        for v_pair in cutting_border_vert:
            v0_id, v1_id = v_pair
            if v0_id > v1_id:
                cutting_border_vert_ordered.append([v1_id, v0_id])
            else:
                cutting_border_vert_ordered.append([v0_id, v1_id])
        # cutting_border_vert = np.array(cutting_border_vert)
        # cutting_mesh = om.TriMesh(new_mesh_vertices, new_mesh_faces)
        # cutting_border_edge = []
        # for i in range(cutting_border_vert.shape[0]):
        #     v0_id, v1_id = cutting_border_edge[i]
        #     cutting_border_edge.append(v_pair2e(cutting_mesh, v0_id, v1_id))
        
        cutting_graph = nx.Graph()
        # add node
        for i in range(new_mesh_faces.shape[0]):
            cutting_graph.add_node(i)
        # edge to face
        edge2face = defaultdict(list)
        for f_id in range(new_mesh_faces.shape[0]):
            for i in range(3):
                v0, v1 = new_mesh_faces[f_id][i], new_mesh_faces[f_id][(i + 1) % 3]
                if v0 > v1:
                    v0, v1 = v1, v0
                edge2face[(v0, v1)].append(f_id)
        # add edges
        for edge, n_f in edge2face.items():
            if len(n_f) == 2:
                if [edge[0], edge[1]] in cutting_border_vert_ordered:
                    continue
                cutting_graph.add_edge(n_f[0], n_f[1])
        components = list(nx.connected_components(cutting_graph))

        f_splited = [list(components[i]) for i in range(len(components))]
        new_patch2f = []
        for i in range(len(f_splited)):
            patch2f_i = defaultdict(list)
            for f_id in f_splited[i]:
                if f_id >= raw_f_len:
                    p_id = new_f_p_id[f_id]
                else:
                    raw_f_id = new_f_id2raw_f_id[f_id]
                    p_id = self.patch_mesh.f2patch[raw_f_id]
                patch2f_i[p_id].append(f_id)
            patch2f_i = [patch2f_i[p_id] for p_id in patch2f_i]

            for j in range(len(patch2f_i)):
                for k in range(len(patch2f_i[j])):
                    if patch2f_i[j][k] in f_splited[i]:
                        patch2f_i[j][k] = f_splited[i].index(patch2f_i[j][k])
            new_patch2f.append(patch2f_i)
        
        objs = []
        for i in range(len(f_splited)):
            vi, fi = get_submesh(new_mesh_vertices, new_mesh_faces, f_splited[i])
            # hole filling
            objs.append({
                'v': vi,
                'f': fi,
                'patch': new_patch2f[i]
            })
            mp.plot(vi, fi, np.ones((fi.shape[0], 3)), shading={'wireframe': False})
        return True, objs


    '''
        Cut the mesh by cylinder for rotation symmetry items
            - for p_id, next_p_id: cut from common_patch_border[p_id][v_to] to common_patch_border[next_p_id][v_from]
        INPUT
            - common_patch: ordered p_id list
            - common_patch_border: [dict{'v_border', 'v_from', 'v_to'}]
    '''
    def cut_mesh_by_rotation_symmetric_cylinder_tracing(self, cylinder_param, common_patch, common_patch_border):
        mesh = self.patch_mesh.om_mesh

        cutting_border_vert = []
        v_border = []
        for p_id in common_patch_border:
            for i in range(2):
                v_border.extend(common_patch_border[p_id][i]['v_border'])
                cutting_border_vert.append([common_patch_border[p_id][i]['v_border'][0], common_patch_border[p_id][i]['v_border'][-1]])
            # for i in range(2):
            #     v_border.extend(common_patch_border[p_id][i]['v_border'])
            #     for j in range(len(v_border) - 1):
            #         cutting_border_vert.append([v_border[j], v_border[j + 1]])
        
        insert_v = []
        insert_v_num = -1
        raw_v_num = self.patch_mesh.v.shape[0]
        insert_f = defaultdict(list)
        f_visit = set()


        f_colors = np.ones((self.patch_mesh.f.shape[0], 3))
        plot = mp.plot(self.patch_mesh.v, self.patch_mesh.f, c=f_colors, shading={
            'wireframe': False
        })
        for i in range(len(common_patch)):
            p_id = common_patch[i]
            next_p_id = common_patch[(i + 1) % len(common_patch)]

            # initial
            # v_from_id = common_patch_border[p_id][0]['v_from']
            v_from_id = common_patch_border[p_id][0]['v_to']
            v_from_pos = mesh.point(mesh.vertex_handle(v_from_id))
            v_from_type = 'V'
            # v_to = common_patch_border[next_p_id][1]['v_from']
            v_to = common_patch_border[next_p_id][1]['v_to']

            plot.add_points(
                np.array([v_from_pos]),
                shading={
                    'point_color': 'red',
                    'point_size': 1
                }
            )
            plot.add_points(
                np.array([mesh.point(mesh.vertex_handle(v_to))]),
                shading={
                    'point_color': 'blue',
                    'point_size': 1
                }
            )

            prev_v_pos = v_from_pos
            prev_v_id = v_from_id


            # v_from_id = v_from
            # v_from_pos = mesh.point(mesh.vertex_handle(v_from_id))
            # v_from_type = 'V'
            # prev_v_pos = v_from_pos
            # prev_v_id = v_from_id

            # cutting_border_vert = []
            # for i in range(len(v_border) - 1):
            #     cutting_border_vert.append([v_border[i], v_border[i + 1]])
            
            # first direction: vertex to edge

            while True:
                if v_from_id == v_to:
                    print('Reach v_border')
                    break
                plot.add_lines(
                    np.array([prev_v_pos]),
                    np.array([v_from_pos]),
                    shading={
                        "line_color": "red",
                        "line_width": 1
                    }
                )
                plot.update_object()

                if v_from_type == 'V':
                    v_from_h = mesh.vertex_handle(v_from_id)
                else:
                    v_from_h = mesh.edge_handle(v_from_id)

                insert_type = None
                if v_from_type == 'V':
                    neighbor_f = []
                    for f_h in mesh.vf(v_from_h):
                        f_id = f_h.idx()
                        if f_id not in f_visit:
                            neighbor_f.append(f_id)
                            f_visit.add(f_id)
                    
                    next_v_info = []
                    for f_id in neighbor_f:
                        result = insection_cylinder_f_by_vert(mesh, cylinder_param, mesh.face_handle(f_id), v_from_pos, v_from_type, v_from_id)
                        f_colors = np.ones((self.patch_mesh.f.shape[0], 3))
                        f_colors[f_id] = [1, 0, 0]
                        plot.update_object(colors=f_colors)
                        if (result['type'] == 've'):
                            next_v_info.append(result)
                            break
                        if (result['type'] == 'vv'):
                            next_v_info.append(result)
                    if len(next_v_info) == 0:
                        print('Cannot find next intersect face')
                        break



                    # get right next vert
                    # 1. ve > vv
                    # 2. remove vert in v_border
                    # 3. choose the duplicate one
                    next_v_info_id = -1
                    next_v_set = defaultdict(list)
                    for i in range(len(next_v_info)):
                        tmp_next_v_info = next_v_info[i]
                        if tmp_next_v_info['type'] == 've':
                            next_v_info_id = i
                            break
                        elif tmp_next_v_info['next_v_id'] == v_to:
                            next_v_info_id = i
                            break                       
                        elif tmp_next_v_info['next_v_id'] in v_border:
                            continue
                        else:
                            tmp_next_v_id = tmp_next_v_info['next_v_id']
                            if tmp_next_v_id not in next_v_set:
                                next_v_info_id = i
                                next_v_set[tmp_next_v_id].append(tmp_next_v_info)
                            else:
                                next_v_info_id = i
                                break
                    if next_v_info_id == -1:
                        raise('[Trace Cut]: cannot find next v id')
                    next_v_info = next_v_info[next_v_info_id]
                    # next_v_info = next_v_info[0]

                    if next_v_info['type'] == 'vv':
                        prev_v_id = v_from_id
                        prev_v_pos = v_from_pos
                        v_from_id = next_v_info['next_v_id']
                        v_from_pos = mesh.point(mesh.vertex_handle(v_from_id))
                        v_from_type = 'V'
                        f_id = next_v_info['f_id']

                        insert_type = 'vv'
                        cutting_border_vert.append([prev_v_id, v_from_id])
                    else:
                        prev_v_id = v_from_id
                        prev_v_pos = v_from_pos
                        v_from_id = next_v_info['next_e_id']
                        v_from_pos = next_v_info['v_insert']
                        v_from_type = 'E'
                        f_id = next_v_info['f_id']
                        
                        insert_type = 've'
                        insert_v.append(v_from_pos)
                        insert_v_num += 1
                        insert_v_id = raw_v_num + insert_v_num
                        cutting_border_vert.append([prev_v_id, insert_v_id])
                        

                elif v_from_type == 'E':
                    neighbor_f = -1
                    for f_id in e_h2f(mesh, v_from_h):
                    # for f_h in mesh.ef(v_from_h):
                        # f_id = f_h.idx()
                        if f_id not in f_visit:
                            neighbor_f = f_id
                            f_visit.add(f_id)
                    if neighbor_f == -1:
                        print('Cannot find next not visited face')
                        break
                    
                    result = insection_cylinder_f_by_vert(mesh, cylinder_param, mesh.face_handle(neighbor_f), v_from_pos, v_from_type, v_from_id)

                    if result['type'] == 'ev':
                        prev_v_id = v_from_id
                        prev_v_pos = v_from_pos
                        v_from_id = result['next_v_id']
                        v_from_pos = mesh.point(mesh.vertex_handle(v_from_id))
                        v_from_type = 'V'
                        f_id = result['f_id']

                        insert_type = 'ev'
                        prev_insert_v_id = raw_v_num + insert_v_num
                        cutting_border_vert.append([prev_insert_v_id, v_from_id])

                    elif result['type'] == 'ee':
                        prev_v_id = v_from_id
                        prev_v_pos = v_from_pos
                        v_from_id = result['to_e_id']
                        v_from_pos = result['v_insert']
                        v_from_type = 'E'
                        f_id = result['f_id']

                        insert_type = 'ee'
                        prev_insert_v_id = raw_v_num + insert_v_num
                        insert_v.append(v_from_pos)
                        insert_v_num += 1
                        insert_v_id = raw_v_num + insert_v_num
                        cutting_border_vert.append([prev_insert_v_id, insert_v_id])
                

                # insert faces
                if insert_type == 'vv':
                    pass
                elif insert_type == 've':
                    raw_face = self.patch_mesh.f[f_id]
                    pre_v_index = np.where(raw_face == prev_v_id)[0][0]

                    insert_f[f_id].append([
                        raw_face[pre_v_index],
                        raw_face[(pre_v_index + 1) % 3],
                        insert_v_id
                    ])
                    insert_f[f_id].append([
                        raw_face[(pre_v_index - 1) % 3],
                        raw_face[pre_v_index],
                        insert_v_id
                    ])
                elif insert_type == 'ev':
                    raw_face = self.patch_mesh.f[f_id]
                    pre_v_index = np.where(raw_face == v_from_id)[0][0]

                    insert_f[f_id].append([
                        raw_face[pre_v_index],
                        raw_face[(pre_v_index + 1) % 3],
                        prev_insert_v_id
                    ])
                    insert_f[f_id].append([
                        raw_face[(pre_v_index - 1) % 3],
                        raw_face[pre_v_index],
                        prev_insert_v_id
                    ])
                elif insert_type == 'ee':
                    raw_face = self.patch_mesh.f[f_id]
                    
                    prev_e_id = prev_v_id
                    cur_e_id = v_from_id
                    op_v_id0 = e_h2ov_in_face(mesh, mesh.edge_handle(prev_e_id), raw_face)
                    op_v_index0 = np.where(raw_face == op_v_id0)[0][0]
                    op_v_id1 = e_h2ov_in_face(mesh, mesh.edge_handle(cur_e_id), raw_face)
                    op_v_index1 = np.where(raw_face == op_v_id1)[0][0]

            
                    if op_v_index1 == (op_v_index0 + 1) % 3:
                        insert_f[f_id].append([
                            raw_face[op_v_index0],
                            raw_face[(op_v_index0 + 1) % 3],
                            prev_insert_v_id
                        ])
                        insert_f[f_id].append([
                            prev_insert_v_id,
                            raw_face[(op_v_index0 - 1) % 3],
                            insert_v_id
                        ])
                        insert_f[f_id].append([
                            raw_face[op_v_index0],
                            prev_insert_v_id,
                            insert_v_id
                        ])
                    else:
                        insert_f[f_id].append([
                            raw_face[(op_v_index0 - 1) % 3],
                            raw_face[op_v_index0],
                            prev_insert_v_id
                        ])
                        insert_f[f_id].append([
                            raw_face[(op_v_index0 + 1) % 3],
                            prev_insert_v_id,
                            insert_v_id
                        ])
                        insert_f[f_id].append([
                            raw_face[op_v_index0],
                            insert_v_id,
                            prev_insert_v_id
                        ])
            
        for f_id in insert_f.keys():
            insert_f[f_id] = np.array(insert_f[f_id])

        # update vertices
        if len(insert_v) > 0:
            insert_v = np.array(insert_v)
            new_mesh_vertices = np.concatenate((self.patch_mesh.v, insert_v), axis=0)
        else:
            new_mesh_vertices = self.patch_mesh.v

        # remove old faces & insert new faces with patch id
        remove_f = list(insert_f.keys())
        new_mesh_faces = np.delete(self.patch_mesh.f, remove_f, axis = 0)
        new_f_id2raw_f_id = [i for i in range(len(self.patch_mesh.f)) if i not in remove_f]
        raw_f_len = new_mesh_faces.shape[0]
        
        new_f_p_id = {}
        new_f_id = raw_f_len
        for raw_f_id in insert_f.keys():
            for _ in range(insert_f[raw_f_id].shape[0]):
                new_f_p_id[new_f_id] = self.patch_mesh.f2patch[raw_f_id]
                new_f_id += 1
            new_mesh_faces = np.concatenate((new_mesh_faces, insert_f[raw_f_id]))
        
        # check
        f_colors = np.tile(colors[61], (new_mesh_faces.shape[0], 1))
        plot = mp.plot(new_mesh_vertices, new_mesh_faces, f_colors, shading={'wireframe': False, 'point_size': 0})
        cutting_border_vert_array = np.array(cutting_border_vert)
        plot.add_lines(
            new_mesh_vertices[cutting_border_vert_array[:, 0]],
            new_mesh_vertices[cutting_border_vert_array[:, 1]],
            shading={
                "line_color": "red",
                "line_width": 3
            }
        )

        # split mesh
        # noting the insert edge, and split by connectivity of face Graph
        cutting_border_vert_ordered = []
        for v_pair in cutting_border_vert:
            v0_id, v1_id = v_pair
            if v0_id > v1_id:
                cutting_border_vert_ordered.append([v1_id, v0_id])
            else:
                cutting_border_vert_ordered.append([v0_id, v1_id])
        # cutting_border_vert = np.array(cutting_border_vert)
        # cutting_mesh = om.TriMesh(new_mesh_vertices, new_mesh_faces)
        # cutting_border_edge = []
        # for i in range(cutting_border_vert.shape[0]):
        #     v0_id, v1_id = cutting_border_edge[i]
        #     cutting_border_edge.append(v_pair2e(cutting_mesh, v0_id, v1_id))
        
        cutting_graph = nx.Graph()
        # add node
        for i in range(new_mesh_faces.shape[0]):
            cutting_graph.add_node(i)
        # edge to face
        edge2face = defaultdict(list)
        for f_id in range(new_mesh_faces.shape[0]):
            for i in range(3):
                v0, v1 = new_mesh_faces[f_id][i], new_mesh_faces[f_id][(i + 1) % 3]
                if v0 > v1:
                    v0, v1 = v1, v0
                edge2face[(v0, v1)].append(f_id)
        # add edges
        for edge, n_f in edge2face.items():
            if len(n_f) == 2:
                if [edge[0], edge[1]] in cutting_border_vert_ordered:
                    continue
                cutting_graph.add_edge(n_f[0], n_f[1])
        components = list(nx.connected_components(cutting_graph))

        f_splited = [list(components[i]) for i in range(len(components))]
        new_patch2f = []
        for i in range(len(f_splited)):
            patch2f_i = defaultdict(list)
            for f_id in f_splited[i]:
                if f_id >= raw_f_len:
                    p_id = new_f_p_id[f_id]
                else:
                    raw_f_id = new_f_id2raw_f_id[f_id]
                    p_id = self.patch_mesh.f2patch[raw_f_id]
                patch2f_i[p_id].append(f_id)
            patch2f_i = [patch2f_i[p_id] for p_id in patch2f_i]

            for j in range(len(patch2f_i)):
                for k in range(len(patch2f_i[j])):
                    if patch2f_i[j][k] in f_splited[i]:
                        patch2f_i[j][k] = f_splited[i].index(patch2f_i[j][k])
            new_patch2f.append(patch2f_i)
        
        objs = []
        for i in range(len(f_splited)):
            vi, fi = get_submesh(new_mesh_vertices, new_mesh_faces, f_splited[i])
            objs.append({
                'v': vi,
                'f': fi,
                'patch': new_patch2f[i]
            })
            mp.plot(vi, fi, np.ones((fi.shape[0], 3)), shading={'wireframe': False})
        return True, objs
    


    def cut_mesh_by_cylinder_tracing(self, cylinder_params, v_from, v_to, v_border):
        # v_border = self.get_patch_border_v(p1, p2)
        # v_from, v_to = v_border[0], v_border[-1]
        # v_to_pos = self.patch_mesh.v[v_to]
        mesh = self.patch_mesh.om_mesh

        c, axis, r = cylinder_params[:3].squeeze(), cylinder_params[3:6].squeeze(), cylinder_params[6].item()
        threshold_sample_len = r * np.pi / 18. / 2.

        v_from_id = v_from
        v_from_pos = self.patch_mesh.v[v_from_id]
        # v_from_pos = mesh.point(mesh.vertex_handle(v_from_id))
        v_from_type = 'V'
        f_visit = set()
        prev_v_pos = v_from_pos
        prev_v_id = v_from_id
        prev_insert_v_id = -1

        insert_v = []
        insert_v_num = -1
        raw_insert_v_num = 0
        raw_v_num = self.patch_mesh.v.shape[0]
        insert_f = defaultdict(list)
        del_f = []
        
        f_colors = np.ones((self.patch_mesh.f.shape[0], 3))
        # f_colors[self.patch_mesh.patch2f[7]]= [0, 1, 1]
        plot = mp.plot(self.patch_mesh.v, self.patch_mesh.f, c=f_colors, shading={
            'wireframe': True
        })
        plot.add_points(
            self.patch_mesh.v[v_border],
            shading={
                'point_color': 'blue',
                'point_size': 5
            }
        )
        cutting_border_vert = []
        for i in range(len(v_border) - 1):
            cutting_border_vert.append([v_border[i], v_border[i + 1]])

        while True:
            
            if v_from_id == v_to:
                print('Reach v_border')
                break
            # plot.update_object(colors=np.ones((self.patch_mesh.f.shape[0], 3)))
            # plot.add_lines(
            #     np.array([prev_v_pos]),
            #     np.array([v_from_pos]),
            #     shading={
            #         "line_color": "red",
            #         "line_width": 1
            #     }
            # )
            # if insert_v_num > raw_insert_v_num:
            #     tmp_insert_v = np.array(insert_v[raw_insert_v_num:])
            #     raw_insert_v_num = insert_v_num
            #     plot.add_points(
            #         tmp_insert_v,
            #         shading={
            #             'point_color': 'red',
            #             'point_size': 5
            #         }
            #     )
            #     plot.update_object()
            # else:
            #     plot.add_lines(
            #         np.array([prev_v_pos]),
            #         np.array([v_from_pos]),
            #         shading={
            #             "line_color": "red",
            #             "line_width": 1
            #         }
            #     )


            if v_from_type == 'V':
                v_from_h = mesh.vertex_handle(v_from_id)
            else:
                v_from_h = mesh.edge_handle(v_from_id)
            
            insert_type = None
            sample = False
            sample_num = 0
            if v_from_type == 'V':
                neighbor_f = []
                for f_h in mesh.vf(v_from_h):
                    f_id = f_h.idx()
                    if f_id not in f_visit:
                        neighbor_f.append(f_id)
                        f_visit.add(f_id)
                
                next_v_info = []
                for f_id in neighbor_f:
                    # f_colors[f_id] = [1, 0, 0]
                    # plot.update_object(colors=f_colors)
                    # result = insection_cylinder_f(mesh, cylinder_params, mesh.face_handle(f_id), v_from_pos)
                    result = insection_cylinder_f_by_vert(mesh, cylinder_params, mesh.face_handle(f_id), v_from_pos, v_from_type, v_from_id)
                    if (result['type'] == 've'):
                        next_v_info.append(result)
                        break
                    if (result['type'] == 'vv'):
                        next_v_info.append(result)
                if len(next_v_info) == 0:
                    print('Cannot find next intersect face')
                    break

                # get right next vert
                # 1. ve > vv
                # 2. remove vert in v_border
                # 3. choose the duplicate one
                next_v_info_id = -1
                next_v_set = defaultdict(list)
                for i in range(len(next_v_info)):
                    tmp_next_v_info = next_v_info[i]
                    if tmp_next_v_info['type'] == 've':
                        next_v_info_id = i
                        break
                    elif tmp_next_v_info['next_v_id'] == v_to:
                        next_v_info_id = i
                        break                       
                    elif tmp_next_v_info['next_v_id'] in v_border:
                        continue
                    else:
                        tmp_next_v_id = tmp_next_v_info['next_v_id']
                        if tmp_next_v_id not in next_v_set:
                            next_v_info_id = i
                            next_v_set[tmp_next_v_id].append(tmp_next_v_info)
                        else:
                            next_v_info_id = i
                            break
                if next_v_info_id == -1:
                    raise('[Trace Cut]: cannot find next v id')
                next_v_info = next_v_info[next_v_info_id]

                if next_v_info['type'] == 'vv':
                    prev_v_id = v_from_id
                    prev_v_pos = v_from_pos
                    v_from_id = next_v_info['next_v_id']
                    v_from_pos = mesh.point(mesh.vertex_handle(v_from_id))
                    v_from_type = 'V'
                    f_id = next_v_info['f_id']

                    insert_type = 'vv'
                    # cutting_dist = np.linalg.norm(v_from_pos - prev_v_pos)
                    # cutting_border_vert.append([prev_v_id, v_from_id])
                           
                else:
                    prev_v_id = v_from_id
                    prev_v_pos = v_from_pos
                    v_from_id = next_v_info['next_e_id']
                    v_from_pos = next_v_info['v_insert']
                    v_from_type = 'E'
                    f_id = next_v_info['f_id']

                    insert_type = 've'
                    insert_v.append(v_from_pos)
                    insert_v_num += 1
                    insert_v_id = raw_v_num + insert_v_num
                    # prev_insert_v_id = insert_v_id
                    # cutting_border_vert.append([prev_v_id, insert_v_id])
                    # cutting_dist = np.linalg.norm(v_from_pos - prev_v_pos)
                    # insert_v.append(v_from_pos)
                    # insert_v_num += 1
                    # insert_v_id = raw_v_num + insert_v_num
                    # cutting_border_vert.append([prev_v_id, insert_v_id])
            
            elif v_from_type == 'E':
                neighbor_f = -1
                for f_id in e_h2f(mesh, v_from_h):
                    if f_id not in f_visit:
                        neighbor_f = f_id
                        f_visit.add(f_id)
                if neighbor_f == -1:
                    print('Cannot find next not visited face')
                    break
                
                result = insection_cylinder_f_by_vert(mesh, cylinder_params, mesh.face_handle(neighbor_f), v_from_pos, v_from_type, v_from_id)
                # result = insection_cylinder_f(mesh, cylinder_params, mesh.face_handle(neighbor_f), v_from_pos)

                if result['type'] == 'ev':
                    prev_v_id = v_from_id
                    prev_v_pos = v_from_pos
                    v_from_id = result['next_v_id']
                    v_from_pos = mesh.point(mesh.vertex_handle(v_from_id))
                    v_from_type = 'V'
                    f_id = result['f_id']

                    insert_type = 'ev'
                    # prev_insert_v_id = raw_v_num + insert_v_num
                    # cutting_border_vert.append([prev_insert_v_id, v_from_id])

                elif result['type'] == 'ee':
                    prev_v_id = v_from_id
                    prev_v_pos = v_from_pos
                    v_from_id = result['to_e_id']
                    v_from_pos = result['v_insert']
                    v_from_type = 'E'
                    f_id = result['f_id']

                    insert_type = 'ee'
                    # prev_insert_v_id = raw_v_num + insert_v_num
                    insert_v.append(v_from_pos)
                    insert_v_num += 1
                    insert_v_id = raw_v_num + insert_v_num
                    # cutting_border_vert.append([prev_insert_v_id, insert_v_id])
                    # prev_insert_v_id = insert_v_id

                
            # insert faces
            if insert_type == 'vv':

                sample_points = sample_from_cylinder_on_face(
                    mesh, cylinder_params, f_id, 
                    prev_v_pos, v_from_pos, threshold_sample_len
                )

                if sample_points is None:
                    cutting_border_vert.append([prev_v_id, v_from_id])
                    continue
                
                sample_num = sample_points.shape[0]
                insert_v_range = range(raw_v_num + insert_v_num, raw_v_num + insert_v_num + sample_num)
                insert_v_num += sample_num
                for i in range(sample_num):
                    insert_v.append(sample_points[i])
            
                cutting_border_vert.append([prev_v_id, insert_v_range[0]])
                for i in range(sample_num - 1):
                    cutting_border_vert.append([insert_v_range[i], insert_v_range[i + 1]])
                cutting_border_vert.append([insert_v_range[-1], v_from_id])

                # triangulate
                mid_v = (prev_v_pos + v_from_pos) / 2.
                insert_v.append(mid_v)
                insert_v_num += 1
                mid_v_id = raw_v_num + insert_v_num

                op_v_id = v2op_v(mesh, prev_v_id, v_from_id, f_id)
                cur_e_id = v_pair2e(mesh, prev_v_id, v_from_id)
                op_f_id = e2op_f(mesh, cur_e_id, f_id)
                op_f_op_v_id = e_h2op_v_in_f_h(mesh, mesh.edge_handle(cur_e_id), mesh.face_handle(op_f_id))
                
                # 1. samples -- op_v_id
                insert_f[f_id].append([
                    prev_v_id,
                    op_v_id, 
                    insert_v_range[0]
                ])
                for i in range(0, sample_num - 1):
                    insert_f[f_id].append([
                        insert_v_range[i],
                        insert_v_range[i+1],
                        op_v_id
                    ])
                insert_f[f_id].append([
                    insert_v_range[-1],
                    op_v_id,
                    v_from_id
                ])


                # 2. samples -- mid
                insert_f[f_id].append([
                    prev_v_id,
                    mid_v_id, 
                    insert_v_range[0]
                ])
                for i in range(0, sample_num - 1):
                    insert_f[f_id].append([
                        insert_v_range[i],
                        insert_v_range[i+1],
                        mid_v_id
                    ])
                insert_f[f_id].append([
                    insert_v_range[-1],
                    mid_v_id,
                    v_from_id
                ])


                # 3. mid -- op_v_id
                del_f.append(op_f_id)
                insert_f[op_f_id].append([
                    prev_v_id,
                    mid_v_id,
                    op_f_op_v_id
                ])
                insert_f[op_f_id].append([
                    op_f_op_v_id,
                    mid_v_id,
                    v_from_id
                ])


            elif insert_type == 've':
                sample_points = sample_from_cylinder_on_face(
                    mesh, cylinder_params, f_id, 
                    prev_v_pos, v_from_pos, threshold_sample_len
                )

                if sample_points is not None:
                    
                
                    sample_num = sample_points.shape[0]
                    insert_v_range = range(raw_v_num + insert_v_num + 1, raw_v_num + insert_v_num + 1 + sample_num)
                    insert_v_num += sample_num
                    for i in range(sample_num):
                        insert_v.append(sample_points[i])
                
                    cutting_border_vert.append([prev_v_id, insert_v_range[0]])
                    for i in range(sample_num - 1):
                        cutting_border_vert.append([insert_v_range[i], insert_v_range[i + 1]])
                    cutting_border_vert.append([insert_v_range[-1], insert_v_id])

                    # triangulate
                    n_v0_id, n_v1_id = e_h2v(mesh, mesh.edge_handle(v_from_id))

                    # 1. samples -- n_v0_id
                    insert_f[f_id].append([
                        prev_v_id,
                        insert_v_range[0],
                        n_v0_id
                    ])
                    for i in range(0, sample_num - 1):
                        insert_f[f_id].append([
                            insert_v_range[i],
                            insert_v_range[i+1],
                            n_v0_id,
                        ])
                    insert_f[f_id].append([
                        insert_v_range[-1],
                        insert_v_id,
                        n_v0_id
                    ])


                    # 2. samples -- n_v1_id
                    insert_f[f_id].append([
                        prev_v_id,
                        insert_v_range[0],
                        n_v1_id
                    ])
                    for i in range(0, sample_num - 1):
                        insert_f[f_id].append([
                            insert_v_range[i],
                            insert_v_range[i+1],
                            n_v1_id,
                        ])
                    insert_f[f_id].append([
                        insert_v_range[-1],
                        insert_v_id,
                        n_v1_id,
                    ])


                # raw_face = self.patch_mesh.f[f_id]
                # pre_v_index = np.where(raw_face == prev_v_id)[0][0]

                # insert_f[f_id].append([
                #     raw_face[pre_v_index],
                #     raw_face[(pre_v_index + 1) % 3],
                #     insert_v_id
                # ])
                # insert_f[f_id].append([
                #     raw_face[(pre_v_index - 1) % 3],
                #     raw_face[pre_v_index],
                #     insert_v_id
                # ])
            
            elif insert_type == 'ev':
                sample_points = sample_from_cylinder_on_face(
                    mesh, cylinder_params, f_id, 
                    prev_v_pos, v_from_pos, threshold_sample_len
                )

                if sample_points is None:
                    continue
                
                sample_num = sample_points.shape[0]
                insert_v_range = range(raw_v_num + insert_v_num + 1, raw_v_num + insert_v_num + 1 + sample_num)
                insert_v_num += sample_num
                for i in range(sample_num):
                    insert_v.append(sample_points[i])

                cutting_border_vert.append([prev_insert_v_id, insert_v_range[0]])
                for i in range(sample_num - 1):
                    cutting_border_vert.append([insert_v_range[i], insert_v_range[i + 1]])
                cutting_border_vert.append([insert_v_range[-1], v_from_id])

                # triangulate

                n_v0_id, n_v1_id = e_h2v(mesh, mesh.edge_handle(prev_v_id))

                # 1. samples -- n_v0_id
                insert_f[f_id].append([
                    prev_insert_v_id,
                    insert_v_range[0],
                    n_v0_id
                ])
                for i in range(0, sample_num - 1):
                    insert_f[f_id].append([
                        insert_v_range[i],
                        insert_v_range[i+1],
                        n_v0_id,
                    ])
                insert_f[f_id].append([
                    insert_v_range[-1],
                    v_from_id,
                    n_v0_id
                ])


                # 2. samples -- n_v1_id
                insert_f[f_id].append([
                    prev_insert_v_id,
                    insert_v_range[0],
                    n_v1_id
                ])
                for i in range(0, sample_num - 1):
                    insert_f[f_id].append([
                        insert_v_range[i],
                        insert_v_range[i+1],
                        n_v1_id,
                    ])
                insert_f[f_id].append([
                    insert_v_range[-1],
                    v_from_id,
                    n_v1_id,
                ])

                # raw_face = self.patch_mesh.f[f_id]
                # pre_v_index = np.where(raw_face == v_from_id)[0][0]

                # insert_f[f_id].append([
                #     raw_face[pre_v_index],
                #     raw_face[(pre_v_index + 1) % 3],
                #     prev_insert_v_id
                # ])
                # insert_f[f_id].append([
                #     raw_face[(pre_v_index - 1) % 3],
                #     raw_face[pre_v_index],
                #     prev_insert_v_id
                # ])
            
            elif insert_type == 'ee':
                sample_points = sample_from_cylinder_on_face(
                    mesh, cylinder_params, f_id, 
                    prev_v_pos, v_from_pos, threshold_sample_len
                )

                if sample_points is None:
                    cutting_border_vert.append([prev_insert_v_id, insert_v_id])

                    raw_face = self.patch_mesh.f[f_id]
                    
                    prev_e_id = prev_v_id
                    cur_e_id = v_from_id
                    op_v_id0 = e_h2ov_in_face(mesh, mesh.edge_handle(prev_e_id), raw_face)
                    op_v_index0 = np.where(raw_face == op_v_id0)[0][0]
                    op_v_id1 = e_h2ov_in_face(mesh, mesh.edge_handle(cur_e_id), raw_face)
                    op_v_index1 = np.where(raw_face == op_v_id1)[0][0]

            
                    if op_v_index1 == (op_v_index0 + 1) % 3:
                        insert_f[f_id].append([
                            raw_face[op_v_index0],
                            raw_face[(op_v_index0 + 1) % 3],
                            prev_insert_v_id
                        ])
                        insert_f[f_id].append([
                            prev_insert_v_id,
                            raw_face[(op_v_index0 - 1) % 3],
                            insert_v_id
                        ])
                        insert_f[f_id].append([
                            raw_face[op_v_index0],
                            prev_insert_v_id,
                            insert_v_id
                        ])
                    else:
                        insert_f[f_id].append([
                            raw_face[(op_v_index0 - 1) % 3],
                            raw_face[op_v_index0],
                            prev_insert_v_id
                        ])
                        insert_f[f_id].append([
                            raw_face[(op_v_index0 + 1) % 3],
                            prev_insert_v_id,
                            insert_v_id
                        ])
                        insert_f[f_id].append([
                            raw_face[op_v_index0],
                            insert_v_id,
                            prev_insert_v_id
                        ])
                else:
                
                    sample_num = sample_points.shape[0]
                    insert_v_range = range(raw_v_num + insert_v_num + 1, raw_v_num + insert_v_num + 1 + sample_num)
                    insert_v_num += sample_num
                    for i in range(sample_num):
                        insert_v.append(sample_points[i])
                
                    cutting_border_vert.append([prev_insert_v_id, insert_v_range[0]])
                    for i in range(sample_num - 1):
                        cutting_border_vert.append([insert_v_range[i], insert_v_range[i + 1]])
                    cutting_border_vert.append([insert_v_range[-1], insert_v_id])

                    # triangulate
                    # mid_sample_id = raw_v_num + insert_v_num + int(sample_num / 2)

                    common_v_id = e_pair2v(mesh, prev_v_id, v_from_id)
                    prev_ano_v_id = e2ano_v(mesh, prev_v_id, common_v_id)
                    cur_ano_v_id = e2ano_v(mesh, v_from_id, common_v_id)

                    mid_v = (mesh.point(mesh.vertex_handle(prev_ano_v_id)) + mesh.point(mesh.vertex_handle(cur_ano_v_id))) / 2.
                    insert_v.append(mid_v)
                    insert_v_num += 1
                    mid_v_id = raw_v_num + insert_v_num


                    op_e_id = v_pair2e(mesh, prev_ano_v_id, cur_ano_v_id)
                    op_f_id = e2op_f(mesh, op_e_id, f_id)
                    op_f_op_v_id = e_h2op_v_in_f_h(mesh, mesh.edge_handle(op_e_id), mesh.face_handle(op_f_id))

                    
                    # 1. samples -- common_v_id
                    insert_f[f_id].append([
                        prev_insert_v_id,
                        common_v_id, 
                        insert_v_range[0]
                    ])
                    for i in range(0, sample_num - 1):
                        insert_f[f_id].append([
                            insert_v_range[i],
                            insert_v_range[i+1],
                            common_v_id
                        ])
                    insert_f[f_id].append([
                        insert_v_range[-1],
                        common_v_id, 
                        insert_v_id
                    ])

                    
                    # 2.1 part1-samples -- mid
                    insert_f[f_id].append([
                        prev_insert_v_id,
                        prev_ano_v_id, 
                        insert_v_range[0]
                    ])
                    for i in range(0, int((sample_num - 1) / 2)):
                        insert_f[f_id].append([
                            insert_v_range[i],
                            insert_v_range[i+1],
                            prev_ano_v_id
                        ])
                    insert_f[f_id].append([
                        insert_v_range[int((sample_num - 1) / 2)],
                        mid_v_id,
                        prev_ano_v_id
                    ])

                    # 2.2 part2-samples -- mid
                    insert_f[f_id].append([
                        insert_v_range[int((sample_num - 1) / 2)],
                        mid_v_id,
                        cur_ano_v_id
                    ])
                    for i in range(int((sample_num - 1) / 2), sample_num - 1):
                        insert_f[f_id].append([
                            insert_v_range[i],
                            insert_v_range[i+1],
                            cur_ano_v_id
                        ])
                    insert_f[f_id].append([
                        insert_v_range[-1],
                        insert_v_id,
                        cur_ano_v_id
                    ])


                    # 3. mid -- op_v_id
                    del_f.append(op_f_id)
                    insert_f[op_f_id].append([
                        prev_ano_v_id,
                        mid_v_id,
                        op_f_op_v_id
                    ])
                    insert_f[op_f_id].append([
                        op_f_op_v_id,
                        mid_v_id,
                        cur_ano_v_id
                    ])

            prev_insert_v_id = insert_v_id

        # update vertices
        insert_v = np.array(insert_v)
        new_mesh_vertices = np.concatenate((self.patch_mesh.v, insert_v), axis=0)
        
        # check normal direction
        for f_id in insert_f:
            insert_f[f_id] = np.array(insert_f[f_id])

            raw_face_normal = face_normal(new_mesh_vertices, self.patch_mesh.f[f_id])

            insert_face_normal = [face_normal(new_mesh_vertices, insert_f[f_id][i]) for i in range(insert_f[f_id].shape[0])]

            for i in range(len(insert_face_normal)):
                if np.dot(raw_face_normal, insert_face_normal[i]) < 0:
                    insert_f[f_id][i] = insert_f[f_id][i][[0, 2, 1]]
        
        
              

        # remove old faces & insert new faces with patch id
        remove_f = list(set(insert_f.keys()) | set(del_f))
        new_mesh_faces = np.delete(self.patch_mesh.f, remove_f, axis = 0)
        new_f_id2raw_f_id = [i for i in range(len(self.patch_mesh.f)) if i not in remove_f]
        raw_f_len = new_mesh_faces.shape[0]
        
        new_f_p_id = {}
        new_f_id = raw_f_len
        for raw_f_id in insert_f.keys():
            for _ in range(insert_f[raw_f_id].shape[0]):
                new_f_p_id[new_f_id] = self.patch_mesh.f2patch[raw_f_id]
                new_f_id += 1
            new_mesh_faces = np.concatenate((new_mesh_faces, insert_f[raw_f_id]))
        1
        # check
        plot = mp.plot(new_mesh_vertices, new_mesh_faces, np.ones((new_mesh_faces.shape[0], 3)), shading={'wireframe': True})
        cutting_border_vert_array = np.array(cutting_border_vert)
        plot.add_lines(
            new_mesh_vertices[cutting_border_vert_array[:, 0]],
            new_mesh_vertices[cutting_border_vert_array[:, 1]],
            shading={
                "line_color": "red",
                "line_width": 1
            }
        )

        # split mesh
        # noting the insert edge, and split by connectivity of face Graph
        cutting_border_vert_ordered = []
        for v_pair in cutting_border_vert:
            v0_id, v1_id = v_pair
            if v0_id > v1_id:
                cutting_border_vert_ordered.append([v1_id, v0_id])
            else:
                cutting_border_vert_ordered.append([v0_id, v1_id])

        cutting_graph = nx.Graph()
        # add node
        for i in range(new_mesh_faces.shape[0]):
            cutting_graph.add_node(i)
        # edge to face
        edge2face = defaultdict(list)
        for f_id in range(new_mesh_faces.shape[0]):
            for i in range(3):
                v0, v1 = new_mesh_faces[f_id][i], new_mesh_faces[f_id][(i + 1) % 3]
                if v0 > v1:
                    v0, v1 = v1, v0
                edge2face[(v0, v1)].append(f_id)
        # add edges
        for edge, n_f in edge2face.items():
            if len(n_f) == 2:
                if [edge[0], edge[1]] in cutting_border_vert_ordered:
                    continue
                cutting_graph.add_edge(n_f[0], n_f[1])
        components = list(nx.connected_components(cutting_graph))
        if len(components) != 2:
            raise ValueError("SPLIT ERROR]: tracing cut graph components != 2")
        f_in, f_out = list(components[0]), list(components[1])


        self.plot_mesh_with_cutting_cylinder(
            new_mesh_vertices,
            new_mesh_faces[f_in],
            new_mesh_faces[f_out],
            # new_mesh_faces[f_in],
            cylinder_params,
            elev=20,
            azim=-60
        )
        self.plot_mesh_with_cutting_cylinder(
            new_mesh_vertices,
            new_mesh_faces[f_in],
            new_mesh_faces[f_out],
            # new_mesh_faces[f_in],
            cylinder_params,
            elev=40,
            azim=-60
        )
        self.plot_mesh_with_cutting_cylinder(
            new_mesh_vertices,
            new_mesh_faces[f_in],
            new_mesh_faces[f_out],
            # new_mesh_faces[f_in],
            cylinder_params,
            elev=60,
            azim=-60
        )
        self.plot_mesh_with_cutting_cylinder(
            new_mesh_vertices,
            new_mesh_faces[f_in],
            new_mesh_faces[f_out],
            # new_mesh_faces[f_in],
            cylinder_params,
            elev=80,
            azim=-60
        )
        self.plot_mesh_with_cutting_cylinder(
            new_mesh_vertices,
            new_mesh_faces[f_in],
            new_mesh_faces[f_out],
            # new_mesh_faces[f_in],
            cylinder_params,
            elev=100,
            azim=-60
        )
        self.plot_mesh_with_cutting_cylinder(
            new_mesh_vertices,
            new_mesh_faces[f_in],
            new_mesh_faces[f_out],
            # new_mesh_faces[f_in],
            cylinder_params,
            elev=120,
            azim=-60
        )
        self.plot_mesh_with_cutting_cylinder(
            new_mesh_vertices,
            new_mesh_faces[f_in],
            new_mesh_faces[f_out],
            # new_mesh_faces[f_in],
            cylinder_params,
            elev=140,
            azim=-60
        )


        # convert patch
        patch2f_in = defaultdict(list)
        for f_id in f_in:
            if f_id >= raw_f_len:
                p_id = new_f_p_id[f_id]
            else:
                raw_f_id = new_f_id2raw_f_id[f_id]
                p_id = self.patch_mesh.f2patch[raw_f_id]
            patch2f_in[p_id].append(f_id)
        patch2f_in = [patch2f_in[p_id] for p_id in patch2f_in]

        for i in range(len(patch2f_in)):
            for j in range(len(patch2f_in[i])):
                if patch2f_in[i][j] in f_in:
                    patch2f_in[i][j] = f_in.index(patch2f_in[i][j])


        patch2f_out = defaultdict(list)
        for f_id in f_out:
            if f_id >= raw_f_len:
                p_id = new_f_p_id[f_id]
            else:
                raw_f_id = new_f_id2raw_f_id[f_id]
                p_id = self.patch_mesh.f2patch[raw_f_id]
            patch2f_out[p_id].append(f_id)
        patch2f_out = [patch2f_out[p_id] for p_id in patch2f_out]

        for i in range(len(patch2f_out)):
            for j in range(len(patch2f_out[i])):
                if patch2f_out[i][j] in f_out:
                    patch2f_out[i][j] = f_out.index(patch2f_out[i][j])
        

        # final split result
        c, axis, r = cylinder_params[:3].squeeze(), cylinder_params[3:6].squeeze(), cylinder_params[6].item()
        v0, f0 = get_submesh(new_mesh_vertices, new_mesh_faces, f_in)
        cylinder_filling = CylinderHoleFilling(v0, f0, c, axis, r, max_area=1.0, visualize=True)
        v0, f0, adding_f_id = cylinder_filling.new_v, cylinder_filling.new_f, cylinder_filling.adding_f_id
        if len(adding_f_id) != 0:
            patch2f_in.append(list(adding_f_id))
        # hole_filling0 = HoleFilling(v0, f0, visualize=False)
        # new_v0, new_f0, filled_regions = hole_filling0.new_v, hole_filling0.new_f, hole_filling0.region_triangles
        # if len(filled_regions) != 0:
        #     cur_id = f0.shape[0]
        #     for region in filled_regions:
        #         patch2f_in.append(list(np.arange(cur_id, cur_id + region.shape[0])))
        #         cur_id += len(region)
        # v0, f0 = new_v0, new_f0
    

        v1, f1 = get_submesh(new_mesh_vertices, new_mesh_faces, f_out)
        cylinder_filling = CylinderHoleFilling(v1, f1, c, axis, r, max_area=1.0, visualize=True)
        v1, f1, adding_f_id = cylinder_filling.new_v, cylinder_filling.new_f, cylinder_filling.adding_f_id
        if len(adding_f_id) != 0:
            patch2f_out.append(list(adding_f_id))
        # hole_filling1 = HoleFilling(v1, f1, visualize=False)
        # new_v1, new_f1, filled_regions =  hole_filling1.new_v, hole_filling1.new_f, hole_filling1.region_triangles
        # if len(filled_regions) != 0:
        #     cur_id = f1.shape[0]
        #     for region in filled_regions:
        #         patch2f_out.append(list(np.arange(cur_id, cur_id + region.shape[0])))
        #         cur_id += len(region)
        # v1, f1 = new_v1, new_f1
        

        mp.plot(v0, f0, np.ones((f0.shape[0], 3)), shading={'wireframe': True})
        mp.plot(v1, f1, np.ones((f1.shape[0], 3)), shading={'wireframe': True})

        return True, [
            {
                'v': v0, 'f': f0, 'patch': patch2f_in
            },
            {
                'v': v1, 'f': f1, 'patch': patch2f_out
            }
        ]
        

    def cut_mesh_by_plane(self, plane_param):
        e_cut = {}
        mesh = self.patch_mesh.om_mesh
        insert_v = []
        insert_v_num = 0
        raw_v_num = self.patch_mesh.v.shape[0]
        for e_h in mesh.edges():
            v0, v1 = e_h2v_p(mesh, e_h)
            
            d0 = np.dot(plane_param[:3], v0) + plane_param[3]
            d1 = np.dot(plane_param[:3], v1) + plane_param[3]
            

            epsilon = 1e-10
            # if (d0 * d1 < 0):
            if (d0 * d1 < 0) and (abs(d0) > epsilon) and (abs(d1) > epsilon):
                x0, y0, z0 = v0
                x1, y1, z1 = v1
                a, b, c, d = plane_param
                
                denominator = a * (x1 - x0) + b * (y1 - y0) + c * (z1 - z0)
                numerator = -(a * x0 + b * y0 + c * z0 + d)
                
                
                if abs(denominator) > epsilon:
                    t = numerator / denominator
                    # intersection in line segment
                    # if -epsilon <= t <= 1 + epsilon:
                        # t_clamped = np.clip(t, 0, 1)
                        # x_new = x0 + t_clamped * (x1 - x0)
                        # y_new = y0 + t_clamped * (y1 - y0)
                        # z_new = z0 + t_clamped * (z1 - z0)
                    t = np.clip(t, 0, 1)
                    in_v = v0 + t * (v1 - v0)
                    insert_v.append(in_v)
                    e_cut[e_h.idx()] = raw_v_num + insert_v_num
                    insert_v_num += 1

                        # v_new_id = self.patch_mesh.v.shape[0]
                        # self.patch_mesh.v = np.concatenate((self.patch_mesh.v, np.array([x_new, y_new, z_new])[np.newaxis, :]))
                        # e_cut[e_h.idx()] = v_new_id
                    # else:
                        # print(e_h.idx())
                        # print("Intersection out of line segment, but points on different sides")
                else:
                    # parallel or on plane, has been covered
                    # print(e_h.idx())
                    print("Parallel or on plane, but points on different sides")

        insert_v = np.array(insert_v)
        new_mesh_v = np.concatenate((self.patch_mesh.v, insert_v), axis=0)

        # show
        # plot = mp.plot(self.patch_mesh.v, self.patch_mesh.f, np.ones((self.patch_mesh.f_num, 3)), shading={"wireframe": True})
        # plot.add_points(
        #     insert_v,
        #     shading={
        #         "point_color": "red",
        #         "point_size": 3
        #     }
        # )
        # f_center = np.mean(self.patch_mesh.v[self.patch_mesh.f], axis=1)
        # f_normal = np.cross(self.patch_mesh.v[self.patch_mesh.f[:, 1]] - self.patch_mesh.v[self.patch_mesh.f[:, 0]], self.patch_mesh.v[self.patch_mesh.f[:, 2]] - self.patch_mesh.v[self.patch_mesh.f[:, 1]])
        # f_normal = f_normal / np.linalg.norm(f_normal, axis=1)[:, np.newaxis]
        # # draw normal direction
        # plot.add_lines(
        #     f_center,
        #     f_center + f_normal,
        #     shading={
        #         "line_color": "red",
        #         "line_width": 1
        #     }
        # )
        
        # triangulate
        # cut and remove faces
        f_cut = {}
        for e in e_cut:
            e_h = mesh.edge_handle(e)
            nf0, nf1 = e_h2nf(mesh, e_h)
            # he_h0, he_h1 = mesh.halfedge_handle(e_h, 0), mesh.halfedge_handle(e_h, 1)
            # f0, f1 = mesh.face_handle(he_h0), mesh.face_handle(he_h1)

            if nf0 not in f_cut:
                f_cut[nf0] = []
            if nf1 not in f_cut:
                f_cut[nf1] = []
            f_cut[nf0].append(e)
            f_cut[nf1].append(e)
            # if (f0.idx() != -1) & (f0.idx() not in f_cut):
            #     f_cut[f0.idx()] = []
            # if (f1.idx() != -1) & (f1.idx() not in f_cut):
            #     f_cut[f1.idx()] = []
            # if f0.idx() != -1:
            #     f_cut[f0.idx()].append(e)
            # if f1.idx() != -1:
            #     f_cut[f1.idx()].append(e)

        
        # insert faces
        # insert new_f[f] in face f
        new_f = {}
        for f in f_cut.keys():
            v_new_id = [e_cut[e] for e in f_cut[f]]
            new_f[f] = []
            ''' 
                insert one
            '''
            if len(v_new_id) == 1:
                v_new_id = v_new_id[0]

                # find opposite vertex
                e = f_cut[f][0]
                op_v_id = e_h2ov_in_face(mesh, mesh.edge_handle(e), self.patch_mesh.f[f])
                v_raw = self.patch_mesh.f[f]
                op_v_index = np.where(v_raw == op_v_id)[0][0]
                
                # insert
                new_f[f].append([v_raw[op_v_index], v_raw[(op_v_index + 1)%3], v_new_id])
                new_f[f].append([v_raw[(op_v_index - 1)%3], v_raw[op_v_index], v_new_id])
                
            elif len(v_new_id) == 2:
                # find first opposite vertex
                v_new_id0 = v_new_id[0]
                e = f_cut[f][0]
                op_v_id0 = e_h2ov_in_face(mesh, mesh.edge_handle(e), self.patch_mesh.f[f])
                v_raw = self.patch_mesh.f[f]
                op_v_index0 = np.where(v_raw == op_v_id0)[0][0]
                
                # find second opposite vertex
                v_new_id1 = v_new_id[1]
                e = f_cut[f][1]
                op_v_id1 = e_h2ov_in_face(mesh, mesh.edge_handle(e), self.patch_mesh.f[f])
                op_v_index1 = np.where(v_raw == op_v_id1)[0][0]
                
                if op_v_index1 == (op_v_index0 + 1)%3:
                    new_f[f].append([v_raw[op_v_index0], v_raw[(op_v_index0 + 1)%3], v_new_id0])
                    new_f[f].append([v_new_id0, v_raw[(op_v_index0 - 1)%3], v_new_id1])
                    new_f[f].append([v_raw[op_v_index0], v_new_id0, v_new_id1])
                else:
                    new_f[f].append([v_raw[(op_v_index0 - 1)%3], v_raw[op_v_index0], v_new_id0])
                    new_f[f].append([v_new_id0, v_raw[op_v_index0], v_new_id1])
                    new_f[f].append([v_raw[(op_v_index0 + 1)%3], v_new_id0, v_new_id1])
            new_f[f] = np.array(new_f[f])

        # cleanup duplicated faces
        f_cut_id = list(f_cut.keys())
        new_mesh_f = np.delete(self.patch_mesh.f, f_cut_id, axis=0)
        new_f_id2raw_f_id = [i for i in range(len(self.patch_mesh.f)) if i not in f_cut_id]
        raw_f_len = new_mesh_f.shape[0]
        # new_mesh_f = self.patch_mesh.f.copy()
        new_f_p_id = {}
        new_f_id = new_mesh_f.shape[0]
        for f in new_f.keys():
            if len(new_f[f]) == 0:
                continue
            for _ in range(new_f[f].shape[0]):
                new_f_p_id[new_f_id] = self.patch_mesh.f2patch[f]
                new_f_id += 1
            new_mesh_f = np.concatenate((new_mesh_f, new_f[f]))
        
        f_colors = np.ones((new_mesh_f.shape[0], 3))
        for i in range(f_colors.shape[0]):
            if i < raw_f_len:
                f_colors[i] = colors[self.patch_mesh.f2patch[new_f_id2raw_f_id[i]]]
            else:
                f_colors[i] = colors[new_f_p_id[i]]
        mp.plot(new_mesh_v, new_mesh_f, f_colors)

        # split mesh
        # if   |center - plane| > threshold, split
        # else consider (face-normal · plane-normal)
        f_in = set()
        f_out = set()
        threshold = 1e-6
        for i in range(new_mesh_f.shape[0]):
            center = np.mean(new_mesh_v[new_mesh_f[i]], axis=0)
            d = np.dot(plane_param[:3], center) + plane_param[3]
            if d > threshold:
                f_in.add(i)
            elif d < -threshold:
                f_out.add(i)
            else:
                v = new_mesh_v[new_mesh_f[i]]
                n = np.cross(v[1] - v[0], v[2] - v[1])
                if np.dot(n, plane_param[:3]) < 0:
                    f_in.add(i)
                else:
                    f_out.add(i)
        f_in, f_out = list(f_in), list(f_out)

        # convert patch
        patch2f_in = {}
        for f_id in f_in:
            if f_id < raw_f_len:
                raw_f_id = new_f_id2raw_f_id[f_id]
                p_id = self.patch_mesh.f2patch[raw_f_id]
            else:
                p_id = new_f_p_id[f_id]
            if p_id not in patch2f_in:
                patch2f_in[p_id] = []
            patch2f_in[p_id].append(f_id)
        patch2f_in = [patch2f_in[p_id] for p_id in patch2f_in]
        patch2f_out = {}
        for f_id in f_out:
            if f_id < raw_f_len:
                raw_f_id = new_f_id2raw_f_id[f_id]
                p_id = self.patch_mesh.f2patch[raw_f_id]
            else:
                p_id = new_f_p_id[f_id]
            if p_id not in patch2f_out:
                patch2f_out[p_id] = []
            patch2f_out[p_id].append(f_id)
        patch2f_out = [patch2f_out[p_id] for p_id in patch2f_out]
        
        v0, f0 = get_submesh(new_mesh_v, new_mesh_f, f_in)
        mp.plot(v0, f0, np.ones((f0.shape[0], 3)), shading={"wireframe": True})
        # substitute f in patch2f0/patch2f1 with f_in/f_out
        for i in range(len(patch2f_in)):
            for j in range(len(patch2f_in[i])):
                if patch2f_in[i][j] in f_in:
                    patch2f_in[i][j] = f_in.index(patch2f_in[i][j])
        for i in range(len(patch2f_out)):
            for j in range(len(patch2f_out[i])):
                if patch2f_out[i][j] in f_out:
                    patch2f_out[i][j] = f_out.index(patch2f_out[i][j])
        
        
        v1, f1 = get_submesh(new_mesh_v, new_mesh_f, f_out)
        mp.plot(v1, f1, np.ones((f1.shape[0], 3)), shading={"wireframe": True})
        
        
        return True, [{"v": v0, "f": f0, "patch": patch2f_in}, {"v": v1, "f": f1, "patch": patch2f_out}]


    def cut_mesh_by_cylinder(self, cylinder_param, v_id, p1, p2):
        cylinder_param = np.squeeze(np.asarray(cylinder_param))
        cylinder_center = cylinder_param[0:3]
        cylinder_normal = cylinder_param[3:6]
        cylinder_normal /= np.linalg.norm(cylinder_normal)
        cylinder_radius = cylinder_param[6]
        
        def point_in_cylinder(v):
            dist = np.linalg.norm(np.cross(v - cylinder_center, cylinder_normal), axis=1)
            return dist <= cylinder_radius + 1e-1
        
        # mesh = self.patch_mesh.om_mesh
        # v_in, v_out = [], []
        # for v_h in mesh.vertices():
        #     v_pos = mesh.point(v_h)
        #     v_in_cylinder = point_in_cylinder(v_pos.reshape(1, -1))
        #     if v_in_cylinder:
        #         v_in.append(v_h.idx())
        #     else:
        #         v_out.append(v_h.idx())
        
        # v_color = np.zeros((self.patch_mesh.v.shape[0], 3))
        # v_color[v_in] = [0., 1., 0.]
        # v_color[v_out] = [1., 0., 0.]
        # mp.plot(self.patch_mesh.v, self.patch_mesh.f, c=v_color, shading={"wireframe": True})

        # e_cut = {}
        # for e_h in mesh.edges():
        #     he_h = mesh.halfedge_handle(e_h, 0)
        #     v0_h = mesh.to_vertex_handle(he_h)
        #     v1_h = mesh.from_vertex_handle(he_h)
        #     v0 = mesh.point(v0_h)
        #     v1 = mesh.point(v1_h)
        #     v0_in_cylinder = point_in_cylinder(v0.reshape(1, -1))
        #     v1_in_cylinder = point_in_cylinder(v1.reshape(1, -1))

        #     if v0_in_cylinder != v1_in_cylinder:


        
        
        # find patches where cylinder project to
        target_p_id = []
        for p_id in self.patch_mesh.graph.nodes:
            if p_id == p1:
                continue
            if p_id == p2:
                continue
            if self.patch_mesh.graph.nodes[p_id]["type"] == "Plane":
                patch_normal = self.patch_mesh.graph.nodes[p_id]["params"][:3]
                patch_normal /= np.linalg.norm(patch_normal)
                if abs(np.dot(cylinder_normal, patch_normal)) < 5*1e-2:
                    continue
                target_p_id.append(p_id)
            else:
                #TODO: test other shape
                continue
        
        # # show
        # f_colors = np.ones((self.patch_mesh.f.shape[0], 3))
        # target_p_id = [0]
        # for p_id in target_p_id:
        #     f_id = self.patch_mesh.graph.nodes[p_id]['f']
        #     f_colors[f_id] = [1., 0., 0.]
        # plot = mp.plot(self.patch_mesh.v, self.patch_mesh.f, f_colors, shading={"wireframe": True})
        
        
        plane_param = self.patch_mesh.graph.nodes[target_p_id[0]]["params"]
        
        v_project = np.zeros((len(v_id), 3))
        for i in range(len(v_id)):
            v_pos = self.patch_mesh.v[v_id[i]]
            # project to plane
            d = np.dot(plane_param[:3], v_pos) + plane_param[3]
            v_project[i] = v_pos - d*plane_param[:3]
        
        # show
        # plot = mp.plot(self.patch_mesh.v, self.patch_mesh.f, np.ones((self.patch_mesh.f.shape[0], 3)), shading={"wireframe": True})
        # plot.add_points(
        #     v_project,
        #     shading={
        #         "point_color": "red",
        #         "point_size": 3
        #     }
        # )
        
        # test point in triangle
        def is_point_in_triangle(P, A, B, C):
            epsilon = 1e-6
            vec_ab = B - A
            vec_ac = C - A
            cross_abc = np.cross(vec_ab, vec_ac)
            area_abc = abs(cross_abc) / 2
            
            if area_abc < epsilon:
                return False
            
            vec_ap = P - A
            cross_abp = np.cross(vec_ab, vec_ap)
            area_abp = abs(cross_abp) / 2
            
            vec_bc = C - B
            vec_bp = P - B
            cross_bcp = np.cross(vec_bc, vec_bp)
            area_bcp = abs(cross_bcp) / 2
            
            vec_ca = A - C
            vec_cp = P - C
            cross_cap = np.cross(vec_ca, vec_cp)
            area_cap = abs(cross_cap) / 2
            
            total_area = area_abp + area_bcp + area_cap
            
            return abs(total_area - area_abc) < epsilon
        
        v_project_2d = v_project[:, 1:]
        v_in_f = np.zeros(v_project.shape[0] - 2, dtype=int)
        for i in range(v_project.shape[0] - 2):
            cur_v = v_project_2d[i+1]
            for f_id in self.patch_mesh.graph.nodes[target_p_id[0]]["f"]:
                A, B, C = self.patch_mesh.v[self.patch_mesh.f[f_id]][:, 1:]
                if is_point_in_triangle(cur_v, A, B, C):
                    v_in_f[i] = f_id
                    break
        

        v_new = v_project[1:-1]
        f_remove = set()
        v_triangle_id = {
            0: 0,
            3745: 0,
            3744: 0,
            35: 0,
            256: 0,
            74: 0
        }
        for f in v_in_f:
            f_remove.add(f)
            v_triangle_id[f] += 1
        
        visit_v_triangle_id = {
            0: 0,
            3745: 0,
            3744: 0,
            35: 0,
            256: 0,
            74: 0
        }
        for i in range(v_project.shape[0] - 2):
            cur_v_id = self.patch_mesh.v.shape[0]
            cur_f = self.patch_mesh.f[v_in_f[i]]
            self.patch_mesh.v = np.concatenate((self.patch_mesh.v, v_new[i].reshape(1, 3)), axis=0)
            visit_v_triangle_id[v_in_f[i]] += 1
            
            if (v_in_f[i] == 35):
                if visit_v_triangle_id[v_in_f[i]] == 1:
                    self.patch_mesh.f = np.concatenate((self.patch_mesh.f, np.array([
                        [cur_v_id, cur_f[0], cur_f[1]]
                    ])), axis=0)
                else:
                    if visit_v_triangle_id[v_in_f[i]] == v_triangle_id[v_in_f[i]]:
                        self.patch_mesh.f = np.concatenate((self.patch_mesh.f, np.array([
                            [cur_v_id, cur_f[2], cur_f[0]]
                        ])), axis=0)
                    self.patch_mesh.f = np.concatenate((self.patch_mesh.f, np.array([
                        [cur_v_id-1, cur_v_id, cur_f[0]],
                        [cur_v_id, cur_v_id-1, cur_f[1]]
                    ])), axis=0)
            elif (v_in_f[i] == 0):
                self.patch_mesh.f = np.concatenate((self.patch_mesh.f, np.array([
                    [cur_v_id-1, cur_v_id, cur_f[0]],
                    [cur_v_id, cur_v_id-1, cur_f[1]]
                ])), axis=0)
                if visit_v_triangle_id[v_in_f[i]] == v_triangle_id[v_in_f[i]]:
                    self.patch_mesh.f = np.concatenate((self.patch_mesh.f, np.array([
                        [cur_v_id, cur_f[1], cur_f[2]]
                    ])), axis=0)
            elif (v_in_f[i] == 3745):
                self.patch_mesh.f = np.concatenate((self.patch_mesh.f, np.array([
                    [cur_v_id-1, cur_v_id, cur_f[1]],
                    [cur_v_id, cur_v_id-1, cur_f[2]]
                ])), axis=0)
                if visit_v_triangle_id[v_in_f[i]] == v_triangle_id[v_in_f[i]]:
                    self.patch_mesh.f = np.concatenate((self.patch_mesh.f, np.array([
                        [cur_v_id, cur_f[0], cur_f[1]]
                    ])), axis=0)
            elif (v_in_f[i] == 3744):
                self.patch_mesh.f = np.concatenate((self.patch_mesh.f, np.array([
                    [cur_v_id-1, cur_v_id, cur_f[0]],
                    [cur_v_id, cur_v_id-1, cur_f[2]]
                ])), axis=0)
                if visit_v_triangle_id[v_in_f[i]] == v_triangle_id[v_in_f[i]]:
                    self.patch_mesh.f = np.concatenate((self.patch_mesh.f, np.array([
                        [cur_v_id, cur_f[0], cur_f[1]]
                    ])), axis=0)
            elif (v_in_f[i] == 74):
                self.patch_mesh.f = np.concatenate((self.patch_mesh.f, np.array([
                    [cur_v_id-1, cur_v_id, cur_f[1]],
                    [cur_v_id, cur_v_id-1, cur_f[2]]
                ])), axis=0)
                if visit_v_triangle_id[v_in_f[i]] == v_triangle_id[v_in_f[i]]:
                    self.patch_mesh.f = np.concatenate((self.patch_mesh.f, np.array([
                        [cur_v_id, cur_f[0], cur_f[2]]
                    ])), axis=0)
            elif (v_in_f[i] == 256):
                self.patch_mesh.f = np.concatenate((self.patch_mesh.f, np.array([
                    [cur_v_id-1, cur_v_id, cur_f[1]],
                    [cur_v_id, cur_v_id-1, cur_f[2]]
                ])), axis=0)
                if visit_v_triangle_id[v_in_f[i]] == v_triangle_id[v_in_f[i]]:
                    self.patch_mesh.f = np.concatenate((self.patch_mesh.f, np.array([
                        [cur_v_id, cur_f[0], cur_f[2]],
                        [cur_v_id, cur_f[0], cur_f[1]],
                    ])), axis=0)
            

            
        f_remove_arr = np.array(list(f_remove))        
        self.patch_mesh.f = np.delete(self.patch_mesh.f, f_remove_arr, axis=0)

        # plot = mp.plot(self.patch_mesh.v, self.patch_mesh.f, np.ones((self.patch_mesh.f.shape[0], 3)), shading={"wireframe": True})
        # for v_id in range(len(v_in_f)):
        #     _v = self.patch_mesh.v[raw_v_len + v_id]
        #     plot.add_points(
        #         _v.reshape(1, 3),
        #         c=np.array(colors[v_in_f[v_id] % 500]).reshape(1, 3),
        #         shading={
        #             "point_size": 3
        #         }
        #     )

        
        # split mesh
        f_in = []
        f_out = []
        for i in range(self.patch_mesh.f.shape[0]):
            center = np.mean(self.patch_mesh.v[self.patch_mesh.f[i]], axis=0)
            in_cylinder = point_in_cylinder(center.reshape(1, -1))
            if in_cylinder:
                f_in.append(i)
            else:
                f_out.append(i)
        
        v0, f0 = get_submesh(self.patch_mesh.v, self.patch_mesh.f, f_in)
        # mp.plot(v0, f0, np.ones((f0.shape[0], 3)), shading={"wireframe": True})
        
        v1, f1 = get_submesh(self.patch_mesh.v, self.patch_mesh.f, f_out)
        # mp.plot(v1, f1, np.ones((f1.shape[0], 3)), shading={"wireframe": True})

        # hole_filling0 = HoleFilling(v0, f0, visualize=True)
        # filled_f0 = hole_filling0.new_f
        # mp.plot(v0, filled_f0, np.ones((filled_f0.shape[0], 3)), shading={"wireframe": True})
        
        # hole_filling1 = HoleFilling(v1, f1, visualize=True)
        # filled_f1 = hole_filling1.new_f
        # mp.plot(v1, filled_f1, np.ones((filled_f1.shape[0], 3)), shading={"wireframe": True})
    
        
        return True, [{"v": [v0], "f": [f0], "patch": []}, {"v": [v1], "f": [f1], "patch": []}]

    
    def project_vert_to_patch(self, v, axis, p_id):
        p_type = self.patch_mesh.graph.nodes[p_id]['type']
        p_param = self.patch_mesh.graph.nodes[p_id]['params']
        
        if p_type == 'Plane':
            v_proj = proj_vert_to_plane(v, axis, p_param)

        else:
            raise Exception(f'[PROJECT]: not support project to {p_type}')
        
        return v_proj


    '''
        Cut mesh by embedded cylinder with closed cylinder curve
            1.distinguish v by cylinder
            2.find projection patch p
            3.project v_border to p
            4.triangulate
                4.1 triangulate: v(p, out), boundary(p, out), v_border_proj, boundary(v_border_proj)
                4.2 triangulate: v(p, in), boundary(p, in), v_border_proj, boundary(v_border_proj)
                4.3 side: v_border, v_border_proj
        INPUT
            - neighbor patch pair: dict{
                    p0:{id, type, params},
                    p1:{id, type, params},
                    border_v: [...]}
            - cylinder id: p \in {p0, p1}
    '''
    def cut_mesh_by_close_cylinder(self, neighbor_patch_pair):
        # initial
        p0_type, p1_type = neighbor_patch_pair['p0']['type'], neighbor_patch_pair['p1']['type']
        border_v = neighbor_patch_pair['border_v'][:-1]
        if (p0_type == 'Cylinder') & (p1_type == 'Plane'):
            p_n_side_param = neighbor_patch_pair['p0']['params']
            p_top_id = neighbor_patch_pair['p1']['id']
        elif (p1_type == 'Cylinder') & (p0_type == 'Plane'):
            p_n_side_param = neighbor_patch_pair['p1']['params']
            p_top_id = neighbor_patch_pair['p0']['id']
        else:
            raise Exception(f"close cylinder border with {p0_type} and {p1_type} neighbor")
        
        c, axis, r = p_n_side_param[0:3].squeeze(), p_n_side_param[3:6].squeeze(), p_n_side_param[6].item()

        # 1.distinguish v by cylinder
        v_on_patch = [vert_on_cylinder(c, axis, r, p) for p in self.patch_mesh.v]
        
        
        # 2.find destination p
        # 2.1 all edges intersect with cylinder
        threshold_parallel = 1e-2
        mesh = self.patch_mesh.om_mesh
        e_cut = {} # dict{e_id: [v_in, v_out]}
        for e_h in mesh.edges():
            v0_id, v1_id = e_h2v(mesh, e_h)            
            v0_type, v1_type = v_on_patch[v0_id], v_on_patch[v1_id]
            if v0_type * v1_type < 0:
                if (v0_type == 1) & (v1_type == -1):
                    e_cut[e_h.idx()] = [v0_id, v1_id]
                elif (v0_type == -1) & (v1_type == 1):
                    e_cut[e_h.idx()] = [v1_id, v0_id]
                
        # 2.2 find destinate patch
        v_all = set()
        for e in e_cut.values():
            v_all = v_all.union(set(e))
        v_all = list(v_all)
        v_all_position = []
        for v_id in v_all:
            v_all_position.extend(self.patch_mesh.v2patch[v_id])

        counter = Counter(v_all_position)
        most_count = counter.most_common()
        most_count_id = [item[0] for item in most_count]
        
        p_proj = most_count_id[0]

        self.show_single_patch(p_proj)


        # 3. project to patch & get all boundary and holes
        p_proj_param = self.patch_mesh.graph.nodes[p_proj]['params']
        # 3.1 project
        plane_normal = p_proj_param[:3]
        origin_3d = get_origin_from_plane(p_proj_param)
        u, v = get_uv_from_normal(plane_normal)
        all_v_proj_2d = np.array([project_point_to_2d(p, origin_3d, u, v) for p in self.patch_mesh.v])
        v_border_proj_3d = np.array([self.project_vert_to_patch(self.patch_mesh.v[v_id], axis, p_proj) for v_id in border_v])
        v_border_proj_2d = all_v_proj_2d[border_v]

        # 3.2 get boundary and holes of patch
        p_f = self.patch_mesh.f[self.patch_mesh.patch2f[p_proj]]
        all_boundarys = igl.boundary_loop_all(p_f)
        # distinguish boundary and holes: loop with largest area
        def polygon_area_2d(points):
            x, y = points[:,0], points[:,1]
            return 0.5 * np.sum(x*np.roll(y, -1) - y*np.roll(x, -1))
        areas = []
        for loop in all_boundarys:
            pts = all_v_proj_2d[loop]
            area = polygon_area_2d(pts)
            areas.append(area)
        
        outer_idx = np.argmax(np.abs(areas))
        outer = all_boundarys[outer_idx]
        holes = [loop for i, loop in enumerate(all_boundarys) if i != outer_idx]

        # distinguish holes inside or outside projection
        holes_inside = []
        holes_inside_id = []
        holes_outside = []
        holes_outside_id = []
        for hole in holes:
            sample_p = all_v_proj_2d[hole[0]]
            hole_2d = np.array(all_v_proj_2d[hole])
            if point_in_2d_polygon(sample_p, v_border_proj_2d):
                holes_inside.append(hole_2d)
                holes_inside_id.append(hole)
            else:
                holes_outside.append(hole_2d)
                holes_outside_id.append(hole)
        
        # 4. triangulate
        # 4.1 projection plane
        # triangulate [boundary, [projection, hole_outside]]
        outer_v_proj_2d, outer_triangles = triangulate_2d_plane_with_hole(
            np.array(all_v_proj_2d[outer]),
            holes_outside + [v_border_proj_2d]
            # [v_border_proj_2d] + holes_outside
            # min_angle=10
        )
        # triangulate [projection, [hole_inside]]
        inner_v_proj_2d, inner_triangles = triangulate_2d_plane_with_hole(
            v_border_proj_2d,
            holes_inside
            # min_angle=10
        )
        # output1: [outer, hole_outside, v_border_proj, other1]
        # output2: [v_border_proj, hole_inside, other2]
        # recover v
        # adding v_border_proj_3d, adding others1, adding others2
        len_v_proj = v_border_proj_3d.shape[0]
        len_outer = len(outer)
        len_holes_inside = sum([hole.shape[0] for hole in holes_inside])
        len_holes_outside = sum([hole.shape[0] for hole in holes_outside])

        len_other1 = outer_v_proj_2d.shape[0] - len_outer - len_holes_outside - len_v_proj
        len_other2 = inner_v_proj_2d.shape[0] - len_v_proj - len_holes_inside
        assert (len_other1 >= 0) & (len_other2 >= 0)
        
        if len_other1 == 0:
            v_other1 = np.empty((0, 3))
        else:
            v_other1 = np.array([reproject_point_2d_to_plane(p, origin_3d, u, v, p_proj_param) \
                    for p in outer_v_proj_2d[-len_other1:]])
        if len_other2 == 0:
            v_other2 = np.empty((0, 3))
        else:
            v_other2 = np.array([reproject_point_2d_to_plane(p, origin_3d, u, v, p_proj_param) \
                    for p in inner_v_proj_2d[-len_other2:]])
        
        cur_v_len = self.patch_mesh.v.shape[0]
        id_v_border_proj = list(range(cur_v_len, cur_v_len + len_v_proj))
        cur_v_len += len_v_proj
        id_v_other1 = list(range(cur_v_len, cur_v_len + len_other1))
        cur_v_len += len_other1
        id_v_other2 = list(range(cur_v_len, cur_v_len + len_other2))
        
        new_v = np.concatenate((
            self.patch_mesh.v,
            v_border_proj_3d,
            v_other1,
            v_other2
        ), axis=0)

        # recover f
        # map: output1 -> new_v
        map_output1 = outer.copy()
        for hole_outside_id in holes_outside_id:
            map_output1.extend(hole_outside_id)
        map_output1.extend(id_v_border_proj)
        map_output1.extend(id_v_other1)
        outer_triangles = np.array([[map_output1[i] for i in tri] for tri in outer_triangles])

        # map: output2 -> new_v
        map_output2 = id_v_border_proj.copy()
        for hole_inside_id in holes_inside_id:
            map_output2.extend(hole_inside_id)
        map_output2.extend(id_v_other2)
        inner_triangles = np.array([[map_output2[i] for i in tri] for tri in inner_triangles])


        # check direction
        example_f_id = self.patch_mesh.patch2f[p_proj][0]
        example_f_coord = self.patch_mesh.v[self.patch_mesh.f[example_f_id]]
        raw_p_proj_normal = np.cross(
            example_f_coord[1] - example_f_coord[0],
            example_f_coord[2] - example_f_coord[1]
        )
        for i in range(outer_triangles.shape[0]):
            f_coord = new_v[outer_triangles[i]]
            f_normal = np.cross(
                f_coord[1] - f_coord[0],
                f_coord[2] - f_coord[1]
            )
            if np.dot(f_normal, raw_p_proj_normal) < 0:
                # reverse
                outer_triangles[i] = outer_triangles[i][::-1]
        for i in range(inner_triangles.shape[0]):
            f_coord = new_v[inner_triangles[i]]
            f_normal = np.cross(
                f_coord[1] - f_coord[0],
                f_coord[2] - f_coord[1]
            )
            if np.dot(f_normal, raw_p_proj_normal) < 0:
                # reverse
                inner_triangles[i] = inner_triangles[i][::-1]

        # new_f = np.concatenate((
        #     self.patch_mesh.f,
        #     outer_triangles,
        #     # inner_triangles
        # ), axis=0)
        # new_f = np.delete(new_f, self.patch_mesh.patch2f[p_proj], axis=0)

        # mp.plot(new_v, new_f, np.ones((new_f.shape[0], 3)), shading={
        #     'wireframe': True
        # })

        # new_f = np.concatenate((
        #     self.patch_mesh.f,
        #     # outer_triangles,
        #     inner_triangles
        # ), axis=0)
        # new_f = np.delete(new_f, self.patch_mesh.patch2f[p_proj], axis=0)

        # mp.plot(new_v, new_f, np.ones((new_f.shape[0], 3)), shading={
        #     'wireframe': True
        # })

        
        new_f = np.delete(self.patch_mesh.f, self.patch_mesh.patch2f[p_proj], axis=0)
        # new_f_id2raw_f_id = {i: i for i in range(raw_f_len) if i not in self.patch_mesh.patch2f[p_proj]}
        new_f = np.concatenate((
            new_f,
            outer_triangles,
            inner_triangles
        ), axis=0)
        raw_f_len = new_f.shape[0]

        mp.plot(new_v, new_f, np.ones((new_f.shape[0], 3)), shading={
            'wireframe': True
        })

        # 4.2 construct new mesh
        # split by v_border and project_v_border
        cutting_border_vert = []
        for i in range(len(border_v)):
            cutting_border_vert.append([border_v[i], border_v[(i+1)%len(border_v)]])
        for i in range(len(id_v_border_proj)):
            cutting_border_vert.append([id_v_border_proj[i], id_v_border_proj[(i+1)%len(id_v_border_proj)]])

        # check
        plot = mp.plot(new_v, new_f, np.ones((new_f.shape[0], 3)), shading={'wireframe': True})
        cutting_border_vert_array = np.array(cutting_border_vert)
        plot.add_lines(
            new_v[cutting_border_vert_array[:, 0]],
            new_v[cutting_border_vert_array[:, 1]],
            shading={
                "line_color": "red",
                "line_width": 1
            }
        )

        # cutting_border_vert_ordered = np.sort(cutting_border_vert, axis=1)
        cutting_border_set = set((int(min(a, b)), int(max(a, b))) for a, b in cutting_border_vert)

        cutting_graph = nx.Graph()
        # add node
        for i in range(new_f.shape[0]):
            cutting_graph.add_node(i)
        # edge to face
        edge2face = defaultdict(list)
        for f_id in range(new_f.shape[0]):
            for i in range(3):
                v0, v1 = new_f[f_id][i], new_f[f_id][(i + 1) % 3]
                if v0 > v1:
                    v0, v1 = v1, v0
                edge2face[(v0, v1)].append(f_id)
        # add edges
        for edge, n_f in edge2face.items():
            if len(n_f) == 2:
                if (edge[0], edge[1]) in cutting_border_set:
                    continue
                cutting_graph.add_edge(n_f[0], n_f[1])
        components = list(nx.connected_components(cutting_graph))
        
        # f_colors = np.ones((new_f.shape[0], 3))
        # plot = mp.plot(new_v, new_f, f_colors, shading={'wireframe': True})
        # for comp in components:
        #     f_colors = np.ones((new_f.shape[0], 3))
        #     f_colors[list(comp), :] = [1, 0, 0]
        #     plot.update_object(colors = f_colors)
        if len(components) != 2:
            raise ValueError(f"[SPLIT ERROR]: tracing cut graph components {len(components)} != 2")
        raw_f_out, raw_f_in = list(components[0]), list(components[1])

        # inside mesh: v_border and project_v_border
        f_in_side_faces = []
        for i in range(len_v_proj):
            f_in_side_faces.append([id_v_border_proj[i], id_v_border_proj[(i+1)%len_v_proj], border_v[(i+1)%len_v_proj]])
            f_in_side_faces.append([id_v_border_proj[i], border_v[(i+1)%len_v_proj], border_v[i]])
            # f_in_side_faces.append([id_v_border_proj[(i+1)%len_v_proj], id_v_border_proj[i], border_v[(i+1)%len_v_proj]])
            # f_in_side_faces.append([border_v[(i+1)%len_v_proj], id_v_border_proj[i], border_v[i]])
        f_in_side_faces = np.array(f_in_side_faces)
        # check normal direction outof cylinder axis
        f_in_side_faces, _ = fix_face_normals_for_cylinder(new_v, f_in_side_faces, c, axis)

        # f_in = np.concatenate((new_f[raw_f_in], f_in_side_faces))
        f_in = raw_f_in
        # f_in = raw_f_in + list(range(raw_f_len, raw_f_len + f_in_side_faces.shape[0]))
        new_f = np.concatenate((new_f, f_in_side_faces), axis=0)
        raw_f_len = new_f.shape[0]

        # outside mesh: v_border and project_v_border
        f_out_side_faces = f_in_side_faces[:, [0, 2, 1]]
        # f_out = np.concatenate((new_f[raw_f_out], f_out_side_faces))
        f_out = raw_f_out + list(range(raw_f_len, raw_f_len + f_out_side_faces.shape[0]))
        new_f = np.concatenate((new_f, f_out_side_faces), axis=0)
        # raw_f_len = new_f.shape[0]

        # show_mesh(new_v, f_in, show_normal=False)
        # show_mesh(new_v, f_out, show_normal=False)


        # patch2f_in = defaultdict(list)
        # for f_id in raw_f_in:
        #     if f_id < raw_f_len:
        #         raw_f_id = new_f_id2raw_f_id[f_id]
        #         p_id = self.patch_mesh.f2patch[raw_f_id]
        #         p_max_id = max(p_max_id, p_id)
        #         patch2f_in[p_id].append(f_id)
        # patch2f_in = [patch2f_in[p_id] for p_id in patch2f_in]

        # for i in range(len(patch2f_in)):
        #     for j in range(len(patch2f_in[i])):
        #         if patch2f_in[i][j] in raw_f_in:
        #             patch2f_in[i][j] = raw_f_in.index(patch2f_in[i][j])

        # patch2f_out = defaultdict(list)
        # for f_id in raw_f_out:
        #     if f_id < raw_f_len:
        #         raw_f_id = new_f_id2raw_f_id[f_id]
        #         p_id = self.patch_mesh.f2patch[raw_f_id]
        # patch2f_out = [patch2f_out[p_id] for p_id in patch2f_out]

        # for i in range(len(patch2f_out)):
        #     for j in range(len(patch2f_out[i])):
        #         if patch2f_out[i][j] in raw_f_out:
        #             patch2f_out[i][j] = raw_f_out.index(patch2f_out[i][j])

        
        
        v1, f1 = get_submesh(new_v, new_f, f_in)
        v2, f2 = get_submesh(new_v, new_f, f_out)
        
        plot = mp.plot(new_v, new_f[f_in], np.ones((f1.shape[0], 3)), shading={'wireframe': True})
        plot.add_points(
            self.patch_mesh.v[border_v],
            shading={
                'point_size': 3,
                'point_color': 'red'
            }
        )
        plot.add_points(
            new_v[id_v_border_proj],
            shading={
                'point_size': 3,
                'point_color': 'blue'
            }
        )

        # show_mesh(v1, f1, wireframe=True, show_normal=True)
        # show_mesh(v2, f2, wireframe=True, show_normal=True)

        mp.plot(v1, f1, np.ones((f1.shape[0], 3)), shading={'wireframe': True})
        mp.plot(v2, f2, np.ones((f2.shape[0], 3)), shading={'wireframe': True})

        patch2f_in = self.pre_split(v1, f1)
        patch2f_out = self.pre_split(v2, f2)

        return True, [
            {"v": v1, "f": f1, "patch": patch2f_in}, 
            {"v": v2, "f": f2, "patch": patch2f_out}
        ]


    ''' 
        Pre Split by concavity

    '''
    def pre_split(self, v, f):
        mesh = om.TriMesh(v, f)
    
        f_center = np.mean(v[f], axis=1)
        f_normal = igl.per_face_normals(v, f, np.array([1., 0., 0.]))
        f_normal = f_normal / np.linalg.norm(f_normal, axis=1).reshape(-1, 1)
        threshold_convexity = np.cos(87.5 * np.pi / 180)
        

        convex_edge, concave_edge = get_border_edge(mesh, f_center, f_normal, threshold_convexity, get_boundary=False)
        
        edge = []
        if len(convex_edge) > 0:
            edge.extend(convex_edge[:, 0])
        if len(concave_edge) > 0:
            edge.extend(concave_edge[:, 0])
        
        edges = []
        for i in range(convex_edge.shape[0]):
            edges.append([convex_edge[i][1], convex_edge[i][2]])
        for i in range(concave_edge.shape[0]):
            edges.append([concave_edge[i][1], concave_edge[i][2]])
        edges = np.array(edges)

        # show cutting edge
        plot = mp.plot(v, f, np.ones((f.shape[0], 3)), shading={'wireframe': True})
        if len(edges) > 0:
            plot.add_lines(
                v[edges[:, 0]],
                v[edges[:, 1]],
                shading={
                    "line_color": "red",
                    "line_width": 1
                }
            )
            centroid = np.mean(v[f], axis=1)
            normal = igl.per_face_normals(v, f, np.array([1., 0., 0.]))
            normal = (normal.T / np.linalg.norm(normal, axis=1)).T
            plot.add_lines(
                centroid,
                centroid + normal,
                shading={
                    'line_color': 'blue',
                    'line_width': 2
                }

            )

        
        splited_f = split(mesh, edge)

        return splited_f




    '''
        INPUT:
        + nodes in patch mesh, each node has
            + v, f
            + type, params
        + edges in patch mesh, each edge has
            + borde_ type
    '''
    def cut_by_concave_curve(self):
        concave_curve_cut = ConcaveCurveCut(self.patch_mesh)
        return concave_curve_cut.aggrete_all_borders()
        
        # all concave border
        concave_edges = []
        for edge in self.patch_mesh.graph.edges:
            p1, p2 = edge
            border_type = self.patch_mesh.graph[p1][p2]["border_type"]
            if (border_type == 2) | (border_type == 3):
                continue
            concave_edges.append([p1, p2])
        if len(concave_edges) == 0:
            return False, []
        
        '''
            get all candidate neighbor patch's params
            [
                border: {
                    p0:{id, type, params},
                    p1:{id, type, params},
                    border_v: [...]
                }
            ]
        '''
        n_p_param = []
        for pair in concave_edges:
            p0, p1 = pair
            v_border = self.get_patch_border_v(p0, p1)
            p0_type = self.patch_mesh.graph.nodes[p0]["type"]
            p0_param = self.patch_mesh.graph.nodes[p0]["params"]
            p1_type = self.patch_mesh.graph.nodes[p1]["type"]
            p1_param = self.patch_mesh.graph.nodes[p1]["params"]
            for single_border in v_border:
                n_p_param.append({
                    'p0': {
                        'id': p0,
                        'type': p0_type,
                        'params': p0_param
                    },
                    'p1': {
                        'id': p1,
                        'type': p1_type,
                        'params': p1_param,
                    },
                    'border_v': single_border
                })
            # n_p_param.append({
            #     'p0': {
            #         'id': p0,
            #         'type': p0_type,
            #         'params': p0_param
            #     },
            #     'p1': {
            #         'id': p1,
            #         'type': p1_type,
            #         'params': p1_param,
            #     },
            #     'border_v': v_border
            # })
        
        # # case1: 2 borders share same patch 1 & 2
        # patches_of_border = []
        # for _border in n_p_param:
        #     patches_of_border.append([_border['p0']['id'], _border['p1']['id']].sort())
        # patches_of_border = np.array(patches_of_border)

        # same_patch_borders = None
        # find = False
        # for i in range(patches_of_border.shape[0]):
        #     if find:
        #         break
        #     for j in range(i+1, patches_of_border.shape[0]):
        #         if np.array_equal(patches_of_border[i], patches_of_border[j]):
        #             same_patch_borders = [i, j]
        #             find = True
        #             break
        # if same_patch_borders is not None:
        #     border1, border2 = n_p_param[same_patch_borders[0]], n_p_param[same_patch_borders[1]]

        #     # get direction
        #     def get_line_dir(v0, v1):
        #         dir = v1 - v0
        #         dir = dir / np.linalg.norm(dir)
        #         return dir

        #     border_dir = get_line_dir(border1['border_v'][0], border1['border_v'][-1])

        #     border2_v0, border2_v1 = border2['border_v'][0], border2['border_v'][-1]
        #     # test which larger on direction of border dir
        #     border2_dir = get_line_dir(border2_v0, border2_v1)
        #     border2_inserse = np.dot(border2_dir, border_dir) > 0

        #     # generate two cutting operations and cut


        
        # detect symmetric situation
        p_plane = defaultdict(list)
        p_cylinder = defaultdict(list)
        for i in range(len(n_p_param)):
            n_p = n_p_param[i]

            p0, p0_type, p0_param = n_p['p0']['id'], n_p['p0']['type'], n_p['p0']['params']
            p1, p1_type, p1_param = n_p['p1']['id'], n_p['p1']['type'], n_p['p1']['params']
            v_border = n_p['border_v']
            if p0_type == 'Plane':
                p_plane[p0].append(i)
            if p1_type == 'Plane':
                p_plane[p1].append(i)
            if p0_type == 'Cylinder':
                p_cylinder[p0].append(i)
            if p1_type == 'Cylinder':
                p_cylinder[p1].append(i)
        

        plane_cluster = []
        for p_id in p_plane:
            cur_plane_param = self.patch_mesh.graph.nodes[p_id]["params"]
            find_cluster = False
            for c_id in range(len(plane_cluster)):
                c_p_id = plane_cluster[c_id][0]
                c_plane_param = self.patch_mesh.graph.nodes[c_p_id]["params"]
                if is_plane_same(cur_plane_param, c_plane_param):
                    find_cluster = True                    
                    plane_cluster[c_id].append(p_id)
            if not find_cluster:
                plane_cluster.append([p_id])
        
        # globally same patches >= 3
        # get same plane
        common_patch = None
        for c in plane_cluster:
            if len(c) > 2:
                common_patch = c
                break
        if common_patch is not None:
            common_patch_border = {}
            # show
            f_colors = np.ones((self.patch_mesh.f.shape[0], 3))
            for p_id in common_patch:
                f_colors[self.patch_mesh.patch2f[p_id]] = [1, 0, 0]
            plot = mp.plot(self.patch_mesh.v, self.patch_mesh.f, f_colors, shading={"wireframe": False})

            for p_id in common_patch:
                n_p_list = p_plane[p_id]
                if len(n_p_list) == 1:
                    common_patch_border[p_id] = {
                        'v_border': n_p_param[n_p_list[0]]['border_v'],
                        'v_from': n_p_param[n_p_list[0]]['border_v'][0],
                        'v_to': n_p_param[n_p_list[0]]['border_v'][-1]
                    }
                else:
                    # connect border_v
                    border_v_pair = []
                    for n_p_id in n_p_list:
                        v_border = n_p_param[n_p_id]['border_v']
                        border_v_pair.append([v_border[0], v_border[-1]])
                    ordered_border_id = connect_v_pair_to_line(border_v_pair)

                    

                    if len(ordered_border_id) != 1:
                        raise Exception('[BORDER CONNECT]:cannot connect border of one patch to 1 line')
                    
                    ordered_border_id = ordered_border_id[0]

                    first_order = ordered_border_id[0][1]
                    ordered_border_v = [n_p_param[n_p_list[ordered_border_id[0][0]]]['border_v'][0] if first_order == 0 else n_p_param[n_p_list[ordered_border_id[0][0]]]['border_v'][-1]]
                    for i in range(len(ordered_border_id)):
                        cur_id, cur_dir = ordered_border_id[i]
                        if cur_dir == 0:
                            ordered_border_v.extend(n_p_param[n_p_list[cur_id]]['border_v'][1:])
                        else:
                            ordered_border_v.extend(n_p_param[n_p_list[cur_id]]['border_v'][::-1][1:])
                    v_from, v_to = ordered_border_v[0], ordered_border_v[-1]
                    common_patch_border[p_id] = {
                        'v_border': ordered_border_v,
                        'v_from': v_from,
                        'v_to': v_to
                    }
            # print(common_patch_border)
            plot = mp.plot(self.patch_mesh.v, self.patch_mesh.f, np.ones((self.patch_mesh.f.shape[0], 3)), shading={"wireframe": False})
            for p_id in common_patch_border:
                plot.add_lines(
                    self.patch_mesh.v[common_patch_border[p_id]['v_border'][:-1]],
                    self.patch_mesh.v[common_patch_border[p_id]['v_border'][1:]],
                    shading={
                        'line_width': 2,
                        'line_color': 'blue'
                    }
                )
            
            # get order
            # project to 2d:
            plane_param = self.patch_mesh.graph.nodes[common_patch[0]]['params']
            plane_normal = plane_param[:3]
            origin_3d = get_origin_from_plane(plane_param)
            u, v = get_uv_from_normal(plane_normal)

            c_2d = []
            for p_id in common_patch:
                patch_v_id = np.unique(self.patch_mesh.f[self.patch_mesh.patch2f[p_id]].flatten())
                c = np.mean(self.patch_mesh.v[patch_v_id], axis=0)
                vec = c - origin_3d
                u_coord = np.dot(vec, u)
                v_coord = np.dot(vec, v)
                c_2d.append([u_coord, v_coord])
            c_2d = np.array(c_2d)

            # plot c_2d on plt
            figure = plt.figure()
            ax = figure.add_subplot(111)
            ax.scatter(c_2d[:, 0], c_2d[:, 1], c='red')
            plt.show()

            # fit to circle
            center, radius = fit_circle(c_2d)
            
            # clockwise order
            vectors = c_2d - center
            angles = np.arctan2(vectors[:,1], vectors[:,0])

            angles = np.mod(angles, 2 * np.pi)
            order = np.argsort(-angles)

            # change order
            common_patch = [common_patch[i] for i in order]
            c_2d = c_2d[order]

            # judge order of v_border
            for i in range(len(common_patch)):
                p_id = common_patch[i]
                v_from, v_to = common_patch_border[p_id]['v_from'], common_patch_border[p_id]['v_to']
                v_from_pos, v_to_pos = self.patch_mesh.v[v_from], self.patch_mesh.v[v_to]
                v_from_2d, v_to_2d = project_point_to_2d(v_from_pos, origin_3d, u, v), project_point_to_2d(v_to_pos, origin_3d, u, v)

                cross_from = np.cross(c_2d[i] - center, v_from_2d - c_2d[i])
                cross_to = np.cross(c_2d[i] - center, v_to_2d - c_2d[i])
                if cross_from > 0 and cross_to < 0:
                    continue
                elif cross_from < 0 and cross_to > 0:
                    common_patch_border[p_id]['v_border'] = common_patch_border[p_id]['v_border'][::-1]
                    common_patch_border[p_id]['v_from'], common_patch_border[p_id]['v_to'] = common_patch_border[p_id]['v_to'], common_patch_border[p_id]['v_from']
                else:
                    raise Exception('[BORDER ORDER]: cannot judge border order by cross product')
                
                # show
            plot = mp.plot(self.patch_mesh.v, self.patch_mesh.f, np.ones((self.patch_mesh.f.shape[0], 3)), shading={"wireframe": False})
            for p_id in common_patch:
                print(p_id)
                v_from, v_to = common_patch_border[p_id]['v_from'], common_patch_border[p_id]['v_to']
                plot.add_points(
                    np.array([self.patch_mesh.v[v_from]]),
                    shading={
                        'point_size': 1,
                        'point_color': 'red'
                    }
                )
                plot.add_points(
                    np.array([self.patch_mesh.v[v_to]]),
                    shading={
                        'point_size': 1,
                        'point_color': 'blue'
                    }
                )

            # cut for rotation symmetric
            # from common_patch_border[p_id]['v_to'] to common_patch_border[next_p_id]['v_from']
            return self.cut_mesh_by_rotation_symmetric_plane_tracing(plane_param, common_patch, common_patch_border)

        cylinder_cluster = []
        for p_id in p_cylinder:
            cur_cylinder_param = self.patch_mesh.graph.nodes[p_id]["params"]
            find_cluster = False
            for c_id in range(len(cylinder_cluster)):
                c_p_id = cylinder_cluster[c_id][0]
                c_cylinder_param = self.patch_mesh.graph.nodes[c_p_id]["params"]
                if is_cylinder_same(cur_cylinder_param, c_cylinder_param):
                    find_cluster = True
                    cylinder_cluster[c_id].append(p_id)
            if not find_cluster:
                cylinder_cluster.append([p_id])
        for c in cylinder_cluster:
            if len(c) > 2:
                common_patch = c
                break
        if common_patch is not None:
            common_patch_border = {}
            # show
            f_colors = np.ones((self.patch_mesh.f.shape[0], 3))
            # for p_id in common_patch:
            #     f_colors[self.patch_mesh.patch2f[p_id]] = [1, 0, 0]
            plot = mp.plot(self.patch_mesh.v, self.patch_mesh.f, f_colors, shading={"wireframe": False})

            for p_id in common_patch:
                n_p_list = p_cylinder[p_id]
                if len(n_p_list) == 2:
                    common_patch_border[p_id] = [
                    {
                        'v_border': n_p_param[n_p_list[0]]['border_v'],
                        'v_from': n_p_param[n_p_list[0]]['border_v'][0],
                        'v_to': n_p_param[n_p_list[0]]['border_v'][-1]
                    },
                    {
                        'v_border': n_p_param[n_p_list[1]]['border_v'],
                        'v_from': n_p_param[n_p_list[1]]['border_v'][0],
                        'v_to': n_p_param[n_p_list[1]]['border_v'][-1]
                    }]
                
            cylinder_param = self.patch_mesh.graph.nodes[common_patch[0]]['params']
            cylinder_c, cylinder_axis, cylinder_r = cylinder_param[0:3].squeeze(), cylinder_param[3:6].squeeze(), cylinder_param[6].item()
            
            # concate cylinder_axis and 0 to plane param
            plane_param = np.zeros((4,))
            plane_param[:3] = cylinder_axis
            origin_3d = get_origin_from_plane(plane_param)
            u, v = get_uv_from_normal(cylinder_axis)

            c_2d = []
            for p_id in common_patch:
                patch_v_id = np.unique(self.patch_mesh.f[self.patch_mesh.patch2f[p_id]].flatten())
                c = np.mean(self.patch_mesh.v[patch_v_id], axis=0)
                vec = c - origin_3d
                u_coord = np.dot(vec, u)
                v_coord = np.dot(vec, v)
                c_2d.append([u_coord, v_coord])
            c_2d = np.array(c_2d)

            figure = plt.figure()
            ax = figure.add_subplot(111)
            ax.scatter(c_2d[:, 0], c_2d[:, 1])
            plt.show()

            center, radius = fit_circle(c_2d)

            vectors = c_2d - center
            angles = np.arctan2(vectors[:,1], vectors[:,0])
            angles = np.mod(angles, 2 * np.pi)
            order = np.argsort(-angles)

            common_patch = [common_patch[i] for i in order]

            # reverse v_border
            base_v_pos = self.patch_mesh.v[common_patch_border[common_patch[0]][0]['v_from']]
            d = - np.dot(base_v_pos, cylinder_axis)
            base_plane_param = np.zeros((4,))
            base_plane_param[:3] = cylinder_axis
            base_plane_param[3] = d

            for i in range(len(common_patch)):
                for j in range(2):
                    p_id = common_patch[i]
                    tmp_v_from = common_patch_border[p_id][j]['v_from']
                    if abs(np.dot(self.patch_mesh.v[tmp_v_from], cylinder_axis) + d) > 1e-1:
                        common_patch_border[p_id][j]['v_border'] = common_patch_border[p_id][j]['v_border'][::-1]
                        common_patch_border[p_id][j]['v_from'] = common_patch_border[p_id][j]['v_border'][0]
                        common_patch_border[p_id][j]['v_to'] = common_patch_border[p_id][j]['v_border'][-1]
            

            return self.cut_mesh_by_rotation_symmetric_cylinder_tracing(cylinder_param, common_patch, common_patch_border)
        
        
        print('start_cutting')
        # connect border to curve
        border_v_pair = defaultdict(list)
        for i in range(len(n_p_param)):
            border = n_p_param[i]
            
            border_v_pair[i] = [border['border_v'][0], border['border_v'][-1]]
        # border_set_visited = set()
        border_set_unvisited = set(range(len(n_p_param)))
        border_ordered = []
        while len(border_set_unvisited) > 0:
            cur_border_id = border_set_unvisited.pop()
            # border_set_visited.add(cur_border_id)

            cur_to_v = border_v_pair[cur_border_id][1]
            cur_border_list = [[cur_border_id, 0]]
            find_next_border = True
            while find_next_border:
                find_next_border = False
                # find next of cur_to_v
                for next_border_id in border_set_unvisited:
                    next_from_v, next_to_v = border_v_pair[next_border_id]
                    if cur_to_v == next_from_v:
                        cur_border_list.append([next_border_id, 0])
                        cur_to_v = next_to_v
                        find_next_border = True
                        border_set_unvisited.remove(next_border_id)
                        break
                    elif cur_to_v == next_to_v:
                        cur_border_list.append([next_border_id, 1])
                        cur_to_v = next_from_v
                        find_next_border = True
                        border_set_unvisited.remove(next_border_id)
                        break
                if find_next_border:
                    continue
                # reverse cur_border_list, change 0 / 1 to 1/0
                new_cur_border_list = []
                for i in range(len(cur_border_list)-1, -1, -1):
                    b_id, dir = cur_border_list[i]
                    new_cur_border_list.append((b_id, 1 - dir))
                cur_border_list = new_cur_border_list
                # find next of cur_from_v
                dir = cur_border_list[-1][1]
                cur_to_v = border_v_pair[cur_border_list[-1][0]][1 - dir]
                for next_border_id in border_set_unvisited:
                    next_from_v, next_to_v = border_v_pair[next_border_id]
                    if cur_to_v == next_from_v:
                        cur_border_list.append([next_border_id, 0])
                        cur_to_v = next_to_v
                        find_next_border = True
                        border_set_unvisited.remove(next_border_id)
                        break
                    elif cur_to_v == next_to_v:
                        cur_border_list.append([next_border_id, 1])
                        cur_to_v = next_from_v
                        find_next_border = True
                        border_set_unvisited.remove(next_border_id)
                        break
            border_ordered.append(cur_border_list)
        

        # select border
        for i in range(len(border_ordered)):
            # mp.offline()
            print(f"border {i}:")

            # save_ply_with_edges(
            #     f'border{i}.ply',
            #     self.patch_mesh.v,
            #     self.patch_mesh.f,
            #     border_ordered[i]
            # )
            
            cur_border_ordered = border_ordered[i]
            b_id_from, dir_from = cur_border_ordered[0]
            b_id_to, dir_to = cur_border_ordered[-1]
            border_v = [n_p_param[b_id_from]['border_v'][0] if dir_from == 0 else n_p_param[b_id_from]['border_v'][-1]]
            for i in range(len(cur_border_ordered)):
                b_id, dir = cur_border_ordered[i]
                if dir == 0:
                    border_v.extend(n_p_param[b_id]['border_v'][1:])
                else:
                    border_v.extend(n_p_param[b_id]['border_v'][::-1][1:])
            
            plot = mp.plot(self.patch_mesh.v, self.patch_mesh.f, np.ones((self.patch_mesh.f.shape[0], 3)), shading={"wireframe": False})
            plot.add_lines(
                self.patch_mesh.v[border_v[:-1]],
                self.patch_mesh.v[border_v[1:]],
                shading={
                    'line_width': 2,
                    'line_color': 'red'
                }
            )
        border_id = int(input(f"Input border id:"))
        border_ordered = border_ordered[border_id]

        # form vertex border
        b_id_from, dir_from = border_ordered[0]
        b_id_to, dir_to = border_ordered[-1]
        border_v = [n_p_param[b_id_from]['border_v'][0] if dir_from == 0 else n_p_param[b_id_from]['border_v'][-1]]
        for i in range(len(border_ordered)):
            b_id, dir = border_ordered[i]
            if dir == 0:
                border_v.extend(n_p_param[b_id]['border_v'][1:])
            else:
                border_v.extend(n_p_param[b_id]['border_v'][::-1][1:])
        v_from, v_to = border_v[0], border_v[-1]


        # check: if close -> cut by projection
        if v_from == v_to:
            return self.cut_mesh_by_close_cylinder(n_p_param[b_id_from])


        # else: not close -> cut by surface-tracing
        # 1. select patch/param of border
        if len(border_ordered) > 1:
            border_from_p0, border_from_p0_type, border_from_p0_params = n_p_param[b_id_from]['p0']['id'], n_p_param[b_id_from]['p0']['type'], n_p_param[b_id_from]['p0']['params']
            border_from_p1, border_from_p1_type, border_from_p1_params = n_p_param[b_id_from]['p1']['id'], n_p_param[b_id_from]['p1']['type'], n_p_param[b_id_from]['p1']['params']
            border_to_p0, border_to_p0_type, border_to_p0_params = n_p_param[b_id_to]['p0']['id'], n_p_param[b_id_to]['p0']['type'], n_p_param[b_id_to]['p0']['params']
            border_to_p1, border_to_p1_type, border_to_p1_params = n_p_param[b_id_to]['p1']['id'], n_p_param[b_id_to]['p1']['type'], n_p_param[b_id_to]['p1']['params']
            
            candidate_border_p = [border_from_p0, border_from_p1, border_to_p0, border_to_p1]
            candidate_border_type = [border_from_p0_type, border_from_p1_type, border_to_p0_type, border_to_p1_type]

            for i in range(4):
                print(f'[{i+1}]: {candidate_border_type[i]} patch {candidate_border_p[i]}')

                f_colors = np.ones((self.patch_mesh.f.shape[0], 3))
                f_colors[self.patch_mesh.patch2f[candidate_border_p[i]]] = [0., 0., 1.]
                plot = mp.plot(self.patch_mesh.v, self.patch_mesh.f, f_colors, shading={"wireframe": False})
                plot.add_lines(
                    self.patch_mesh.v[border_v[:-1]],
                    self.patch_mesh.v[border_v[1:]],
                    shading={
                        'line_width': 2,
                        'line_color': 'red'
                    }
                )
        
            select_param = input(f'Input which cut param (1: {border_from_p0}, 2: {border_from_p1}, 3: {border_to_p0}, 4: {border_to_p1}):')
            select_type = None
    
            if select_param == '1':
                cut_param = border_from_p0_params
                select_type = border_from_p0_type
            elif select_param == '2':
                cut_param = border_from_p1_params
                select_type = border_from_p1_type
            elif select_param == '3':
                cut_param = border_to_p0_params
                select_type = border_to_p0_type
            elif select_param == '4':
                cut_param = border_to_p1_params
                select_type = border_to_p1_type
            else:
                print('Invalid cut param')
                return False, []
        else:
            b_id, _ = border_ordered[0]
            border_p0, border_p0_type, border_p0_params = n_p_param[b_id]['p0']['id'], n_p_param[b_id]['p0']['type'], n_p_param[b_id]['p0']['params']
            border_p1, border_p1_type, border_p1_params = n_p_param[b_id]['p1']['id'], n_p_param[b_id]['p1']['type'], n_p_param[b_id]['p1']['params']
            
            candidate_border_p = [border_p0, border_p1]
            candidate_border_type = [border_p0_type, border_p1_type]

            for i in range(2):
                print(f'[{i+1}]: {candidate_border_type[i]} patch {candidate_border_p[i]}')

                f_colors = np.ones((self.patch_mesh.f.shape[0], 3))
                f_colors[self.patch_mesh.patch2f[candidate_border_p[i]]] = [0., 0., 1.]
                plot = mp.plot(self.patch_mesh.v, self.patch_mesh.f, f_colors, shading={"wireframe": False})
                plot.add_lines(
                    self.patch_mesh.v[border_v[:-1]],
                    self.patch_mesh.v[border_v[1:]],
                    shading={
                        'line_width': 2,
                        'line_color': 'red'
                    }
                )

        
            select_param = input(f'Input which cut param (1: {border_p0}, 2: {border_p1}')
            select_type = None
    
            if select_param == '1':
                cut_param = border_p0_params
                select_type = border_p0_type
            elif select_param == '2':
                cut_param = border_p1_params
                select_type = border_p1_type
            else:
                print('Invalid cut param')
                return False, []


        # 2.cut through param by different types 
        if select_type == 'Plane':
            print(f"cut by plane: {cut_param}")
            time_start = time.time()
            cut_bool, cut_result = self.cut_mesh_by_plane_tracing(cut_param, v_from, v_to, border_v)
            # planar_cut = PlaneTracingCut(self.patch_mesh)
            # objs = planar_cut.cut_from_to([(v_from, v_to, border_v)], cut_param, viz=True)
            time_end = time.time()
            print(f"cut by plane time: {time_end - time_start:.4f} seconds")
            # return True, objs
            return True, cut_result
        elif select_type == 'Cylinder':
            print(f"cut by cylinder: {cut_param}")
            time_start = time.time()
            cut_bool, cut_result = self.cut_mesh_by_cylinder_tracing(cut_param, v_from, v_to, border_v)
            # cylinder_cut = CylinderTracingCut(self.patch_mesh)
            # objs = cylinder_cut.cut_from_to([(v_from, v_to, border_v)], cut_param, viz=True)
            time_end = time.time()
            print(f"cut by cylinder time: {time_end - time_start:.4f} seconds")
            return cut_bool, cut_result
            # return True, objs
        else:
            print('Invalid cut type')
            return False, []


    def _make_plane_quad(self, plane_abcd, center, diag_len, scale=1.2):
        """
        生成位于平面上的一个矩形（四边形），用于可视化。
        - plane_abcd: (a,b,c,d) 表示 ax+by+cz+d=0
        - center: 平面上的一个中心点
        - diag_len: 根据 mesh 尺度选择的边长标度
        - scale: 放大系数
        """
        a, b, c, _ = plane_abcd
        n = np.array([a, b, c], dtype=float)
        n_norm = np.linalg.norm(n)
        if n_norm < 1e-12:
            raise ValueError("平面法向量长度为0，请检查参数 (a,b,c,d)")
        n = n / n_norm
        def _orthonormal_basis_from_normal(n):
            """给定单位法向量 n，构造与其正交的二维基 (u, v)，均归一化。"""
            n = n / (np.linalg.norm(n) + 1e-12)
            # 选择与 n 不平行的参考向量
            ref = np.array([1.0, 0.0, 0.0]) if abs(n[0]) < 0.9 else np.array([0.0, 1.0, 0.0])
            u = np.cross(n, ref)
            u /= (np.linalg.norm(u) + 1e-12)
            v = np.cross(n, u)
            v /= (np.linalg.norm(v) + 1e-12)
            return u, v
        u, v = _orthonormal_basis_from_normal(n)
        half = 0.5 * diag_len * scale
        corners = [
            center + (+half)*u + (+half)*v,
            center + (-half)*u + (+half)*v,
            center + (-half)*u + (-half)*v,
            center + (+half)*u + (-half)*v,
        ]
        return np.array(corners)


    
    def plot_mesh_cut_with_plane(self, 
        v, f, f_in, f_out, v_cut_idx, plane_abcd,
        mesh_color='#CCCCCC', mesh_alpha=1.0,
        plane_alpha=0.18, cut_color='crimson', line_width=3.0,
        view=(45, 30), zoom=1.4
    ):
        v = np.asarray(v, float)
        f = np.asarray(f, int)
        a, b, c, d = plane_abcd
        n = np.array([a, b, c], float)
        n /= (np.linalg.norm(n) + 1e-12)

        # --- plane quad (for visualization) ---
        center = v.mean(0)
        center_on_plane = center - (np.dot(center, n) + d) * n
        ref = np.array([1,0,0]) if abs(n[0]) < 0.9 else np.array([0,1,0])
        u = np.cross(n, ref); u /= (np.linalg.norm(u) + 1e-12)
        vdir = np.cross(n, u)
        diag = np.linalg.norm(v.max(0) - v.min(0))
        s = diag * 0.6
        quad = np.array([
            center_on_plane + s*u + s*vdir,
            center_on_plane - s*u + s*vdir,
            center_on_plane - s*u - s*vdir,
            center_on_plane + s*u - s*vdir
        ])

        # --- classify faces relative to plane ---
        # sd = v @ n + d                          # signed distance per vertex
        # tri_sd = sd[f]                           # (M,3)
        # eps = 1e-9
        # all_pos = (tri_sd >  eps).all(axis=1)
        # all_neg = (tri_sd < -eps).all(axis=1)
        # cross  = ~(all_pos | all_neg)           # faces intersecting/straddling plane

        # Back (far) and Front (near) sets:
        # 为了稳定可视化：把跨越平面的三角形同时放入两侧，避免出现裂缝
        # back_faces  = f[all_neg | cross]
        # front_faces = f[all_pos | cross]
        back_faces = f[f_in]
        front_faces = f[f_out]

        # --- figure ---
        fig = plt.figure(figsize=(10, 9))
        ax = fig.add_subplot(111, projection='3d')

        # helper to add a surface (no wireframe)
        def add_surface(faces, color, alpha=1.0):
            if len(faces) == 0: return
            coll = Poly3DCollection([v[fi] for fi in faces],
                                    facecolor=color, edgecolor='none', alpha=alpha)
            ax.add_collection3d(coll)

        # --- draw order: back mesh -> plane -> front mesh ---
        # add_surface(back_faces, mesh_color, mesh_alpha)

        # plane_poly = Poly3DCollection([quad], facecolor='gold', edgecolor='none', alpha=plane_alpha)
        # 细分成两个三角有时能稍微改善透明排序
        tri1 = [quad[0], quad[1], quad[2]]; tri2 = [quad[0], quad[2], quad[3]]
        plane_poly = Poly3DCollection([tri1, tri2], facecolor='gold', edgecolor='none', alpha=plane_alpha)
        ax.add_collection3d(plane_poly)

        add_surface(front_faces, mesh_color, mesh_alpha)

        # --- cutting polyline ---
        if v_cut_idx is not None and len(v_cut_idx) >= 2:
            cut = v[np.asarray(v_cut_idx, int)]
            segs = [[cut[i], cut[i+1]] for i in range(len(cut)-1)]
            ax.add_collection3d(Line3DCollection(segs, colors=cut_color, linewidths=line_width))

        # --- framing / zoom / view ---
        allp = np.vstack([v, quad])
        mins, maxs = allp.min(0), allp.max(0)
        mid, r = (maxs + mins)/2, (maxs - mins).max()/2
        r /= zoom
        ax.set_xlim(mid[0]-r, mid[0]+r)
        ax.set_ylim(mid[1]-r, mid[1]+r)
        ax.set_zlim(mid[2]-r, mid[2]+r)
        ax.set_box_aspect([1,1,1])
        ax.view_init(elev=view[1], azim=view[0])

        # clean figure
        ax.axis('off'); ax.grid(False)
        ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
        ax.set_facecolor('white')
        plt.subplots_adjust(left=0, right=1, top=1, bottom=0)
        plt.show()
        

In [12]:
def mesh_segment(v, f, iteration_time, patch2f=None, init_patch_file=None, init_region_file=None):
    print("iteration time: ", iteration_time)
    Objects = []
    
    # hole filling
    mp.plot(v, f, np.ones((f.shape[0], 3)), shading={"wireframe": False})
    do_hole_filling = input("Do hole filling? (y/n):")
    if do_hole_filling == 'y':
        time_start = time.time()
        hole_filling = HoleFilling(v, f, visualize=True)
        new_v, new_f, filled_regions = hole_filling.new_v, hole_filling.new_f, hole_filling.region_triangles
        if len(filled_regions) != 0:
            cur_id = f.shape[0]
            for region in filled_regions:
                patch2f.append(list(np.arange(cur_id, cur_id + region.shape[0])))
                cur_id += len(region)
        v, f = new_v, new_f
        time_end = time.time()
        print(f"[TIME-HOLEFILLING] {time_end - time_start:.4f} seconds")
        mp.plot(v, f, np.ones((f.shape[0], 3)), shading={"wireframe": False})
    
    
    # generate patch or add patch
    time_start = time.time()
    if patch2f is None:
        patch_mesh = InitPatchMesh(v, f, init_patch_file, init_region_file)
    else:
        patch_mesh = PatchMesh(v, f, patch2f)
    time_end = time.time()
    print(f'[TIME-PATCH]: {time_end - time_start:.4f} seconds')
    
    # get patch type
    time_start = time.time()
    border_type = BorderType(patch_mesh)
    time_end1 = time.time()
    # border_type.show_each_patch()
    border_type.show_patch()
    border_type.get_patch_type()
    time_end2 = time.time()
    border_type.get_all_borders()
    time_end = time.time()
    print(f"[TIME-BORDER]: {time_end - time_start:.4f} seconds \n \
        init border_type use {time_end1 - time_start:.4f} seconds, \n \
        get patch type use {time_end2 - time_end1:.4f} seconds, \n \
        get all borders use {time_end - time_end2:.4f} seconds")


    # combine patches
    time_start = time.time()
    border_type.combine_patches_by_fitting(visualize=True)
    time_end1 = time.time()
    border_type.get_all_borders()
    time_end = time.time()
    border_type.show_patch()
    patch2f = border_type.patch_mesh.patch2f
    if iteration_time == 0:
        if patch2f is None:
            patch2f = []
        return [{'v':v, 'f':f, 'patch': patch2f}]
    time_end = time.time()
    print(f"[TIME-COMBINE]: {time_end - time_start:.4f} seconds \n\
        combine patches use {time_end1 - time_start:.4f} seconds, \n\
        get all borders use {time_end - time_end1:.4f} seconds")

    # border_type.show_each_patch()
    # border_type.get_all_borders()

    
    time_start = time.time()
    concave_circle_cut, concave_circle_cut_list = border_type.cut_by_concave_border()
    time_end = time.time()
    print(f"[TIME]:try concave circle time: {time_end - time_start:.4f} seconds")    
    if concave_circle_cut:
        print("find concave circle cut")
        concave_circle_cut_list.reverse()
        for sub_obj in concave_circle_cut_list:
            result = mesh_segment(sub_obj['v'], sub_obj['f'], iteration_time-1, sub_obj['patch'])
            Objects.extend(result)
    else:
        # objs = border_type.cut_by_convex_concave_border()
        # for sub_obj in objs:
        #     result = mesh_segment(sub_obj['v'], sub_obj['f'], iteration_time-1, sub_obj['patch'])
        #     Objects.extend(result)
        # return objs


        concave_curve_cut, concave_curve_cut_list = border_type.cut_by_concave_curve()
        time_end = time.time()
        print(f"[TIME]:try concave curve time: {time_end - time_start:.4f} seconds")
        if concave_curve_cut:
            print("find concave curve cut")
            for sub_obj in concave_curve_cut_list:
                result = mesh_segment(sub_obj['v'], sub_obj['f'], iteration_time-1, sub_obj['patch'])
                Objects.extend(result)
        else:
            print("no concave curve")
            # consider convex cut in object
            return [{'v':v, 'f':f, 'patch': patch2f}]
    return Objects




In [13]:
# obj_name = "2"
# obj_path = f"example_data/example_CAD/{obj_name}.obj"
# # patch_path = f"example_results/vsa/{obj_name}_proxy_cluster.txt"
# patch_path = f"example_results/vsa/{obj_name}_merged_proxy_cluster.txt"
# # patch_path = f"example_results/vsa/{obj_name}_pre_proxy_cluster.txt"
# region_path = f"example_results/vsa/{obj_name}_region_cluster.txt"
# output_ply = f"example_results/merge/selected/{obj_name}_proxy_new.ply"

# obj_name = "cad16"
# obj_path = f"example_data/example_CAD/{obj_name}.obj"
# patch_path = f"example_results/vsa/cad/{obj_name}_proxy_cluster.txt"
# region_path = f"example_results/vsa/cad/{obj_name}_region_cluster.txt"
# output_ply = f"example_results/merge/cad/{obj_name}.ply"

# obj_name = "00140179_7d7f0dc4c3b8202c03a32241_trimesh_000"
# obj_name = "00140180_7b0779cf3270ccee61e01b3e_trimesh_000"
# obj_name = "00140223_5374ce893909eddbc7536b71_trimesh_000"
# obj_name = "00140302_dce0fb9e808b32ded73025c1_trimesh_000"
# obj_name = '00140316_62dfc752c2876cd94647532a_trimesh_011'
# obj_name = "00140400_e4a5e7862a9beda824b2e00c_trimesh_003"
# obj_name = "00140553_e2c0841b6c86e3bfdcc8c477_trimesh_000"
# obj_name = "00140682_572115b8e4b01ff83d93b7c0_trimesh_000"
# obj_name = '00140708_1c9a6a91c5121e671690124c_trimesh_000'
# obj_name = '00140710_33073a6cb55fadd20ec0e3fb_trimesh_001'
# obj_name = '00140722_0752ced969c82bbb19ce666b_trimesh_000'
# obj_name = "00140750_0da4d13f288d4cd70deecf20_trimesh_001"
# obj_name = "00140756_eaf6606df2a105f356a99677_trimesh_000"
# obj_name = "00140780_d56b3836d8a40e3035d42380_trimesh_000"
# obj_name = "00140919_3023c3cfd7d6b1a80700e17e_trimesh_000"
# obj_name = "00140923_0e818d7ff8780682567c824e_trimesh_000"
# obj_name = "00140936_9a2b76f7e489ea90f5e41ef8_trimesh_007"
# obj_name = "00140967_631426bb5199fa39fc8eeb1c_trimesh_008"
# obj_name = "00140982_f2e481e34cff3a450799acac_trimesh_000"
# obj_name = "00141000_e5a1548042d09e9e460ec51e_trimesh_000"
# obj_name = "00141019_e19ef37e52d5182509d42ac9_trimesh_000"
# obj_name = "00141097_57222034e4b08229a097b208_trimesh_010"
# obj_name = "00141172_5722364be4b0e6559901e691_trimesh_000"
# obj_name = "00141176_5ff893b2e5c70dac030147d6_trimesh_001"
# obj_path = f'example_data/selected/{obj_name}.obj'
# # patch_path = f'example_results/merge/selected/{obj_name}_proxy_cluster.txt'
# patch_path = f'example_results/vsa/selected/{obj_name}_proxy_cluster.txt'
# # patch_path = f'example_results/vsa/selected/{obj_name}_merged_proxy_cluster.txt'
# region_path = f'example_results/vsa/selected/{obj_name}_region_cluster.txt'
# output_ply = f'example_results/merge/selected/{obj_name}_seg.ply'


# obj_name = "00023435_385b221a0d58490b84f3edee_trimesh_000"
# obj_name = "00023471_7f45ff9e8c754def8bb4b1cb_trimesh_000"
# obj_name = "00023576_35c7024f831f44048104c331_trimesh_000"
# obj_name = "00023582_9c917172a61b472fb0e6ae3c_trimesh_002"
# obj_name = "00023792_30c31f050d2c40139e9b36ca_trimesh_001"
# obj_name = "00024036_de448e53313b4faa9dff2245_trimesh_003"
# obj_name = "00024040_de448e53313b4faa9dff2245_trimesh_007"
# obj_name = "00024044_de448e53313b4faa9dff2245_trimesh_011"
# obj_name = "00024082_de448e53313b4faa9dff2245_trimesh_049"
# obj_name = "00024089_de448e53313b4faa9dff2245_trimesh_056"
# obj_name = "00024132_a087bf84d26046349bbbd6f6_trimesh_000"
# obj_name = "00024448_274a7c9d6def4a699961a747_trimesh_002"
# obj_name = "00024792_34a17822747a4b20a8c2954b_trimesh_009"
# obj_name = "00024961_e26f4b039c6c48c9878b21fe_trimesh_000"
# obj_name = "00025063_9347be1a9dad4e70852dcce2_trimesh_000"
# obj_name = "00025183_021c990acfc648618a49bd47_trimesh_000"
# obj_name = "00025611_37f3a717a9b842cfbe5f6005_trimesh_002"
# obj_name = "00025633_2aaf78a0f41a47b1bae8c447_trimesh_003"
# obj_name = "00025710_6fe81cf35e2740b3bbde9aa6_trimesh_020"
# obj_name = "00025728_6fe81cf35e2740b3bbde9aa6_trimesh_038"
# obj_name = "00025749_c01a880ceccb4e05908f8255_trimesh_000"
obj_name = "00026055_11394f3abb83419b88f6eed9_trimesh_000"
# obj_name = "00026133_8010d7708c06499e9e99ce1c_trimesh_008"

obj_path = f"example_data/abc_all/meshes/{obj_name}.off"
patch_path = f"example_results/vsa/abc_all/{obj_name}_proxy_cluster.txt"
region_path = ''
output_ply = f"example_results/merge/abc_all/{obj_name}_seg.ply"


# obj_name = "fillet_transition"
# obj_path = f"{obj_name}.obj"
# patch_path = f"example_results/vsa/{obj_name}_proxy_cluster.txt"
# region_path = f"example_results/vsa/{obj_name}_region_cluster.txt"
# output_ply = f"example_results/merge/test/{obj_name}_merged.ply"

# obj_name = "iter_1_part_0"
# obj_path = f"example_results/merge/cad/cad16/{obj_name}.ply"
# patch_path = f"example_results/vsa/cad/{obj_name}_proxy_cluster.txt"
# region_path = f"example_results/vsa/cad/{obj_name}_region_cluster.txt"
# output_ply = f"example_results/merge/cad/{obj_name}_merged.ply"

v, f = igl.read_triangle_mesh(obj_path)
iteration_time = 1
Objects = mesh_segment(v, f, iteration_time, init_patch_file=patch_path, init_region_file=region_path)


def save_raw_ply(obj_path, patch_path):
    v, f = igl.read_triangle_mesh(obj_path)
    patch2f = []
    patch_file = open(patch_path, 'r')
    lines = patch_file.readlines()
    for line in lines:
        f_list = line.split(' ')
        f_list = [int(f_id) for f_id in f_list if f_id != '' and f_id != '\n']
        patch2f.append(f_list)
    patch_file.close()
    f_colors = np.ones((f.shape[0], 3))
    for i in range(len(patch2f)):
        f_colors[patch2f[i]] = colors[i%500]
    mp.plot(v, f, f_colors, shading={"wireframe": False})
    write_ply_file(v, f, f_colors, output_ply.replace('.ply', '_raw.ply'))
# save_raw_ply(obj_path, patch_path)



iteration time:  1


Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.0, 0.0,…

[TIME-PATCH]: 0.1507 seconds


Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.0, 0.0,…

d:\anaconda\envs\py39\lib\site-packages\numpy\_core\fromnumeric.py:3596: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
d:\anaconda\envs\py39\lib\site-packages\numpy\_core\_methods.py:130: RuntimeWarning: invalid value encountered in divide
  ret = um.true_divide(


...

[TIME-BORDER]: 5.6678 seconds 
         init border_type use 0.0000 seconds, 
         get patch type use 0.0856 seconds, 
         get all borders use 5.5822 seconds
before merge:


Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.0, 0.0,…

after merge:


Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.0, 0.0,…

Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.0, 0.0,…

In [ ]:
seg_f_id = [2372, 2373, 2374, 2375, 2376, 2377, 2378, 2379, 2380, 2381, 2382, 2383, 2384, 2385, 2386, 2387, 2388, 2389, 2390, 2391, 2392, 2393, 2394, 2395, 2396, 2397, 2398, 2399, 2400, 2401, 2402, 2403, 2404, 2405, 2406, 2407, 2408, 2409, 2410, 2411, 2412, 2413, 2414, 2415, 2416, 2417, 2418, 2419, 2420, 2421, 2422, 2423, 2424, 2425, 2426, 2427, 2428, 2429, 2430, 2431, 2432, 2433, 2434, 2435, 2436, 2437, 2438, 2439, 2440, 2441, 2442, 2443, 2444, 2445, 2446, 2447, 2448, 2449, 2450, 2451, 2452, 2453, 2454, 2455, 2456, 2457, 2458, 2459, 2460, 2461, 2462, 2463, 2464, 2465, 2466, 2467, 2468, 2469, 2470, 2471, 2472, 2473, 2474, 2475, 2476, 2477, 2478, 2479, 2480, 2481, 2482, 2483, 2484, 2485, 2486, 2487, 2488, 2489, 2490, 2491, 2492, 2493, 2494, 2495, 2496, 2497, 2498, 2499, 2500, 2501, 2502, 2503, 2504, 2505, 2506, 2507, 2508, 2509, 2510, 2511, 2512, 2513, 2514, 2515, 2516, 2517, 2518, 2519, 2520, 2521, 2522, 2523, 2524, 2525, 2526, 2527, 2528, 2529, 2530, 2531, 2532, 2533, 2534, 2535, 2536, 2537, 2538, 2539, 2540, 2541, 2542, 2543, 2544, 2545, 2546, 2547, 2548, 2549, 2550, 2551, 2552, 2553, 2554, 2555, 2556, 2557, 2558, 2559, 2560, 2561, 2562, 2563, 2564, 2565, 2566, 2567, 2568, 2569, 2570, 2571, 2572, 2573, 2574, 2575, 2576, 2577, 2578, 2579, 2580, 2581, 2582, 2583, 2584, 2585, 2586, 2587, 2588, 2589, 2590, 2591, 2592, 2593, 2594, 2595, 2596, 2597, 2598, 2599, 2600, 2601, 2602, 2603, 2604, 2605, 2606, 2607, 2608, 2609, 2610, 2611, 2612, 2613, 2614, 2615, 2616, 2617, 2618, 2619, 2620, 2621, 2622, 2623, 2624, 2625, 2626, 2627, 2628, 2629, 2630, 2631, 2632, 2633, 2634, 2635, 2636, 2637, 2638, 2639, 2640, 2641, 2642, 2643, 2644, 2645, 2646, 2647, 2648, 2649, 2650, 2651, 2652, 2653, 2654, 2655, 2656, 2657, 2658, 2659, 2660, 2661, 2662, 2663, 2664, 2665, 2666, 2667, 2668, 2669, 2670, 2671, 2672, 2673, 2674, 2675, 2676, 2677, 2678, 2679, 2680, 2681, 2682, 2683, 2684, 2685, 2686, 2687, 2688, 2689, 2690, 2691, 2692, 2693, 2694, 2695, 2696, 2697, 2698, 2699, 2700, 2701, 2702, 2703, 2704, 2705, 2706, 2707, 2708, 2709, 2710, 2711, 2712, 2713, 2714, 2715, 2716, 2717, 2718, 2719, 2720, 2721, 2722, 2723, 2724, 2725, 2726, 2727, 2728, 2729, 2730, 2731, 2732, 2733, 2734, 2735, 2736, 2737, 2738, 2739, 2740, 2741, 2742, 2743, 2744, 2745, 2746, 2747, 2748, 2749, 2750, 2751, 2752, 2753, 2754, 2755, 2756, 2757, 2758, 2759, 2760, 2761, 2762, 2763, 2764, 2765, 2766, 2767, 2768, 2769, 2770, 2771, 2772, 2773, 2774, 2775, 2776, 2777, 2778, 2779, 2780, 2781, 2782, 2783, 2784, 2785, 2786, 2787, 2788, 2789, 2790, 2791, 2792, 2793, 2794, 2795, 2796, 2797, 2798, 2799, 2800, 2801, 2802, 2803, 2804, 2805, 2806, 2807, 2808, 2809, 2810, 2811, 2812, 2813, 2814, 2815, 2816, 2817, 2818, 2819, 2820, 2821, 2822, 2823, 2824, 2825, 2826, 2827, 2828, 2829, 2830, 2831, 2832, 2833, 2834, 2835, 2836, 2837, 2838, 2839, 2840, 2841, 2842, 2843, 2844, 2845, 2846, 2847, 2848, 2849, 2850, 2851, 2852, 2853, 2854, 2855, 2856, 2857, 2858, 2859, 2860, 2861, 2862, 2863, 2864, 2865, 2866, 2867, 2868, 2869, 2870, 2871, 2872, 2873, 2874, 2875, 2876, 2877, 2878, 2879, 2880, 2881, 2882, 2883, 2884, 2885, 2886, 2887, 2888, 2889, 2890, 2891, 2892, 2893, 2894, 2895, 2896, 2897, 2898, 2899, 2900, 2901, 2902, 2903, 2904, 2905, 2906, 2907, 2908, 2909, 2910, 2911, 2912, 2913, 2914, 2915, 2916, 2917, 2918, 2919, 2920, 2921, 2922, 2923, 2924, 2925, 2926, 2927, 2928, 2929, 2930, 2931, 2932, 2933, 2934, 2935, 2936, 2937, 2938, 2939, 2940, 2941, 2942, 2943, 2944, 2945, 2946, 2947, 2948, 2949, 2950, 2951, 2952, 2953, 2954, 2955, 2956, 2957, 2958, 2959, 2960, 2961, 2962, 2963, 2964, 2965, 2966, 2967, 2968, 2969, 2970, 2971, 2972, 2973, 2974, 2975, 2976, 2977, 2978, 2979, 2980, 2981, 2982, 2983, 2984, 2985, 2986, 2987, 2988, 2989, 2990, 2991, 2992, 2993, 2994, 2995, 2996, 2997, 2998, 2999, 3000, 3001, 3002, 3003, 3004, 3005, 3006, 3007, 3008, 3009, 3010, 3011, 3012, 3013, 3014, 3015, 3016, 3017, 3018, 3019, 3020, 3021, 3022, 3023, 3024, 3025, 3026, 3027, 3028, 3029, 3030, 3031, 3032, 3033, 3034, 3035, 3036, 3037, 3038, 3039, 3040, 3041, 3042, 3043, 3044, 3045, 3046, 3047, 3048, 3049, 3050, 3051, 3052, 3053, 3054, 3055, 3056, 3057, 3058, 3059, 3060, 3061, 3062, 3063, 3064, 3065, 3066, 3067, 3068, 3069, 3070, 3071, 3072, 3073, 3074, 3075, 3076, 3077, 3078, 3079, 3080, 3081, 3082, 3083, 3084, 3085, 3086, 3087, 3088, 3089, 3090, 3091, 3092, 3093, 3094, 3095, 3096, 3097, 3098, 3099, 3100, 3101, 3102, 3103, 3104, 3105, 3106, 3107, 3108, 3109, 3110, 3111, 3112, 3113, 3114, 3115, 3116, 3117, 3118, 3119, 3120, 3121, 3122, 3123, 3124, 3125, 3126, 3127, 3128, 3129, 3130, 3131, 3132, 3133, 3134, 3135, 3136, 3137, 3138, 3139, 3140, 3141, 3142, 3143, 3144, 3145, 3146, 3147, 3148, 3149, 3150, 3151, 3152, 3153, 3154, 3155, 3156, 3157, 3158, 3159, 3160, 3161, 3162, 3163, 3164, 3165, 3166, 3167, 3168, 3169, 3170, 3171, 3172, 3173, 3174, 3175, 3176, 3177, 3178, 3179, 3180, 3181, 3182, 3183, 3184, 3185, 3186, 3187, 3188, 3189, 3190, 3191, 3192, 3193, 3194, 3195, 3196, 3197, 3198, 3199, 3200, 3201, 3202, 3203, 3204, 3205, 3206, 3207, 3208, 3209, 3210, 3211, 3212, 3213, 3214, 3215, 3216, 3217, 3218, 3219, 3220, 3221, 3222, 3223, 3224, 3225, 3226, 3227, 3228, 3229, 3230, 3231, 3232, 3233, 3234, 3235, 3236, 3237, 3238, 3239, 3240, 3241, 3242, 3243, 3244, 3245, 3246, 3247, 3248, 3249, 3250, 3251, 3252, 3253, 3254, 3255, 3256, 3257, 3258, 3259, 3260, 3261, 3262, 3263, 3264, 3265, 3266, 3267, 3268, 3269, 3270, 3271, 3272, 3273, 3274, 3275, 3276, 3277, 3278, 3279, 3280, 3281, 3282, 3283, 3284, 3285, 3286, 3287, 3288, 3289, 3290, 3291, 3292, 3293, 3294, 3295, 3296, 3297, 3298, 3299, 3300, 3301, 3302, 3303, 3304, 3305, 3306, 3307, 3308, 3309, 3310, 3311, 3312, 3313, 3314, 3315, 3316, 3317, 3318, 3319, 3320, 3321, 3322, 3323, 3324, 3325, 3326, 3327, 3328, 3329, 3330, 3331, 3332, 3333, 3334, 3335, 3336, 3337, 3338, 3339, 3340, 3341, 3342, 3343, 3344, 3345, 3346, 3347, 3348, 3349, 3350, 3351, 3352, 3353, 3354, 3355, 3356, 3357, 3358, 3359, 3360, 3361, 3362, 3363, 3364, 3365, 3366, 3367, 3368, 3369, 3370, 3371, 3372, 3373, 3374, 3375, 3376, 3377, 3378, 3379, 3380, 3381, 3382, 3383, 3384, 3385, 3386, 3387, 3388, 3389, 3390, 3391, 3392, 3393, 3394, 3395, 3396, 3397, 3398, 3399, 3400, 3401, 3402, 3403, 3404, 3405, 3406, 3407, 3408, 3409, 3410, 3411, 3412, 3413, 3414, 3415, 3416, 3417, 3418, 3419, 3420, 3421, 3422, 3423, 3424, 3425, 3426, 3427, 3428, 3429, 3430, 3431, 3432, 3433, 3434, 3435, 3436, 3437, 3438, 3439, 3440, 3441, 3442, 3443, 3444, 3445, 3446, 3447, 3448, 3449, 3450, 3451, 3452, 3453, 3454, 3455, 3456, 3457, 3458, 3459, 3460, 3461, 3462, 3463, 3464, 3465, 3466, 3467, 3468, 3469, 3470, 3471, 3472, 3473, 3474, 3475, 3476, 3477, 3478, 3479, 3480, 3481, 3482, 3483, 3484, 3485, 3486, 3487, 3488, 3489, 3490, 3491, 3492, 3493, 3494, 3495, 3496, 3497, 3498, 3499, 3500, 3501, 3502, 3503, 3504, 3505, 3506, 3507, 3508, 3509, 3510, 3511, 3512, 3513, 3514, 3515, 3516, 3517, 3518, 3519, 3520, 3521, 3522, 3523, 3524, 3525, 3526, 3527, 3528, 3529, 3530, 3531, 3532, 3533, 3534, 3535, 3536, 3537, 3538, 3539, 3540, 3541, 3542, 3543, 3544, 3545, 3546, 3547, 3548, 3549, 3550, 3551, 3552, 3553, 3554, 3555, 3556, 3557, 3558, 3559, 3560, 3561, 3562, 3563, 3564, 3565, 3566, 3567, 3568, 3569, 3570, 3571, 3572, 3573, 3574, 3575, 3576, 3577, 3578, 3579, 3580, 3581, 3582, 3583, 3584, 3585, 3586, 3587, 3588, 3589, 3590, 3591, 3592, 3593, 3594, 3595, 3596, 3597, 3598, 3599, 3600, 3601, 3602, 3603, 3604, 3605, 3606, 3607, 3608, 3609, 3610, 3611, 3612, 3613, 3614, 3615, 3616, 3617, 3618, 3619, 3620, 3621, 3622, 3623, 3624, 3625, 3626, 3627, 3628, 3629, 3630, 3631, 3632, 3633, 3634, 3635, 3636, 3637, 3638, 3639, 3640, 3641, 3642, 3643, 3644, 3645, 3646, 3647, 3648, 3649, 3650, 3651, 3652, 3653, 3654, 3655, 3656, 3657, 3658, 3659, 3660, 3661, 3662, 3663, 3664, 3665, 3666, 3667, 3668, 3669, 3670, 3671, 3672, 3673, 3674, 3675, 3676, 3677, 3678, 3679, 3680, 3681, 3682, 3683, 3684, 3685, 3686, 3687, 3688, 3689, 3690, 3691, 3692, 3693, 3694, 3695, 3696, 3697, 3698, 3699, 3700, 3701, 3702, 3703, 3704, 3705, 3706, 3707, 3708, 3709, 3710, 3711, 3712, 3713, 3714, 3715, 3716, 3717, 3718, 3719, 3720, 3721, 3722, 3723, 3724, 3725, 3726, 3727, 3728, 3729, 3730, 3731, 3732, 3733, 3734, 3735, 3736, 3737, 3738, 3739, 3740, 3741, 3742, 3743, 3744, 3745, 3746, 3747, 3748, 3749, 3750, 3751, 3752, 3753, 3754, 3755, 3756, 3757, 3758, 3759, 3760, 3761, 3762, 3763, 3764, 3765, 3766, 3767, 3768, 3769, 3770, 3771, 3772, 3773, 3774, 3775, 3776, 3777, 3778, 3779, 3780, 3781, 3782, 3783, 3784, 3785, 3786, 3787, 3788, 3789, 3790, 3791, 3792, 3793, 3794, 3795, 3796, 3797, 3798, 3799, 3800, 3801, 3802, 3803, 3804, 3805, 3806, 3807, 3808, 3809, 3810, 3811, 3812, 3813, 3814, 3815, 3816, 3817, 3818, 3819, 3820, 3821, 3822, 3823, 3824, 3825, 3826, 3827, 3828, 3829, 3830, 3831, 3832, 3833, 3834, 3835, 3836, 3837, 3838, 3839, 3840, 3841, 3842, 3843, 3844, 3845, 3846, 3847, 3848, 3849, 3850, 3851, 3852, 3853, 3854, 3855, 3856, 3857, 3858, 3859, 3860, 3861, 3862, 3863, 3864, 3865, 3866, 3867, 3868, 3869, 3870, 3871, 3872, 3873, 3874, 3875, 3876, 3877, 3878, 3879, 3880, 3881, 3882, 3883, 3884, 3885, 3886, 3887, 3888, 3889, 3890, 3891, 3892, 3893, 3894, 3895, 3896, 3897, 3898, 3899, 3900, 3901, 3902, 3903, 3904, 3905, 3906, 3907, 3908, 3909, 3910, 3911, 3912, 3913, 3914, 3915, 3916, 3917, 3918, 3919, 3920, 3921, 3922, 3923, 3924, 3925, 3926, 3927, 3928, 3929, 3930, 3931, 3932, 3933, 3934, 3935, 3936, 3937, 3938, 3939, 3940, 3941, 3942, 3943, 3944, 3945, 3946, 3947, 3948, 3949, 3950, 3951, 3952, 3953, 3954, 3955, 3956, 3957, 3958, 3959, 3960, 3961, 3962, 3963, 3964, 3965, 3966, 3967, 3968, 3969, 3970, 3971, 3972, 3973, 3974, 3975, 3976, 3977, 3978, 3979, 3980, 3981, 3982, 3983, 3984, 3985, 3986, 3987, 3988, 3989, 3990, 3991, 3992, 3993, 3994, 3995, 3996, 3997, 3998, 3999, 4000, 4001, 4002, 4003, 4004, 4005, 4006, 4007, 4008, 4009, 4010, 4011, 4012, 4013, 4014, 4015, 4016, 4017, 4018, 4019, 4020, 4021, 4022, 4023, 4024, 4025, 4026, 4027, 4028, 4029, 4030, 4031, 4032, 4033, 4034, 4035, 4036, 4037, 4038, 4039, 4040, 4041, 4042, 4043, 4044, 4045, 4046, 4047, 4048, 4049, 4050, 4051, 4052, 4053, 4054, 4055, 4056, 4057, 4058, 4059, 4060, 4061, 4062, 4063, 4064, 4065, 4066, 4067, 4068, 4069, 4070, 4071, 4072, 4073, 4074, 4075, 4076, 4077, 4078, 4079, 4080, 4081, 4082, 4083, 4084, 4085, 4086, 4087, 4088, 4089, 4090, 4091, 4092, 4093, 4094, 4095, 4096, 4097, 4098, 4099, 4100, 4101, 4102, 4103, 4104, 4105, 4106, 4107, 4108, 4109, 4110, 4111, 4112, 4113, 4114, 4115, 4116, 4117, 4118, 4119, 4120, 4121, 4122, 4123, 4124, 4125, 4126, 4127, 4128, 4129, 4130, 4131, 4132, 4133, 4134, 4135, 4136, 4137, 4138, 4139, 4140, 4141, 4142, 4143, 4144, 4145, 4146, 4147, 4148, 4149, 4150, 4151, 4152, 4153, 4154, 4155, 4156, 4157, 4158, 4159, 4160, 4161, 4162, 4163, 4164, 4165, 4166, 4167, 4168, 4169, 4170, 4171, 4172, 4173, 4174, 4175, 4176, 4177, 4178, 4179, 4180, 4181, 4182, 4183, 4184, 4185, 4186, 4187, 4188, 4189, 4190, 4191, 4192, 4193, 4194, 4195, 4196, 4197, 4198, 4199, 4200, 4201, 4202, 4203, 4204, 4205, 4206, 4207, 4208, 4209, 4210, 4211, 4212, 4213, 4214, 4215, 4216, 4217, 4218, 4219, 4220, 4221, 4222, 4223, 4224, 4225, 4226, 4227, 4228, 4229, 4230, 4231, 4232, 4233, 4234, 4235, 4236, 4237, 4238, 4239, 4240, 4241, 4242, 4243, 4244, 4245, 4246, 4247, 4248, 4249, 4250, 4251, 4252, 4253, 4254, 4255, 4256, 4257, 4258, 4259, 4260, 4261, 4262, 4263, 4264, 4265, 4266, 4267, 4268, 4269, 4270, 4271, 4272, 4273, 4274, 4275, 4276, 4277, 4278, 4279, 4280, 4281, 4282, 4283, 4284, 4285, 4286, 4287, 4288, 4289, 4290, 4291, 4292, 4293, 4294, 4295, 4296, 4297, 4298, 4299, 4300, 4301, 4302, 4303, 4304, 4305, 4306, 4307, 4308, 4309, 4310, 4311, 4312, 4313, 4314, 4315, 4316, 4317, 4318, 4319, 4320, 4321, 4322, 4323, 4324, 4325, 4326, 4327, 4328, 4329, 4330, 4331, 4332, 4333, 4334, 4335, 4336, 4337, 4338, 4339, 4340, 4341, 4342, 4343, 4344, 4345, 4346, 4347, 4348, 4349, 4350, 4351, 4352, 4353, 4354, 4355, 4356, 4357, 4358, 4359, 4360, 4361, 4362, 4363, 4364, 4365, 4366, 4367, 4368, 4369, 4370, 4371, 4372, 4373, 4374, 4375, 4376, 4377, 4378, 4379, 4380, 4381, 4382, 4383, 4384, 4385, 4386, 4387, 4388, 4389, 4390, 4391, 4392, 4393, 4394, 4395, 4396, 4397, 4398, 4399, 4400, 4401, 4402, 4403, 4404, 4405, 4406, 4407, 4408, 4409, 4410, 4411, 4412, 4413, 4414, 4415, 4416, 4417, 4418, 4419, 4420, 4421, 4422, 4423, 4424, 4425, 4426, 4427, 4428, 4429, 4430, 4431, 4432, 4433, 4434, 4435, 4436, 4437, 4438, 4439, 4440, 4441, 4442, 4443, 4444, 4445, 4446, 4447, 4448, 4449, 4450, 4451, 4452, 4453, 4454, 4455, 4456, 4457, 4458, 4459, 4460, 4461, 4462, 4463, 4464, 4465, 4466, 4467, 4468, 4469, 4470, 4471, 4472, 4473, 4474, 4475, 4476, 4477, 4478, 4479, 4480, 4481, 4482, 4483, 4484, 4485, 4486, 4487, 4488, 4489, 4490, 4491, 4492, 4493, 4494, 4495, 4496, 4497, 4498, 4499, 4500, 4501, 4502, 4503, 4504, 4505, 4506, 4507, 4508, 4509, 4510, 4511, 4512, 4513, 4514, 4515, 4516, 4517, 4518, 4519, 4520, 4521, 4522, 4523, 4524, 4525, 4526, 4527, 4528, 4529, 4530, 4531, 4532, 4533, 4534, 4535, 4536, 4537, 4538, 4539, 4540, 4541, 4542, 4543, 4544, 4545, 4546, 4547, 4548, 4549, 4550, 4551, 4552, 4553, 4554, 4555, 4556, 4557, 4558, 4559, 4560, 4561, 4562, 4563, 4564, 4565, 4566, 4567, 4568, 4569, 4570, 4571, 4572, 4573, 4574, 4575, 4576, 4577, 4578, 4579, 4580, 4581, 4582, 4583, 4584, 4585, 4586, 4587, 4588, 4589, 4590, 4591, 4592, 4593, 4594, 4595, 4596, 4597, 4598, 4599, 4600, 4601, 4602, 4603, 4604, 4605, 4606, 4607, 4608, 4609, 4610, 4611, 4612, 4613, 4614, 4615, 4616, 4617, 4618, 4619, 4620, 4621, 4622, 4623, 4624, 4625, 4626, 4627, 4628, 4629, 4630, 4631, 4632, 4633, 4634, 4635, 4636, 4637, 4638, 4639, 4640, 4641, 4642, 4643, 4644, 4645, 4646, 4647, 4648, 4649, 4650, 4651, 4652, 4653, 4654, 4655, 4656, 4657, 4658, 4659, 4660, 4661, 4662, 4663, 4664, 4665, 4666, 4667, 4668, 4669, 4670, 4671, 4672, 4673, 4674, 4675, 4676, 4677, 4678, 4679, 4680, 4681, 4682, 4683, 4684, 4685, 4686, 4687, 4688, 4689, 4690, 4691, 4692, 4693, 4694, 4695, 4696, 4697, 4698, 4699, 4700, 4701, 4702, 4703, 4704, 4705, 4706, 4707, 4708, 4709, 4710, 4711, 4712, 4713, 4714, 4715, 4716, 4717, 4718, 4719, 4720, 4721, 4722, 4723, 4724, 4725, 4726, 4727, 4728, 4729, 4730, 4731, 4732, 4733, 4734, 4735, 4736, 4737, 4738, 4739, 4740, 4741, 4742, 4743, 4744, 4745, 4746, 4747, 4748, 4749, 4750, 4751, 4752, 4753, 4754, 4755, 4756, 4757, 4758, 4759, 4760, 4761, 4762, 4763, 4764, 4765, 4766, 4767, 4768, 4769, 4770, 4771, 4772, 4773, 4774, 4775, 4776, 4777, 4778, 4779, 4780, 4781, 4782, 4783, 4784, 4785, 4786, 4787, 4788, 4789, 4790, 4791, 4792, 4793, 4794, 4795, 4796, 4797, 4798, 4799, 4800, 4801, 4802, 4803, 4804, 4805, 4806, 4807, 4808, 4809, 4810, 4811, 4812, 4813, 4814, 4815, 4816, 4817, 4818, 4819, 4820, 4821, 4822, 4823, 4824, 4825, 4826, 4827, 4828, 4829, 4830, 4831, 4832, 4833, 4834, 4835, 4836, 4837, 4838, 4839, 4840, 4841, 4842, 4843, 4844, 4845, 4846, 4847, 4848, 4849, 4850, 4851, 4852, 4853, 4854, 4855, 4856, 4857, 4858, 4859, 4860, 4861, 4862, 4863, 4864, 4865, 4866, 4867, 4868, 4869, 4870, 4871, 4872, 4873, 4874, 4875, 4876, 4877, 4878, 4879, 4880, 4881, 4882, 4883, 4884, 4885, 4886, 4887, 4888, 4889, 4890, 4891, 4892, 4893, 4894, 4895, 4896, 4897, 4898, 4899, 4900, 4901, 4902, 4903, 4904, 4905, 4906, 4907, 4908, 4909, 4910, 4911, 4912, 4913, 4914, 4915, 4916, 4917, 4918, 4919, 4920, 4921, 4922, 4923, 4924, 4925, 4926, 4927, 4928, 4929, 4930, 4931, 4932, 4933, 4934, 4935, 4936, 4937, 4938, 4939, 4940, 4941, 4942, 4943, 4944, 4945, 4946, 4947, 4948, 4949, 4950, 4951, 4952, 4953, 4954, 4955, 4956, 4957, 4958, 4959, 4960, 4961, 4962, 4963, 4964, 4965, 4966, 4967, 4968, 4969, 4970, 4971, 4972, 4973, 4974, 4975, 4976, 4977, 4978, 4979, 4980, 4981, 4982, 4983, 4984, 4985, 4986, 4987, 4988, 4989, 4990, 4991, 4992, 4993, 4994, 4995, 4996, 4997, 4998, 4999, 5000, 5001, 5002, 5003, 5004, 5005, 5006, 5007, 5008, 5009, 5010, 5011, 5012, 5013, 5014, 5015, 5016, 5017, 5018, 5019, 5020, 5021, 5022, 5023, 5024, 5025, 5026, 5027, 5028, 5029, 5030, 5031, 5032, 5033, 5034, 5035, 5036, 5037, 5038, 5039, 5040, 5041, 5042, 5043, 5044, 5045, 5046, 5047, 5048, 5049, 5050, 5051, 5052, 5053, 5054, 5055, 5056, 5057, 5058, 5059, 5060, 5061, 5062, 5063, 5064, 5065, 5066, 5067, 5068, 5069, 5070, 5071, 5072, 5073, 5074, 5075, 5076, 5077, 5078, 5079, 5080, 5081, 5082, 5083, 5084, 5085, 5086, 5087, 5088, 5089, 5090, 5091, 5092, 5093, 5094, 5095, 5096, 5097, 5098, 5099, 5100, 5101, 5102, 5103, 5104, 5105, 5106, 5107, 5108, 5109, 5110, 5111, 5112, 5113, 5114, 5115, 5116, 5117, 5118, 5119, 5120, 5121, 5122, 5123, 5124, 5125, 5126, 5127, 5128, 5129, 5130, 5131, 5132, 5133, 5134, 5135, 5136, 5137, 5138, 5139, 5140, 5141, 5142, 5143, 5144, 5145, 5146, 5147, 5148, 5149, 5150, 5151, 5152, 5153, 5154, 5155, 5156, 5157, 5158, 5159, 5160, 5161, 5162, 5163, 5164, 5165, 5166, 5167, 5168, 5169, 5170, 5171, 5172, 5173, 5174, 5175, 5176, 5177, 5178, 5179, 5180, 5181, 5182, 5183, 5184, 5185, 5186, 5187, 5188, 5189, 5190, 5191, 5192, 5193, 5194, 5195, 5196, 5197, 5198, 5199, 5200, 5201, 5202, 5203, 5204, 5205, 5206, 5207, 5208, 5209, 5210, 5211, 5212, 5213, 5214, 5215, 5216, 5217, 5218, 5219, 5220, 5221, 5222, 5223, 5224, 5225, 5226, 5227, 5228, 5229, 5230, 5231, 5232, 5233, 5234, 5235, 5236, 5237, 5238, 5239, 5240, 5241, 5242, 5243, 5244, 5245, 5246, 5247, 5248, 5249, 5250, 5251, 5252, 5253, 5254, 5255, 5256, 5257, 5258, 5259, 5260, 5261, 5262, 5263, 5264, 5265, 5266, 5267, 5268, 5269, 5270, 5271, 5272, 5273, 5274, 5275, 5276, 5277, 5278, 5279, 5280, 5281, 5282, 5283, 5284, 5285, 5286, 5287, 5288, 5289, 5290, 5291, 5292, 5293, 5294, 5295, 5296, 5297, 5298, 5299, 5300, 5301, 5302, 5303, 5304, 5305, 5306, 5307, 5308, 5309, 5310, 5311, 5312, 5313, 5314, 5315, 5316, 5317, 5318, 5319, 5320, 5321, 5322, 5323, 5324, 5325, 5326, 5327, 5328, 5329, 5330, 5331, 5332, 5333, 5334, 5335, 5336, 5337, 5338, 5339, 5340, 5341, 5342, 5343, 5344, 5345, 5346, 5347, 5348, 5349, 5350, 5351, 5352, 5353, 5354, 5355, 5356, 5357, 5358, 5359, 5360, 5361, 5362, 5363, 5364, 5365, 5366, 5367, 5368, 5369, 5370, 5371, 5372, 5373, 5374, 5375, 5376, 5377, 5378, 5379, 5380, 5381, 5382, 5383, 5384, 5385, 5386, 5387, 5388, 5389, 5390, 5391, 5392, 5393, 5394, 5395, 5396, 5397, 5398, 5399, 5400, 5401, 5402, 5403, 5404, 5405, 5406, 5407, 5408, 5409, 5410, 5411, 5412, 5413, 5414, 5415, 5416, 5417, 5418, 5419, 5420, 5421, 5422, 5423, 5424, 5425, 5426, 5427, 5428, 5429, 5430, 5431, 5432, 5433, 5434, 5435, 5436, 5437, 5438, 5439, 5440, 5441, 5442, 5443, 5444, 5445, 5446, 5447, 5448, 5449, 5450, 5451, 5452, 5453, 5454, 5455, 5456, 5457, 5458, 5459, 5460, 5461, 5462, 5463, 5464, 5465, 5466, 5467, 5468, 5469, 5470, 5471, 5472, 5473, 5474, 5475, 5476, 5477, 5478, 5479, 5480, 5481, 5482, 5483, 5484, 5485, 5486, 5487, 5488, 5489, 5490, 5491, 5492, 5493, 5494, 5495, 5496, 5497, 5498, 5499, 5500, 5501, 5502, 5503, 5504, 5505, 5506, 5507, 5508, 5509, 5510, 5511, 5512, 5513, 5514, 5515, 5516, 5517, 5518, 5519, 5520, 5521, 5522, 5523, 5524, 5525, 5526, 5527, 5528, 5529, 5530, 5531, 5532, 5533, 5534, 5535, 5536, 5537, 5538, 5539, 5540, 5541, 5542, 5543, 5544, 5545, 5546, 5547, 5548, 5549, 5550, 5551, 5552, 5553, 5554, 5555, 5556, 5557, 5558, 5559, 5560, 5561, 5562, 5563, 5564, 5565, 5566, 5567, 5568, 5569, 5570, 5571, 5572, 5573, 5574, 5575, 5576, 5577, 5578, 5579, 5580, 5581, 5582, 5583, 5584, 5585, 5586, 5587, 5588, 5589, 5590, 5591, 5592, 5593, 5594, 5595, 5596, 5597, 5598, 5599, 5600, 5601, 5602, 5603, 5604, 5605, 5606, 5607, 5608, 5609, 5610, 5611, 5612, 5613, 5614, 5615, 5616, 5617, 5618, 5619, 5620, 5621, 5622, 5623, 5624, 5625, 5626, 5627, 5628, 5629, 5630, 5631, 5632, 5633, 5634, 5635, 5636, 5637, 5638, 5639, 5640, 5641, 5642, 5643, 5644, 5645, 5646, 5647, 5648, 5649, 5650, 5651, 5652, 5653, 5654, 5655, 5656, 5657, 5658, 5659, 5660, 5661, 5662, 5663, 5664, 5665, 5666, 5667, 5668, 5669, 5670, 5671, 5672, 5673, 5674, 5675, 5676, 5677, 5678, 5679, 5680, 5681, 5682, 5683, 5684, 5685, 5686, 5687, 5688, 5689, 5690, 5691, 5692, 5693, 5694, 5695, 5696, 5697, 5698, 5699, 5700, 5701, 5702, 5703, 5704, 5705, 5706, 5707, 5708, 5709, 5710, 5711, 5712, 5713, 5714, 5715, 5716, 5717, 5718, 5719, 5720, 5721, 5722, 5723, 5724, 5725, 5726, 5727, 5728, 5729, 5730, 5731, 5732, 5733, 5734, 5735, 5736, 5737, 5738, 5739, 5740, 5741, 5742, 5743, 5744, 5745, 5746, 5747, 5748, 5749, 5750, 5751, 5752, 5753, 5754, 5755, 5756, 5757, 5758, 5759, 5760, 5761, 5762, 5763, 5764, 5765, 5766, 5767, 5768, 5769, 5770, 5771, 5772, 5773, 5774, 5775, 5776, 5777, 5778, 5779, 5780, 5781, 5782, 5783, 5784, 5785, 5786, 5787, 5788, 5789, 5790, 5791, 5792, 5793, 5794, 5795, 5796, 5797, 5798, 5799, 5800, 5801, 5802, 5803, 5804, 5805, 5806, 5807, 5808, 5809, 5810, 5811, 5812, 5813, 5814, 5815, 5816, 5817, 5818, 5819, 5820, 5821, 5822, 5823, 5824, 5825, 5826, 5827, 5828, 5829, 5830, 5831, 5832, 5833, 5834, 5835, 5836, 5837, 5838, 5839, 5840, 5841, 5842, 5843, 5844, 5845, 5846, 5847, 5848, 5849, 5850, 5851, 5852, 5853, 5854, 5855, 5856, 5857, 5858, 5859, 5860, 5861, 5862, 5863, 5864, 5865, 5866, 5867, 5868, 5869, 5870, 5871, 5872, 5873, 5874, 5875, 5876, 5877, 5878, 5879, 5880, 5881, 5882, 5883, 5884, 5885, 5886, 5887, 5888, 5889, 5890, 5891, 5892, 5893, 5894, 5895, 5896, 5897, 5898, 5899, 5900, 5901, 5902, 5903, 5904, 5905, 5906, 5907, 5908, 5909, 5910, 5911, 5912, 5913, 5914, 5915, 5916, 5917, 5918, 5919, 5920, 5921, 5922, 5923, 5924, 5925, 5926, 5927, 5928, 5929, 5930, 5931, 5932, 5933, 5934, 5935, 5936, 5937, 5938, 5939, 5940, 5941, 5942, 5943, 5944, 5945, 5946, 5947, 5948, 5949, 5950, 5951, 5952, 5953, 5954, 5955, 5956, 5957, 5958, 5959, 5960, 5961, 5962, 5963, 5964, 5965, 5966, 5967, 5968, 5969, 5970, 5971, 5972, 5973, 5974, 5975, 5976, 5977, 5978, 5979, 5980, 5981, 5982, 5983, 5984, 5985, 5986, 5987, 5988, 5989, 5990, 5991, 5992, 5993, 5994, 5995, 5996, 5997, 5998, 5999, 6000, 6001, 6002, 6003, 6004, 6005, 6006, 6007, 6008, 6009, 6010, 6011, 6012, 6013, 6014, 6015, 6016, 6017, 6018, 6019, 6020, 6021, 6022, 6023, 6024, 6025, 6026, 6027, 6028, 6029, 6030, 6031, 6032, 6033, 6034, 6035, 6036, 6037, 6038, 6039, 6040, 6041, 6042, 6043, 6044, 6045, 6046, 6047, 6048, 6049, 6050, 6051, 6052, 6053, 6054, 6055, 6056, 6057, 6058, 6059, 6060, 6061, 6062, 6063, 6064, 6065, 6066, 6067, 6068, 6069, 6070, 6071, 6072, 6073, 6074, 6075, 6076, 6077, 6078, 6079, 6080, 6081, 6082, 6083, 6084, 6085, 6086, 6087, 6088, 6089, 6090, 6091, 6092, 6093, 6094, 6095, 6096, 6097, 6098, 6099, 6100, 6101, 6102, 6103, 6104, 6105, 6106, 6107, 6108, 6109, 6110, 6111, 6112, 6113, 6114, 6115, 6116, 6117, 6118, 6119, 6120, 6121, 6122, 6123, 6124, 6125, 6126, 6127, 6128, 6129, 6130, 6131, 6132, 6133, 6134, 6135, 6136, 6137, 6138, 6139, 6140, 6141, 6142, 6143, 6144, 6145, 6146, 6147, 6148, 6149, 6150, 6151, 6152, 6153, 6154, 6155, 6156, 6157, 6158, 6159, 6160, 6161, 6162, 6163, 6164, 6165, 6166, 6167, 6168, 6169, 6170, 6171, 6172, 6173, 6174, 6175, 6176, 6177, 6178, 6179, 6180, 6181, 6182, 6183, 6184, 6185, 6186, 6187, 6188, 6189, 6190, 6191, 6192, 6193, 6194, 6195, 6196, 6197, 6198, 6199, 6200, 6201, 6202, 6203, 6204, 6205, 6206, 6207, 6208, 6209, 6210, 6211, 6212, 6213, 6214, 6215, 6216, 6217, 6218, 6219, 6220, 6221, 6222, 6223, 6224, 6225, 6226, 6227, 6228, 6229, 6230, 6231, 6232, 6233, 6234, 6235, 6236, 6237, 6238, 6239, 6240, 6241, 6242, 6243, 6244, 6245, 6246, 6247, 6248, 6249, 6250, 6251, 6252, 6253, 6254, 6255, 6256, 6257, 6258, 6259, 6260, 6261, 6262, 6263, 6264, 6265, 6266, 6267, 6268, 6269, 6270, 6271, 6272, 6273, 6274, 6275, 6276, 6277, 6278, 6279, 6280, 6281, 6282, 6283, 6284, 6285, 6286, 6287, 6288, 6289, 6290, 6291, 6292, 6293, 6294, 6295, 6296, 6297, 6298, 6299, 6300, 6301, 6302, 6303, 6304, 6305, 6306, 6307, 6308, 6309, 6310, 6311, 6312, 6313, 6314, 6315, 6316, 6317, 6318, 6319, 6320, 6321, 6322, 6323, 6324, 6325, 6326, 6327, 6328, 6329, 6330, 6331, 6332, 6333, 6334, 6335, 6336, 6337, 6338, 6339, 6340, 6341, 6342, 6343, 6344, 6345, 6346, 6347, 6348, 6349, 6350, 6351, 6352, 6353, 6354, 6355, 6356, 6357, 6358, 6359, 6360, 6361, 6362, 6363, 6364, 6365, 6366, 6367, 6368, 6369, 6370, 6371, 6372, 6373, 6374, 6375, 6376, 6377, 6378, 6379, 6380, 6381, 6382, 6383, 6384, 6385, 6386, 6387, 6388, 6389, 6390, 6391, 6392, 6393, 6394, 6395, 6396, 6397, 6398, 6399, 6400, 6401, 6402, 6403, 6404, 6405, 6406, 6407, 6408, 6409, 6410, 6411, 6412, 6413, 6414, 6415, 6416, 6417, 6418, 6419, 6420, 6421, 6422, 6423, 6424, 6425, 6426, 6427, 6428, 6429, 6430, 6431, 6432, 6433, 6434, 6435, 6436, 6437, 6438, 6439, 6440, 6441, 6442, 6443, 6444, 6445, 6446, 6447, 6448, 6449, 6450, 6451, 6452, 6453, 6454, 6455, 6456, 6457, 6458, 6459, 6460, 6461, 6462, 6463, 6464, 6465, 6466, 6467, 6468, 6469, 6470, 6471, 6472, 6473, 6474, 6475, 6476, 6477, 6478, 6479, 6480, 6481, 6482, 6483, 6484, 6485, 6486, 6487, 6488, 6489, 6490, 6491, 6492, 6493, 6494, 6495, 6496, 6497, 6498, 6499, 6500, 6501, 6502, 6503, 6504, 6505, 6506, 6507, 6508, 6509, 6510, 6511, 6512, 6513, 6514, 6515, 6516, 6517, 6518, 6519, 6520, 6521, 6522, 6523, 6524, 6525, 6526, 6527, 6528, 6529, 6530, 6531, 6532, 6533, 6534, 6535, 6536, 6537, 6538, 6539, 6540, 6541, 6542, 6543, 6544, 6545, 6546, 6547, 6548, 6549, 6550, 6551, 6552, 6553, 6554, 6555, 6556, 6557, 6558, 6559, 6560, 6561, 6562, 6563, 6564, 6565, 6566, 6567, 6568, 6569, 6570, 6571, 6572, 6573, 6574, 6575, 6576, 6577, 6578, 6579, 6580, 6581, 6582, 6583, 6584, 6585, 6586, 6587, 6588, 6589, 6590, 6591, 6592, 6593, 6594, 6595, 6596, 6597, 6598, 6599, 6600, 6601, 6602, 6603, 6604, 6605, 6606, 6607, 6608, 6609, 6610, 6611, 6612, 6613, 6614, 6615, 6616, 6617, 6618, 6619, 6620, 6621, 6622, 6623, 6624, 6625, 6626, 6627, 6628, 6629, 6630, 6631, 6632, 6633, 6634, 6635, 6636, 6637, 6638, 6639, 6640, 6641, 6642, 6643, 6644, 6645, 6646, 6647, 6648, 6649, 6650, 6651, 6652, 6653, 6654, 6655, 6656, 6657, 6658, 6659, 6660, 6661, 6662, 6663, 6664, 6665, 6666, 6667, 6668, 6669, 6670, 6671, 6672, 6673, 6674, 6675, 6676, 6677, 6678, 6679, 6680, 6681, 6682, 6683, 6684, 6685, 6686, 6687, 6688, 6689, 6690, 6691, 6692, 6693, 6694, 6695, 6696, 6697, 6698, 6699, 6700, 6701, 6702, 6703, 6704, 6705, 6706, 6707, 6708, 6709, 6710, 6711, 6712, 6713, 6714, 6715, 6716, 6717, 6718, 6719, 6720, 6721, 6722, 6723, 6724, 6725, 6726, 6727, 6728, 6729, 6730, 6731, 6732, 6733, 6734, 6735, 6736, 6737, 6738, 6739, 6740, 6741, 6742, 6743, 6744, 6745, 6746, 6747, 6748, 6749, 6750, 6751, 6752, 6753, 6754, 6755, 6756, 6757, 6758, 6759, 6760, 6761, 6762, 6763, 6764, 6765, 6766, 6767, 6768, 6769, 6770, 6771, 6772, 6773, 6774, 6775, 6776, 6777, 6778, 6779, 6780, 6781, 6782, 6783, 6784, 6785, 6786, 6787, 6788, 6789, 6790, 6791, 6792, 6793, 6794, 6795, 6796, 6797, 6798, 6799, 6800, 6801, 6802, 6803, 6804, 6805, 6806, 6807, 6808, 6809, 6810, 6811, 6812, 6813, 6814, 6815, 6816, 6817, 6818, 6819, 6820, 6821, 6822, 6823, 6824, 6825, 6826, 6827, 6828, 6829, 6830, 6831, 6832, 6833, 6834, 6835, 6836, 6837, 6838, 6839, 6840, 6841, 6842, 6843, 6844, 6845, 6846, 6847, 6848, 6849, 6850, 6851, 6852, 6853, 6854, 6855, 6856, 6857, 6858, 6859, 6860, 6861, 6862, 6863, 6864, 6865, 6866, 6867, 6868, 6869, 6870, 6871, 6872, 6873, 6874, 6875, 6876, 6877, 6878, 6879, 6880, 6881, 6882, 6883, 6884, 6885, 6886, 6887, 6888, 6889, 6890, 6891, 6892, 6893, 6894, 6895, 6896, 6897, 6898, 6899, 6900, 6901, 6902, 6903, 6904, 6905, 6906, 6907, 6908, 6909, 6910, 6911, 6912, 6913, 6914, 6915, 6916, 6917, 6918, 6919, 6920, 6921, 6922, 6923, 6924, 6925, 6926, 6927, 6928, 6929, 6930, 6931, 6932, 6933, 6934, 6935, 6936, 6937, 6938, 6939, 6940, 6941, 6942, 6943, 6944, 6945, 6946, 6947, 6948, 6949, 6950, 6951, 6952, 6953, 6954, 6955, 6956, 6957, 6958, 6959, 6960, 6961, 6962, 6963, 6964, 6965, 6966, 6967, 6968, 6969, 6970, 6971, 6972, 6973, 6974, 6975, 6976, 6977, 6978, 6979, 6980, 6981, 6982, 6983, 6984, 6985, 6986, 6987, 6988, 6989, 6990, 6991, 6992, 6993, 6994, 6995, 6996, 6997, 6998, 6999, 7000, 7001, 7002, 7003, 7004, 7005, 7006, 7007, 7008, 7009, 7010, 7011, 7012, 7013, 7014, 7015, 7016, 7017, 7018, 7019, 7020, 7021, 7022, 7023, 7024, 7025, 7026, 7027, 7028, 7029, 7030, 7031, 7032, 7033, 7034, 7035, 7036, 7037, 7038, 7039, 7040, 7041, 7042, 7043, 7044, 7045, 7046, 7047, 7048, 7049, 7050, 7051, 7052, 7053, 7054, 7055, 7056, 7057, 7058, 7059, 7060, 7061, 7062, 7063, 7064, 7065, 7066, 7067, 7068, 7069, 7070, 7071, 7072, 7073, 7074, 7075, 7076, 7077, 7078, 7079, 7080, 7081, 7082, 7083, 7084, 7085, 7086, 7087, 7088, 7089, 7090, 7091, 7092, 7093, 7094, 7095, 7096, 7097, 7098, 7099, 7100, 7101, 7102, 7103, 7104, 7105, 7106, 7107, 7108, 7109, 7110, 7111, 7112, 7113, 7114, 7115, 7116, 7117, 7118, 7119, 7120, 7121, 7122, 7123, 7124, 7125, 7126, 7127, 7128, 7129, 7130, 7131, 7132, 7133, 7134, 7135, 7136, 7137, 7138, 7139, 7140, 7141, 7142, 7143, 7144, 7145, 7146, 7147, 7148, 7149, 7150, 7151, 7152, 7153, 7154, 7155, 7156, 7157, 7158, 7159, 7160, 7161, 7162, 7163, 7164, 7165, 7166, 7167, 7168, 7169, 7170, 7171, 7172, 7173, 7174, 7175, 7176, 7177, 7178, 7179, 7180, 7181, 7182, 7183, 7184, 7185, 7186, 7187, 7188, 7189, 7190, 7191, 7192, 7193, 7194, 7195, 7196, 7197, 7198, 7199, 7200, 7201, 7202, 7203, 7204, 7205, 7206, 7207, 7208, 7209, 7210, 7211, 7212, 7213, 7214, 7215, 7216, 7217, 7218, 7219, 7220, 7221, 7222, 7223, 7224, 7225, 7226, 7227, 7228, 7229, 7230, 7231, 7232, 7233, 7234, 7235, 7236, 7237, 7238, 7239, 7240, 7241, 7242, 7243, 7244, 7245, 7246, 7247, 7248, 7249, 7250, 7251, 7252, 7253, 7254, 7255, 7256, 7257, 7258, 7259, 7260, 7261, 7262, 7263, 7264, 7265, 7266, 7267, 7268, 7269, 7270, 7271, 7272, 7273, 7274, 7275, 7276, 7277, 7278, 7279, 7280, 7281, 7282, 7283, 7284, 7285, 7286, 7287, 7288, 7289, 7290, 7291, 7292, 7293, 7294, 7295, 7296, 7297, 7298, 7299, 7300, 7301, 7302, 7303, 7304, 7305, 7306, 7307, 7308, 7309, 7310, 7311, 7312, 7313, 7314, 7315, 7316, 7317, 7318, 7319, 7320, 7321, 7322, 7323, 7324, 7325, 7326, 7327, 7328, 7329, 7330, 7331, 7332, 7333, 7334, 7335, 7336, 7337, 7338, 7339, 7340, 7341, 7342, 7343, 7344, 7345, 7346, 7347, 7348, 7349, 7350, 7351, 7352, 7353, 7354, 7355, 7356, 7357, 7358, 7359, 7360, 7361, 7362, 7363, 7364, 7365, 7366, 7367, 7368, 7369, 7370, 7371, 7372, 7373, 7374, 7375, 7376, 7377, 7378, 7379, 7380, 7381, 7382, 7383, 7384, 7385, 7386, 7387, 7388, 7389, 7390, 7391, 7392, 7393, 7394, 7395, 7396, 7397, 7398, 7399, 7400, 7401, 7402, 7403, 7404, 7405, 7406, 7407, 7408, 7409, 7410, 7411, 7412, 7413, 7414, 7415, 7416, 7417, 7418, 7419, 7420, 7421, 7422, 7423, 7424, 7425, 7426, 7427, 7428, 7429, 7430, 7431, 7432, 7433, 7434, 7435, 7436, 7437, 7438, 7439, 7440, 7441, 7442, 7443, 7444, 7445, 7446, 7447, 7448, 7449, 7450, 7451, 7452, 7453, 7454, 7455, 7456, 7457, 7458, 7459, 7460, 7461, 7462, 7463, 7464, 7465, 7466, 7467, 7468, 7469, 7470, 7471, 7472, 7473, 7474, 7475, 7476, 7477, 7478, 7479, 7480, 7481, 7482, 7483, 7484, 7485, 7486, 7487, 7488, 7489, 7490, 7491, 7492, 7493, 7494, 7495, 7496, 7497, 7498, 7499, 7500, 7501, 7502, 7503, 7504, 7505, 7506, 7507, 7508, 7509, 7510, 7511, 7512, 7513, 7514, 7515, 7516, 7517, 7518, 7519, 7520, 7521, 7522, 7523, 7524, 7525, 7526, 7527, 7528, 7529, 7530, 7531, 7532, 7533, 7534, 7535, 7536, 7537, 7538, 7539, 7540, 7541, 7542, 7543, 7544, 7545, 7546, 7547, 7548, 7549, 7550, 7551, 7552, 7553, 7554, 7555, 7556, 7557, 7558, 7559, 7560, 7561, 7562, 7563, 7564, 7565, 7566, 7567, 7568, 7569, 7570, 7571, 7572, 7573, 7574, 7575, 7576, 7577, 7578, 7579, 7580, 7581, 7582, 7583, 7584, 7585, 7586, 7587, 7588, 7589, 7590, 7591, 7592, 7593, 7594, 7595, 7596, 7597, 7598, 7599, 7600, 7601, 7602, 7603, 7604, 7605, 7606, 7607, 7608, 7609, 7610, 7611, 7612, 7613, 7614, 7615, 7616, 7617, 7618, 7619, 7620, 7621, 7622, 7623, 7624, 7625, 7626, 7627, 7628, 7629, 7630, 7631, 7632, 7633, 7634, 7635, 7636, 7637, 7638, 7639, 7640, 7641, 7642, 7643, 7644, 7645, 7646, 7647, 7648, 7649, 7650, 7651, 7652, 7653, 7654, 7655, 7656, 7657, 7658, 7659, 7660, 7661, 7662, 7663, 7664, 7665, 7666, 7667, 7668, 7669, 7670, 7671, 7672, 7673, 7674, 7675, 7676, 7677, 7678, 7679, 7680, 7681, 7682, 7683, 7684, 7685, 7686, 7687, 7688, 7689, 7690, 7691, 7692, 7693, 7694, 7695, 7696, 7697, 7698, 7699, 7700, 7701, 7702, 7703, 7704, 7705, 7706, 7707, 7708, 7709, 7710, 7711, 7712, 7713, 7714, 7715, 7716, 7717, 7718, 7719, 7720, 7721, 7722, 7723, 7724, 7725, 7726, 7727, 7728, 7729, 7730, 7731, 7732, 7733, 7734, 7735, 7736, 7737, 7738, 7739, 7740, 7741, 7742, 7743, 7744, 7745, 7746, 7747, 7748, 7749, 7750, 7751, 7752, 7753, 7754, 7755, 7756, 7757, 7758, 7759, 7760, 7761, 7762, 7763, 7764, 7765, 7766, 7767, 7768, 7769, 7770, 7771, 7772, 7773, 7774, 7775, 7776, 7777, 7778, 7779, 7780, 7781, 7782, 7783, 7784, 7785, 7786, 7787, 7788, 7789, 7790, 7791, 7792, 7793, 7794, 7795, 7796, 7797, 7798, 7799, 7800, 7801, 7802, 7803, 7804, 7805, 7806, 7807, 7808, 7809, 7810, 7811, 7812, 7813, 7814, 7815, 7816, 7817, 7818, 7819, 7820, 7821, 7822, 7823, 7824, 7825, 7826, 7827, 7828, 7829, 7830, 7831, 7832, 7833, 7834, 7835, 7836, 7837, 7838, 7839, 7840, 7841, 7842, 7843, 7844, 7845, 7846, 7847, 7848, 7849, 7850, 7851, 7852, 7853, 7854, 7855, 7856, 7857, 7858, 7859, 7860, 7861, 7862, 7863, 7864, 7865, 7866, 7867, 7868, 7869, 7870, 7871, 7872, 7873, 7874, 7875, 7876, 7877, 7878, 7879, 7880, 7881, 7882, 7883, 7884, 7885, 7886, 7887, 7888, 7889, 7890, 7891, 7892, 7893, 7894, 7895, 7896, 7897, 7898, 7899, 7900, 7901, 7902, 7903, 7904, 7905, 7906, 7907, 7908, 7909, 7910, 7911, 7912, 7913, 7914, 7915, 7916, 7917, 7918, 7919, 7920, 7921, 7922, 7923, 7924, 7925, 7926, 7927, 7928, 7929, 7930, 7931, 7932, 7933, 7934, 7935, 7936, 7937, 7938, 7939, 7940, 7941, 7942, 7943, 7944, 7945, 7946, 7947, 7948, 7949, 7950, 7951, 7952, 7953, 7954, 7955, 7956, 7957, 7958, 7959, 7960, 7961, 7962, 7963, 7964, 7965, 7966, 7967, 7968, 7969, 7970, 7971, 7972, 7973, 7974, 7975, 7976, 7977, 7978, 7979, 7980, 7981, 7982, 7983, 7984, 7985, 7986, 7987, 7988, 7989, 7990, 7991, 7992, 7993, 7994, 7995, 7996, 7997, 7998, 7999, 8000, 8001, 8002, 8003, 8004, 8005, 8006, 8007, 8008, 8009, 8010, 8011, 8012, 8013, 8014, 8015, 8016, 8017, 8018, 8019, 8020, 8021, 8022, 8023, 8024, 8025, 8026, 8027, 8028, 8029, 8030, 8031, 8032, 8033, 8034, 8035, 8036, 8037, 8038, 8039, 8040, 8041, 8042, 8043, 8044, 8045, 8046, 8047, 8048, 8049, 8050, 8051, 8052, 8053, 8054, 8055, 8056, 8057, 8058, 8059, 8060, 8061, 8062, 8063, 8064, 8065, 8066, 8067, 8068, 8069, 8070, 8071, 8072, 8073, 8074, 8075, 8076, 8077, 8078, 8079, 8080, 8081, 8082, 8083, 8084, 8085, 8086, 8087, 8088, 8089, 8090, 8091, 8092, 8093, 8094, 8095, 8096, 8097, 8098, 8099, 8100, 8101, 8102, 8103, 8104, 8105, 8106, 8107, 8108, 8109, 8110, 8111, 8112, 8113, 8114, 8115, 8116, 8117, 8118, 8119, 8120, 8121, 8122, 8123, 8124, 8125, 8126, 8127, 8128, 8129, 8130, 8131, 8132, 8133, 8134, 8135, 8136, 8137, 8138, 8139, 8140, 8141, 8142, 8143, 8144, 8145, 8146, 8147, 8148, 8149, 8150, 8151, 8152, 8153, 8154, 8155, 8156, 8157, 8158, 8159, 8160, 8161, 8162, 8163, 8164, 8165, 8166, 8167, 8168, 8169, 8170, 8171, 8172, 8173, 8174, 8175, 8176, 8177, 8178, 8179, 8180, 8181, 8182, 8183, 8184, 8185, 8186, 8187, 8188, 8189, 8190, 8191, 8192, 8193, 8194, 8195, 8196, 8197, 8198, 8199, 8200, 8201, 8202, 8203, 8204, 8205, 8206, 8207, 8208, 8209, 8210, 8211, 8212, 8213, 8214, 8215, 8216, 8217, 8218, 8219, 8220, 8221, 8222, 8223, 8224, 8225, 8226, 8227, 8228, 8229, 8230, 8231, 8232, 8233, 8234, 8235, 8236, 8237, 8238, 8239, 8240, 8241, 8242, 8243, 8244, 8245, 8246, 8247, 8248, 8249, 8250, 8251, 8252, 8253, 8254, 8255, 8256, 8257, 8258, 8259, 8260, 8261, 8262, 8263, 8264, 8265, 8266, 8267, 8268, 8269, 8270, 8271, 8272, 8273, 8274, 8275, 8276, 8277, 8278, 8279, 8280, 8281, 8282, 8283, 8284, 8285, 8286, 8287, 8288, 8289, 8290, 8291, 8292, 8293, 8294, 8295, 8296, 8297, 8298, 8299, 8300, 8301, 8302, 8303, 8304, 8305, 8306, 8307, 8308, 8309, 8310, 8311, 8312, 8313, 8314, 8315, 8316, 8317, 8318, 8319, 8320, 8321, 8322, 8323, 8324, 8325, 8326, 8327, 8328, 8329, 8330, 8331, 8332, 8333, 8334, 8335, 8336, 8337, 8338, 8339, 8340, 8341, 8342, 8343, 8344, 8345, 8346, 8347, 8348, 8349, 8350, 8351, 8352, 8353, 8354, 8355, 8356, 8357, 8358, 8359, 8360, 8361, 8362, 8363, 8364, 8365, 8366, 8367, 8368, 8369, 8370, 8371, 8372, 8373, 8374, 8375, 8376, 8377, 8378, 8379, 8380, 8381, 8382, 8383, 8384, 8385, 8386, 8387, 8388, 8389, 8390, 8391, 8392, 8393, 8394, 8395, 8396, 8397, 8398, 8399, 8400, 8401, 8402, 8403, 8404, 8405, 8406, 8407, 8408, 8409, 8410, 8411, 8412, 8413, 8414, 8415, 8416, 8417, 8418, 8419, 8420, 8421, 8422, 8423, 8424, 8425, 8426, 8427, 8428, 8429, 8430, 8431, 8432, 8433, 8434, 8435, 8436, 8437, 8438, 8439, 8440, 8441, 8442, 8443, 8444, 8445, 8446, 8447, 8448, 8449, 8450, 8451, 8452, 8453, 8454, 8455, 8456, 8457, 8458, 8459, 8460, 8461, 8462, 8463, 8464, 8465, 8466, 8467, 8468, 8469, 8470, 8471, 8472, 8473, 8474, 8475, 8476, 8477, 8478, 8479, 8480, 8481, 8482, 8483, 8484, 8485, 8486, 8487, 8488, 8489, 8490, 8491, 8492, 8493, 8494, 8495, 8496, 8497, 8498, 8499, 8500, 8501, 8502, 8503, 8504, 8505, 8506, 8507, 8508, 8509, 8510, 8511, 8512, 8513, 8514, 8515, 8516, 8517, 8518, 8519, 8520, 8521, 8522, 8523, 8524, 8525, 8526, 8527, 8528, 8529, 8530, 8531, 8532, 8533, 8534, 8535, 8536, 8537, 8538, 8539, 8540, 8541, 8542, 8543, 8544, 8545, 8546, 8547, 8548, 8549, 8550, 8551, 8552, 8553, 8554, 8555, 8556, 8557, 8558, 8559, 8560, 8561, 8562, 8563, 8564, 8565, 8566, 8567, 8568, 8569, 8570, 8571, 8572, 8573, 8574, 8575, 8576, 8577, 8578, 8579, 8580, 8581, 8582, 8583, 8584, 8585, 8586, 8587, 8588, 8589, 8590, 8591, 8592, 8593, 8594, 8595, 8596, 8597, 8598, 8599, 8600, 8601, 8602, 8603, 8604, 8605, 8606, 8607, 8608, 8609, 8610, 8611, 8612, 8613, 8614, 8615, 8616, 8617, 8618, 8619, 8620, 8621, 8622, 8623, 8624, 8625, 8626, 8627, 8628, 8629, 8630, 8631, 8632, 8633, 8634, 8635, 8636, 8637, 8638, 8639, 8640, 8641, 8642, 8643, 8644, 8645, 8646, 8647, 8648, 8649, 8650, 8651, 8652, 8653, 8654, 8655, 8656, 8657, 8658, 8659, 8660, 8661, 8662, 8663, 8664, 8665, 8666, 8667, 8668, 8669, 8670, 8671, 8672, 8673, 8674, 8675, 8676, 8677, 8678, 8679, 8680, 8681, 8682, 8683, 8684, 8685, 8686, 8687, 8688, 8689, 8690, 8691, 8692, 8693, 8694, 8695, 8696, 8697, 8698, 8699, 8700, 8701, 8702, 8703, 8704, 8705, 8706, 8707, 8708, 8709, 8710, 8711, 8712, 8713, 8714, 8715, 8716, 8717, 8718, 8719, 8720, 8721, 8722, 8723, 8724, 8725, 8726, 8727, 8728, 8729, 8730, 8731, 8732, 8733, 8734, 8735, 8736, 8737, 8738, 8739, 8740, 8741, 8742, 8743, 8744, 8745, 8746, 8747, 8748, 8749, 8750, 8751, 8752, 8753, 8754, 8755, 8756, 8757, 8758, 8759, 8760, 8761, 8762, 8763, 8764, 8765, 8766, 8767, 8768, 8769, 8770, 8771, 8772, 8773, 8774, 8775, 8776, 8777, 8778, 8779, 8780, 8781, 8782, 8783, 8784, 8785, 8786, 8787, 8788, 8789, 8790, 8791, 8792, 8793, 8794, 8795, 8796, 8797, 8798, 8799, 8800, 8801, 8802, 8803, 8804, 8805, 8806, 8807, 8808, 8809, 8810, 8811, 8812, 8813, 8814, 8815, 8816, 8817, 8818, 8819, 8820, 8821, 8822, 8823, 8824, 8825, 8826, 8827, 8828, 8829, 8830, 8831, 8832, 8833, 8834, 8835, 8836, 8837, 8838, 8839, 8840, 8841, 8842, 8843, 8844, 8845, 8846, 8847, 8848, 8849, 8850, 8851, 8852, 8853, 8854, 8855, 8856, 8857, 8858, 8859, 8860, 8861, 8862, 8863, 8864, 8865, 8866, 8867, 8868, 8869, 8870, 8871, 8872, 8873, 8874, 8875, 8876, 8877, 8878, 8879, 8880, 8881, 8882, 8883, 8884, 8885, 8886, 8887, 8888, 8889, 8890, 8891, 8892, 8893, 8894, 8895, 8896, 8897, 8898, 8899, 8900, 8901, 8902, 8903, 8904, 8905, 8906, 8907, 8908, 8909, 8910, 8911, 8912, 8913, 8914, 8915, 8916, 8917, 8918, 8919, 8920, 8921, 8922, 8923, 8924, 8925, 8926, 8927, 8928, 8929, 8930, 8931, 8932, 8933, 8934, 8935, 8936, 8937, 8938, 8939, 8940, 8941, 8942, 8943, 8944, 8945, 8946, 8947, 8948, 8949, 8950, 8951, 8952, 8953, 8954, 8955, 8956, 8957, 8958, 8959, 8960, 8961, 8962, 8963, 8964, 8965, 8966, 8967, 8968, 8969, 8970, 8971, 8972, 8973, 8974, 8975, 8976, 8977, 8978, 8979, 8980, 8981, 8982, 8983, 8984, 8985, 8986, 8987, 8988, 8989, 8990, 8991, 8992, 8993, 8994, 8995, 8996, 8997, 8998, 8999, 9000, 9001, 9002, 9003, 9004, 9005, 9006, 9007, 9008, 9009, 9010, 9011, 9012, 9013, 9014, 9015, 9016, 9017, 9018, 9019, 9020, 9021, 9022, 9023, 9024, 9025, 9026, 9027, 9028, 9029, 9030, 9031, 9032, 9033, 9034, 9035, 9036, 9037, 9038, 9039, 9040, 9041, 9042, 9043, 9044, 9045, 9046, 9047, 9048, 9049, 9050, 9051, 9052, 9053, 9054, 9055, 9056, 9057, 9058, 9059, 9060, 9061, 9062, 9063, 9064, 9065, 9066, 9067, 9068, 9069, 9070, 9071, 9072, 9073, 9074, 9075, 9076, 9077, 9078, 9079, 9080, 9081, 9082, 9083, 9084, 9085, 9086, 9087, 9088, 9089, 9090, 9091, 9092, 9093, 9094, 9095, 9096, 9097, 9098, 9099, 9100, 9101, 9102, 9103, 9104, 9105, 9106, 9107, 9108, 9109, 9110, 9111, 9112, 9113, 9114, 9115, 9116, 9117, 9118, 9119, 9120, 9121, 9122, 9123, 9124, 9125, 9126, 9127, 9128, 9129, 9130, 9131, 9132, 9133, 9134, 9135, 9136, 9137, 9138, 9139, 9140, 9141, 9142, 9143, 9144, 9145, 9146, 9147, 9148, 9149, 9150, 9151, 9152, 9153, 9154, 9155, 9156, 9157, 9158, 9159, 9160, 9161, 9162, 9163, 9164, 9165, 9166, 9167, 9168, 9169, 9170, 9171, 9172, 9173, 9174, 9175, 9176, 9177, 9178, 9179, 9180, 9181, 9182, 9183, 9184, 9185, 9186, 9187, 9188, 9189, 9190, 9191, 9192, 9193, 9194, 9195, 9196, 9197, 9198, 9199, 9200, 9201, 9202, 9203, 9204, 9205, 9206, 9207, 9208, 9209, 9210, 9211, 9212, 9213, 9214, 9215, 9216, 9217, 9218, 9219, 9220, 9221, 9222, 9223, 9224, 9225, 9226, 9227, 9228, 9229, 9230, 9231, 9232, 9233, 9234, 9235, 9236, 9237, 9238, 9239, 9240, 9241, 9242, 9243, 9244, 9245, 9246, 9247, 9248, 9249, 9250, 9251, 9252, 9253, 9254, 9255, 9256, 9257, 9258, 9259, 9260, 9261, 9262, 9263, 9264, 9265, 9266, 9267, 9268, 9269, 9270, 9271, 9272, 9273, 9274, 9275, 9276, 9277, 9278, 9279, 9280, 9281, 9282, 9283, 9284, 9285, 9286, 9287, 9288, 9289, 9290, 9291, 9292, 9293, 9294, 9295, 9296, 9297, 9298, 9299, 9300, 9301, 9302, 9303, 9304, 9305, 9306, 9307, 9308, 9309, 9310, 9311, 9312, 9313, 9314, 9315, 9316, 9317, 9318, 9319, 9320, 9321, 9322, 9323, 9324, 9325, 9326, 9327, 9328, 9329, 9330, 9331, 9332, 9333, 9334, 9335, 9336, 9337, 9338, 9339, 9340, 9341, 9342, 9343, 9344, 9345, 9346, 9347, 9348, 9349, 9350, 9351, 9352, 9353, 9354, 9355, 9356, 9357, 9358, 9359, 9360, 9361, 9362, 9363, 9364, 9365, 9366, 9367, 9368, 9369, 9370, 9371, 9372, 9373, 9374, 9375, 9376, 9377, 9378, 9379, 9380, 9381, 9382, 9383, 9384, 9385, 9386, 9387, 9388, 9389, 9390, 9391, 9392, 9393, 9394, 9395, 9396, 9397, 9398, 9399, 9400, 9401, 9402, 9403, 9404, 9405, 9406, 9407, 9408, 9409, 9410, 9411, 9412, 9413, 9414, 9415, 9416, 9417, 9418, 9419, 9420, 9421, 9422, 9423, 9424, 9425, 9426, 9427, 9428, 9429, 9430, 9431, 9432, 9433, 9434, 9435, 9436, 9437, 9438, 9439, 9440, 9441, 9442, 9443, 9444, 9445, 9446, 9447, 9448, 9449, 9450, 9451, 9452, 9453, 9454, 9455, 9456, 9457, 9458, 9459, 9460, 9461, 9462, 9463, 9464, 9465, 9466, 9467, 9468, 9469, 9470, 9471, 9472, 9473, 9474, 9475, 9476, 9477, 9478, 9479, 9480, 9481, 9482, 9483, 9484, 9485, 9486, 9487, 9488, 9489, 9490, 9491, 9492, 9493, 9494, 9495, 9496, 9497, 9498, 9499, 9500, 9501, 9502, 9503, 9504, 9505, 9506, 9507, 9508, 9509, 9510, 9511, 9512, 9513, 9514, 9515, 9516, 9517, 9518, 9519, 9520, 9521, 9522, 9523, 9524, 9525, 9526, 9527, 9528, 9529, 9530, 9531, 9532, 9533, 9534, 9535, 9536, 9537, 9538, 9539, 9540, 9541, 9542, 9543, 9544, 9545, 9546, 9547, 9548, 9549, 9550, 9551, 9552, 9553, 9554, 9555, 9556, 9557, 9558, 9559, 9560, 9561, 9562, 9563, 9564, 9565, 9566, 9567, 9568, 9569, 9570, 9571, 9572, 9573, 9574, 9575, 9576, 9577, 9578, 9579, 9580, 9581, 9582, 9583, 9584, 9585, 9586, 9587, 9588, 9589, 9590, 9591, 9592, 9593, 9594, 9595, 9596, 9597, 9598, 9599, 9600, 9601, 9602, 9603, 9604, 9605, 9606, 9607, 9608, 9609, 9610, 9611, 9612, 9613, 9614, 9615, 9616, 9617, 9618, 9619, 9620, 9621, 9622, 9623, 9624, 9625, 9626, 9627, 9628, 9629, 9630, 9631, 9632, 9633, 9634, 9635, 9636, 9637, 9638, 9639, 9640, 9641, 9642, 9643, 9644, 9645, 9646, 9647, 9648, 9649, 9650, 9651, 9652, 9653, 9654, 9655, 9656, 9657, 9658, 9659, 9660, 9661, 9662, 9663, 9664, 9665, 9666, 9667, 9668, 9669, 9670, 9671, 9672, 9673, 9674, 9675, 9676, 9677, 9678, 9679, 9680, 9681, 9682, 9683, 9684, 9685, 9686, 9687, 9688, 9689, 9690, 9691, 9692, 9693, 9694, 9695, 9696, 9697, 9698, 9699, 9700, 9701, 9702, 9703, 9704, 9705, 9706, 9707, 9708, 9709, 9710, 9711, 9712, 9713, 9714, 9715, 9716, 9717, 9718, 9719, 9720, 9721, 9722, 9723, 9724, 9725, 9726, 9727, 9728, 9729, 9730, 9731, 9732, 9733, 9734, 9735, 9736, 9737, 9738, 9739, 9740, 9741, 9742, 9743, 9744, 9745, 9746, 9747, 9748, 9749, 9750, 9751, 9752, 9753, 9754, 9755, 9756, 9757, 9758, 9759, 9760, 9761, 9762, 9763, 9764, 9765, 9766, 9767, 9768, 9769, 9770, 9771, 9772, 9773, 9774, 9775, 9776, 9777, 9778, 9779, 9780, 9781, 9782, 9783, 9784, 9785, 9786, 9787, 9788, 9789, 9790, 9791, 9792, 9793, 9794, 9795, 9796, 9797, 9798, 9799, 9800, 9801, 9802, 9803, 9804, 9805, 9806, 9807, 9808, 9809, 9810, 9811, 9812, 9813, 9814, 9815, 9816, 9817, 9818, 9819, 9820, 9821, 9822, 9823, 9824, 9825, 9826, 9827, 9828, 9829, 9830, 9831, 9832, 9833, 9834, 9835, 9836, 9837, 9838, 9839, 9840, 9841, 9842, 9843, 9844, 9845, 9846, 9847, 9848, 9849, 9850, 9851, 9852, 9853, 9854, 9855, 9856, 9857, 9858, 9859, 9860, 9861, 9862, 9863, 9864, 9865, 9866, 9867, 9868, 9869, 9870, 9871, 9872, 9873, 9874, 9875, 9876, 9877, 9878, 9879, 9880, 9881, 9882, 9883, 9884, 9885, 9886, 9887, 9888, 9889, 9890, 9891, 9892, 9893, 9894, 9895, 9896, 9897, 9898, 9899, 9900, 9901, 9902, 9903, 9904, 9905, 9906, 9907, 9908, 9909, 9910, 9911, 9912, 9913, 9914, 9915, 9916, 9917, 9918, 9919, 9920, 9921, 9922, 9923, 9924, 9925, 9926, 9927, 9928, 9929, 9930, 9931, 9932, 9933, 9934, 9935, 9936, 9937, 9938, 9939, 9940, 9941, 9942, 9943, 9944, 9945, 9946, 9947, 9948, 9949, 9950, 9951, 9952, 9953, 9954, 9955, 9956, 9957, 9958, 9959, 9960, 9961, 9962, 9963, 9964, 9965, 9966, 9967, 9968, 9969, 9970, 9971, 9972, 9973, 9974, 9975, 9976, 9977, 9978, 9979, 9980, 9981, 9982, 9983, 9984, 9985, 9986, 9987, 9988, 9989, 9990, 9991, 9992, 9993, 9994, 9995, 9996, 9997, 9998, 9999, 10000, 10001, 10002, 10003, 10004, 10005, 10006, 10007, 10008, 10009, 10010, 10011, 10012, 10013, 10014, 10015, 10016, 10017, 10018, 10019, 10020, 10021, 10022, 10023, 10024, 10025, 10026, 10027, 10028, 10029, 10030, 10031, 10032, 10033, 10034, 10035, 10036, 10037, 10038, 10039, 10040, 10041, 10042, 10043, 10044, 10045, 10046, 10047, 10048, 10049, 10050, 10051, 10052, 10053, 10054, 10055, 10056, 10057, 10058, 10059, 10060, 10061, 10062, 10063, 10064, 10065, 10066, 10067, 10068, 10069, 10070, 10071, 10072, 10073, 10074, 10075, 10076, 10077, 10078, 10079, 10080, 10081, 10082, 10083, 10084, 10085, 10086, 10087, 10088, 10089, 10090, 10091, 10092, 10093, 10094, 10095, 10096, 10097, 10098, 10099, 10100, 10101, 10102, 10103, 10104, 10105, 10106, 10107, 10108, 10109, 10110, 10111, 10112, 10113, 10114, 10115, 10116, 10117, 10118, 10119, 10120, 10121, 10122, 10123, 10124, 10125, 10126, 10127, 10128, 10129, 10130, 10131, 10132, 10133, 10134, 10135, 10136, 10137, 10138, 10139, 10140, 10141, 10142, 10143, 10144, 10145, 10146, 10147, 10148, 10149, 10150, 10151, 10152, 10153, 10154, 10155, 10156, 10157, 10158, 10159, 10160, 10161, 10162, 10163, 10164, 10165, 10166, 10167, 10168, 10169, 10170, 10171, 10172, 10173, 10174, 10175, 10176, 10177, 10178, 10179, 10180, 10181, 10182, 10183, 10184, 10185, 10186, 10187, 10188, 10189, 10190, 10191, 10192, 10193, 10194, 10195, 10196, 10197, 10198, 10199, 10200, 10201, 10202, 10203, 10204, 10205, 10206, 10207, 10208, 10209, 10210, 10211, 10212, 10213, 10214, 10215, 10216, 10217, 10218, 10219, 10220, 10221, 10222, 10223, 10224, 10225, 10226, 10227, 10228, 10229, 10230, 10231, 10232, 10233, 10234, 10235, 10236, 10237, 10238, 10239, 10240, 10241, 10242, 10243, 10244, 10245, 10246, 10247, 10248, 10249, 10250, 10251, 10252, 10253, 10254, 10255, 10256, 10257, 10258, 10259, 10260, 10261, 10262, 10263, 10264, 10265, 10266, 10267, 10268, 10269, 10270, 10271, 10272, 10273, 10274, 10275, 10276, 10277, 10278, 10279, 10280, 10281, 10282, 10283, 10284, 10285, 10286, 10287, 10288, 10289, 10290, 10291, 10292, 10293, 10294, 10295, 10296, 10297, 10298, 10299, 10300, 10301, 10302, 10303, 10304, 10305, 10306, 10307, 10308, 10309, 10310, 10311, 10312, 10313, 10314, 10315, 10316, 10317, 10318, 10319, 10320, 10321, 10322, 10323, 10324, 10325, 10326, 10327, 10328, 10329, 10330, 10331, 10332, 10333, 10334, 10335, 10336, 10337, 10338, 10339, 10340, 10341, 10342, 10343, 10344, 10345, 10346, 10347, 10348, 10349, 10350, 10351, 10352, 10353, 10354, 10355, 10356, 10357, 10358, 10359, 10360, 10361, 10362, 10363, 10364, 10365, 10366, 10367, 10368, 10369, 10370, 10371, 10372, 10373, 10374, 10375, 10376, 10377, 10378, 10379, 10380, 10381, 10382, 10383, 10384, 10385, 10386, 10387, 10388, 10389, 10390, 10391, 10392, 10393, 10394, 10395, 10396, 10397, 10398, 10399, 10400, 10401, 10402, 10403, 10404, 10405, 10406, 10407, 10408, 10409, 10410, 10411, 10412, 10413, 10414, 10415, 10416, 10417, 10418, 10419, 10420, 10421, 10422, 10423, 10424, 10425, 10426, 10427, 10428, 10429, 10430, 10431, 10432, 10433, 10434, 10435, 10436, 10437, 10438, 10439, 10440, 10441, 10442, 10443, 10444, 10445, 10446, 10447, 10448, 10449, 10450, 10451, 10452, 10453, 10454, 10455, 10456, 10457, 10458, 10459, 10460, 10461, 10462, 10463, 10464, 10465, 10466, 10467, 10468, 10469, 10470, 10471, 10472, 10473, 10474, 10475, 10476, 10477, 10478, 10479, 10480, 10481, 10482, 10483, 10484, 10485, 10486, 10487, 10488, 10489, 10490, 10491, 10492, 10493, 10494, 10495, 10496, 10497, 10498, 10499, 10500, 10501, 10502, 10503, 10504, 10505, 10506, 10507, 10508, 10509, 10510, 10511, 10512, 10513, 10514, 10515, 10516, 10517, 10518, 10519, 10520, 10521, 10522, 10523, 10524, 10525, 10526, 10527, 10528, 10529, 10530, 10531, 10532, 10533, 10534, 10535, 10536, 10537, 10538, 10539, 10540, 10541, 10542, 10543, 10544, 10545, 10546, 10547, 10548, 10549, 10550, 10551, 10552, 10553, 10554, 10555, 10556, 10557, 10558, 10559, 10560, 10561, 10562, 10563, 10564, 10565, 10566, 10567, 10568, 10569, 10570, 10571, 10572, 10573, 10574, 10575, 10576, 10577, 10578, 10579, 10580, 10581, 10582, 10583, 10584, 10585, 10586, 10587, 10588, 10589, 10590, 10591, 10592, 10593, 10594, 10595, 10596, 10597, 10598, 10599, 10600, 10601, 10602, 10603, 10604, 10605, 10606, 10607, 10608, 10609, 10610, 10611, 10612, 10613, 10614, 10615, 10616, 10617, 10618, 10619, 10620, 10621, 10622, 10623, 10624, 10625, 10626, 10627, 10628, 10629, 10630, 10631, 10632, 10633, 10634, 10635, 10636, 10637, 10638, 10639, 10640, 10641, 10642, 10643, 10644, 10645, 10646, 10647, 10648, 10649, 10650, 10651, 10652, 10653, 10654, 10655, 10656, 10657, 10658, 10659, 10660, 10661, 10662, 10663, 10664, 10665, 10666, 10667, 10668, 10669, 10670, 10671, 10672, 10673, 10674, 10675, 10676, 10677, 10678, 10679, 10680, 10681, 10682, 10683, 10684, 10685, 10686, 10687, 10688, 10689, 10690, 10691, 10692, 10693, 10694, 10695, 10696, 10697, 10698, 10699, 10700, 10701, 10702, 10703, 10704, 10705, 10706, 10707, 10708, 10709, 10710, 10711, 10712, 10713, 10714, 10715, 10716, 10717, 10718, 10719, 10720, 10721, 10722, 10723, 10724, 10725, 10726, 10727, 10728, 10729, 10730, 10731, 10732, 10733, 10734, 10735, 10736, 10737, 10738, 10739, 10740, 10741, 10742, 10743, 10744, 10745, 10746, 10747, 10748, 10749, 10750, 10751, 10752, 10753, 10754, 10755, 10756, 10757, 10758, 10759, 10760, 10761, 10762, 10763, 10764, 10765, 10766, 10767, 10768, 10769, 10770, 10771, 10772, 10773, 10774, 10775, 10776, 10777, 10778, 10779, 10780, 10781, 10782, 10783, 10784, 10785, 10786, 10787, 10788, 10789, 10790, 10791, 10792, 10793, 10794, 10795, 10796, 10797, 10798, 10799, 10800, 10801, 10802, 10803, 10804, 10805, 10806, 10807, 10808, 10809, 10810, 10811, 10812, 10813, 10814, 10815, 10816, 10817, 10818, 10819, 10820, 10821, 10822, 10823, 10824, 10825, 10826, 10827, 10828, 10829, 10830, 10831, 10832, 10833, 10834, 10835, 10836, 10837, 10838, 10839, 10840, 10841, 10842, 10843, 10844, 10845, 10846, 10847, 10848, 10849, 10850, 10851, 10852, 10853, 10854, 10855, 10856, 10857, 10858, 10859, 10860, 10861, 10862, 10863, 10864, 10865, 10866, 10867, 10868, 10869, 10870, 10871, 10872, 10873, 10874, 10875, 10876, 10877, 10878, 10879, 10880, 10881, 10882, 10883, 10884, 10885, 10886, 10887, 10888, 10889, 10890, 10891, 10892, 10893, 10894, 10895, 10896, 10897, 10898, 10899, 10900, 10901, 10902, 10903, 10904, 10905, 10906, 10907, 10908, 10909, 10910, 10911, 10912, 10913, 10914, 10915, 10916, 10917, 10918, 10919, 10920, 10921, 10922, 10923, 10924, 10925, 10926, 10927, 10928, 10929, 10930, 10931, 10932, 10933, 10934, 10935, 10936, 10937, 10938, 10939, 10940, 10941, 10942, 10943, 10944, 10945, 10946, 10947, 10948, 10949, 10950, 10951, 10952, 10953, 10954, 10955, 10956, 10957, 10958, 10959, 10960, 10961, 10962, 10963, 10964, 10965, 10966, 10967, 10968, 10969, 10970, 10971, 10972, 10973, 10974, 10975, 10976, 10977, 10978, 10979, 10980, 10981, 10982, 10983, 10984, 10985, 10986, 10987, 10988, 10989, 10990, 10991, 10992, 10993, 10994, 10995, 10996, 10997, 10998, 10999, 11000, 11001, 11002, 11003, 11004, 11005, 11006, 11007, 11008, 11009, 11010, 11011, 11012, 11013, 11014, 11015, 11016, 11017, 11018, 11019, 11020, 11021, 11022, 11023, 11024, 11025, 11026, 11027, 11028, 11029, 11030, 11031, 11032, 11033, 11034, 11035, 11036, 11037, 11038, 11039, 11040, 11041, 11042, 11043, 11044, 11045, 11046, 11047, 11048, 11049, 11050, 11051, 11052, 11053, 11054, 11055, 11056, 11057, 11058, 11059, 11060, 11061, 11062, 11063, 11064, 11065, 11066, 11067, 11068, 11069, 11070, 11071, 11072, 11073, 11074, 11075, 11076, 11077, 11078, 11079, 11080, 11081, 11082, 11083, 11084, 11085, 11086, 11087, 11088, 11089, 11090, 11091, 11092, 11093, 11094, 11095, 11096, 11097, 11098, 11099, 11100, 11101, 11102, 11103, 11104, 11105, 11106, 11107, 11108, 11109, 11110, 11111, 11112, 11113, 11114, 11115, 11116, 11117, 11118, 11119, 11120, 11121, 11122, 11123, 11124, 11125, 11126, 11127, 11128, 11129, 11130, 11131, 11132, 11133, 11134, 11135, 11136, 11137, 11138, 11139, 11140, 11141, 11142, 11143, 11144, 11145, 11146, 11147, 11148, 11149, 11150, 11151, 11152, 11153, 11154, 11155, 11156, 11157, 11158, 11159, 11160, 11161, 11162, 11163, 11164, 11165, 11166, 11167, 11168, 11169, 11170, 11171, 11172, 11173, 11174, 11175, 11176, 11177, 11178, 11179, 11180, 11181, 11182, 11183, 11184, 11185, 11186, 11187, 11188, 11189, 11190, 11191, 11192, 11193, 11194, 11195, 11196, 11197, 11198, 11199, 11200, 11201, 11202, 11203, 11204, 11205, 11206, 11207, 11208, 11209, 11210, 11211, 11212, 11213, 11214, 11215, 11216, 11217, 11218, 11219, 11220, 11221, 11222, 11223, 11224, 11225, 11226, 11227, 11228, 11229, 11230, 11231, 11232, 11233, 11234, 11235, 11236, 11237, 11238, 11239, 11240, 11241, 11242, 11243, 11244, 11245, 11246, 11247, 11248, 11249, 11250, 11251, 11252, 11253, 11254, 11255, 11256, 11257, 11258, 11259, 11260, 11261, 11262, 11263, 11264, 11265, 11266, 11267, 11268, 11269, 11270, 11271, 11272, 11273, 11274, 11275, 11276, 11277, 11278, 11279, 11280, 11281, 11282, 11283, 11284, 11285, 11286, 11287, 11288, 11289, 11290, 11291, 11292, 11293, 11294, 11295, 11296, 11297, 11298, 11299, 11300, 11301, 11302, 11303, 11304, 11305, 11306, 11307, 11308, 11309]

In [15]:
def show_each():
    for i in range(len(Objects)):
        obj = Objects[i]
        v, f, patch2f = obj['v'], obj['f'], obj['patch']
        f_colors = np.ones((f.shape[0], 3))
        print(f"Object {i}, Patch {len(patch2f)}")
        for p_id in range(len(patch2f)):
            f_colors[patch2f[p_id]] = colors[p_id % 100]
        mp.plot(v, f, f_colors, shading={
            'wireframe': False
        })

def show_each_patch():
    for i in range(len(Objects)):
        obj = Objects[i]
        v, f, patch2f = obj['v'], obj['f'], obj['patch']
        for p_id in range(len(patch2f)):
            print(f"Object {i}, Patch {p_id}, {len(patch2f[p_id])}")
            f_colors = np.ones((f.shape[0], 3))
            f_colors[patch2f[p_id]] = colors[p_id % 100]
            mp.plot(v, f, f_colors, shading={
                'wireframe': False
            })
        break

def show_all(write = False):
    v_all = None
    f_all = None
    cur_v_len = 0
    cur_f_len = 0
    patch2f_all = []
    for obj in Objects:
        v, f, patch2f = obj['v'], obj['f'], obj['patch']
        if v_all is None:
            v_all = v
        else:
            v_all = np.concatenate((v_all, v), axis=0)
        if f_all is None:
            f_all = f
        else:
            f = f + cur_v_len
            f_all = np.concatenate((f_all, f), axis=0)
        
        for i in range(len(patch2f)):
            patch2f_all.append(list(np.array(patch2f[i]) + cur_f_len))

        cur_v_len = v_all.shape[0]
        cur_f_len = f_all.shape[0]
        

    f_colors = np.ones((f_all.shape[0], 3))
    for i in range(len(patch2f_all)):
        f_colors[patch2f_all[i]] = colors[i % 100]
    mp.plot(v_all, f_all, f_colors, shading={
        'wireframe': True
    })

    if write:
        write_ply_file(v_all, f_all, f_colors, 
            # f'example_results/merge/example_CAD/{obj_name}_seg.ply'
            f'example_results/merge/selected/example_result/part2surf/{obj_name}.ply'
            # f'example_results/merge/selected/example_result/our/{obj_name}.ply'
            # f'example_results/merge/cad/cad16/{obj_name}.ply'
        )

        # save labels
        # label_file = f'example_results/merge/cad/cad16/{obj_name}.seg'
        # with open(label_file, 'w') as file:
        #     for i in range(f_all.shape[0]):
        #         for j in range(len(patch2f_all)):
        #             if i in patch2f_all[j]:
        #                 file.write(f"{j}\n")
        #                 break

# show_each()
# show_each_patch()
# show_all()
show_all(write=True)
import os
def save_all_objects():
    # folder = f'example_results/merge/selected/{obj_name}/seg_choose_2/'
    # if not os.path.exists(folder):
    #     os.makedirs(folder)
    global_color_id = 0
    for i in range(len(Objects)):
        obj = Objects[i]
        v, f, patch2f = obj['v'], obj['f'], obj['patch']
        f_colors = np.ones((f.shape[0], 3))
        for p_id in range(len(patch2f)):
            f_colors[patch2f[p_id]] = colors[global_color_id % 100]
            global_color_id += 1
        write_ply_file(v, f, f_colors,
            # f'example_results/merge/cad/{obj_name}/iter_{iteration_time}_part_{i}.ply'
            # f'example_results/merge/selected/{obj_name}/seg_choose_2/iter_{iteration_time}_part_{i}.ply'
            # f'example_results/merge/selected/example_result/our/{obj_name}.ply'
        )
# save_all_objects()




Renderer(camera=PerspectiveCamera(children=(DirectionalLight(color='white', intensity=0.6, position=(0.0, 0.0,…